# RAMP Replication Notebook

This notebook reproduces the RAMP workflow for the DSS replication package. RAMP is treated here as a diagnostic decision-support artifact for reliability-aware campaign prioritization under biased attribution signals.

The notebook is organized around data preparation, descriptive statistics, selection-adjusted benchmarking, falsification-based diagnostics, campaign aggregation, diagnostic gates, campaign reprioritization, and time-split decision simulation.

## 1. Overview

The workflow starts from the public Criteo Attribution Dataset, builds a 24-hour benchmark sample, estimates selection-adjusted diagnostic signals, aggregates them to the campaign level, and exports the tables and figures used in the manuscript.

## 2. Environment Setup

Run `pip install -r requirements.txt` before executing this notebook locally. In Colab, install the same requirements and set `RAMP_USE_GOOGLE_DRIVE=1` only if you want to read data from Google Drive.

In [ ]:
# ====== RAMP local/Colab configuration ======
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd

os.environ.setdefault("WANDB_DISABLED", "true")


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def maybe_mount_drive(mount_point="/content/drive"):
    """Mount Google Drive only when running in Colab and explicitly requested."""
    if not in_colab() or os.environ.get("RAMP_USE_GOOGLE_DRIVE", "0") != "1":
        return None
    from google.colab import drive
    mount_path = Path(mount_point)
    mount_path.mkdir(parents=True, exist_ok=True)
    drive.mount(mount_point, force_remount=False)
    return mount_path


def find_repo_root():
    env_root = os.environ.get("RAMP_PROJECT_DIR")
    if env_root:
        return Path(env_root).expanduser().resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    if (cwd / "data").exists() and (cwd / "outputs").exists():
        return cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / "data").exists() and (parent / "outputs").exists():
            return parent
    return cwd


maybe_mount_drive()
PROJECT_DIR = find_repo_root()
DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
CHECKPOINT_DIR = PROCESSED_DATA_DIR / "notebook_checkpoints"
PAPER_OUTPUT_DIR = PROJECT_DIR / "outputs"
PAPER_TABLE_DIR = PAPER_OUTPUT_DIR / "tables"
PAPER_FIGURE_DIR = PAPER_OUTPUT_DIR / "figures"

DATA_CANDIDATES = [
    os.environ.get("RAMP_DATA_PATH"),
    RAW_DATA_DIR / "pcb_dataset_final.tsv",
    RAW_DATA_DIR / "criteo_attribution_dataset.tsv",
    RAW_DATA_DIR / "criteo_attribution_dataset.tsv.gz",
    RAW_DATA_DIR / "criteo_attribution.tsv",
]
DATA_PATH = next((Path(p).expanduser().resolve() for p in DATA_CANDIDATES if p and Path(p).expanduser().exists()), None)
if DATA_PATH is None:
    DATA_PATH = RAW_DATA_DIR / "pcb_dataset_final.tsv"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
PAPER_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DF_CLEAN_PATH = CHECKPOINT_DIR / "df_clean.parquet"
STEP_AB_META_PATH = CHECKPOINT_DIR / "step_ab_meta.json"
USER_DAY_PATH = CHECKPOINT_DIR / "user_day.parquet"
STEP_J_META_PATH = CHECKPOINT_DIR / "step_j_meta.json"
CAMPAIGN_DAY_PATH = CHECKPOINT_DIR / "campaign_day.parquet"
PANEL_UD_PATH = CHECKPOINT_DIR / "panel_ud.parquet"
TOP_CAMPAIGNS_PATH = CHECKPOINT_DIR / "top_campaigns.json"
RESULTS_CSV_PATH = CHECKPOINT_DIR / "campaign_results.csv"
RESULTS_PARQUET_PATH = CHECKPOINT_DIR / "campaign_results.parquet"
STATE_JSON_PATH = CHECKPOINT_DIR / "checkpoint_state.json"

ROBUSTNESS_TABLE_PATH = PAPER_TABLE_DIR / "table_step_e_robustness.csv"
OVERLAP_FIG_PATH = PAPER_FIGURE_DIR / "figure_propensity_overlap_24h.png"
MAIN_24H_JSON_PATH = PAPER_TABLE_DIR / "main_result_24h_summary.json"
PAPER_WRITEUP_PATH = PAPER_TABLE_DIR / "paper_ready_step_de_summary.md"

USE_WANDB = False
WANDB_PROJECT = "ramp-dss-replication"
WANDB_RUN_NAME = "ramp_replication_notebook"


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path, default=None):
    path = Path(path)
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return default


def save_df(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".parquet":
        df.to_parquet(path, index=False)
    elif path.suffix == ".csv":
        df.to_csv(path, index=False)
    else:
        raise ValueError(f"Unsupported file format: {path}")


def load_df(path):
    path = Path(path)
    if not path.exists():
        return None
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file format: {path}")


def write_text(text, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)

print("Project dir:", PROJECT_DIR)
print("Raw data path:", DATA_PATH)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Output dir:", PAPER_OUTPUT_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Raw Criteo data were not found. Download the Criteo Attribution Dataset "
        "from the original provider and place it under data/raw/. Expected path: "
        f"{DATA_PATH}"
    )

print("W&B is disabled by default. Set USE_WANDB = True manually if tracking is needed.")


## 3. Data Location and Input Requirements

Place the Criteo raw TSV file under `data/raw/`. The default expected filename is `pcb_dataset_final.tsv`, but `RAMP_DATA_PATH` can point to another local copy.

In [ ]:
# ====== Environment imports ======
# Install dependencies from requirements.txt before running this notebook locally.
import os, re, glob, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
try:
    from IPython.display import Markdown, display
except ImportError:
    display = print
    Markdown = str

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample

import lightgbm as lgb
import networkx as nx


In [ ]:
# ====== Load raw Criteo data ======
print("Reading raw data from:", DATA_PATH)
df = pd.read_csv(DATA_PATH, sep="\t")

required_columns = {"timestamp", "uid", "campaign", "conversion", "click"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Missing required Criteo columns: {missing_columns}")

print("Rows, columns:", df.shape)
display(df.head())


In [ ]:
# ====== Timestamp normalization check ======
# Keep the original numeric timestamp columns for downstream processing.
timestamp_preview = df[["timestamp", "conversion_timestamp"]].copy() if "conversion_timestamp" in df.columns else df[["timestamp"]].copy()
timestamp_preview["timestamp_utc"] = pd.to_datetime(
    pd.to_numeric(df["timestamp"], errors="coerce"),
    unit="s",
    origin="unix",
    utc=True,
)
if "conversion_timestamp" in df.columns:
    conv_ts = pd.to_numeric(df["conversion_timestamp"], errors="coerce")
    timestamp_preview["conversion_timestamp_utc"] = pd.to_datetime(
        conv_ts.where(conv_ts != -1),
        unit="s",
        origin="unix",
        utc=True,
    )
display(timestamp_preview.head())


In [ ]:
import pandas as pd
import numpy as np

# df is assumed already loaded
# Ensure these are numeric (sometimes they come in as object)
num_cols = ["click", "conversion", "attribution", "click_nb", "click_pos", "cost", "cpo"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# A helper: click_nb is only meaningful on rows with conversion==1
df["click_nb_if_conv"] = df["click_nb"].where(df["conversion"] == 1, np.nan)

# Optional: if you want to treat attribution as only defined when conversion==1
# (some datasets already do this; keeping it safe)
df["attribution_if_conv"] = df["attribution"].where(df["conversion"] == 1, 0)

# -------------------------
# 1) Group by uid (user-level)
# -------------------------
uid_stats = (
    df.groupby("uid", observed=True)
      .agg(
          impressions=("uid", "size"),
          click_rate=("click", "mean"),
          conversion_rate=("conversion", "mean"),
          attributed_conv_rate=("attribution_if_conv", "mean"),
          avg_click_nb_given_conv=("click_nb_if_conv", "mean"),
          # Often useful: whether user ever converted (binary per user)
          any_conversion=("conversion", "max"),
          any_click=("click", "max"),
          # Optional spend metrics
          total_cost=("cost", "sum"),
          avg_cost=("cost", "mean"),
      )
      .reset_index()
)

# -------------------------
# 2) Group by (uid, campaign) (user-campaign level)
# -------------------------
uid_campaign_stats = (
    df.groupby(["uid", "campaign"], observed=True)
      .agg(
          impressions=("uid", "size"),
          click_rate=("click", "mean"),
          conversion_rate=("conversion", "mean"),
          attributed_conv_rate=("attribution_if_conv", "mean"),
          avg_click_nb_given_conv=("click_nb_if_conv", "mean"),
          any_conversion=("conversion", "max"),
          any_click=("click", "max"),
          total_cost=("cost", "sum"),
          avg_cost=("cost", "mean"),
      )
      .reset_index()
)

print("uid_stats shape:", uid_stats.shape)
print("uid_campaign_stats shape:", uid_campaign_stats.shape)

uid_stats.head(), uid_campaign_stats.head()


## 5. Descriptive Statistics

In [ ]:
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.6g}")

Dataset snapshot (shape, columns, dtypes, duplicates)

In [ ]:
print("Shape:", df.shape)
print("\nColumns (count={}):".format(len(df.columns)))
print(df.columns.tolist())

# print("\nDtypes:")
# display(df.dtypes.sort_values())

# Duplicate rows (exact duplicates)
dup_rows = df.duplicated().sum()
print("\nExact duplicate rows:", dup_rows)

# Duplicate keys check (uid + timestamp + campaign) if these exist
key_cols = [c for c in ["uid", "timestamp", "campaign"] if c in df.columns]
if len(key_cols) == 3:
    dup_key = df.duplicated(subset=key_cols).sum()
    print("Duplicate (uid,timestamp,campaign) rows:", dup_key)

display(df.head(5))

Missingness & “-1 means missing” audit

In [ ]:
# Missingness (NaN)
na_rate = df.isna().mean().sort_values(ascending=False)
print("Top NA columns:")
display(na_rate.head(25))

# -1 as missing (common in this dataset)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

minus1_rate = {}
for c in num_cols:
    minus1_rate[c] = (df[c] == -1).mean()

minus1_rate = pd.Series(minus1_rate).sort_values(ascending=False)
print("Top '-1' rate columns:")
display(minus1_rate.head(25))

# Combined missing proxy: NA or -1
missing_proxy = {}
for c in num_cols:
    missing_proxy[c] = (df[c].isna() | (df[c] == -1)).mean()
missing_proxy = pd.Series(missing_proxy).sort_values(ascending=False)

print("Top (NA or -1) missing-proxy columns:")
display(missing_proxy.head(25))

Core counts: users, campaigns, conversions, clicks (+ rates)

In [ ]:
def safe_nunique(col):
    return df[col].nunique() if col in df.columns else None

print("Unique users:", safe_nunique("uid"))
print("Unique campaigns:", safe_nunique("campaign"))
print("Unique conversion_id:", safe_nunique("conversion_id"))

if "click" in df.columns:
    print("Click rate (mean click):", df["click"].mean())
    print("Clicks count:", int(df["click"].sum()))

if "conversion" in df.columns:
    print("Conversion rate (mean conversion):", df["conversion"].mean())
    print("Conversions count:", int(df["conversion"].sum()))

# Joint rates if both exist
if all(c in df.columns for c in ["click", "conversion"]):
    ct = pd.crosstab(df["click"], df["conversion"], normalize="all")
    print("\nJoint distribution P(click, conversion):")
    display(ct)

## 4. Data Preparation


In [ ]:
# ====== Shared helpers for Steps A-L ======
from pathlib import Path
import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
from sklearn.linear_model import SGDClassifier

DEFAULT_BENCHMARK_LIGHTGBM_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_data_in_leaf=200,
    n_estimators=400,
    n_jobs=-1,
    verbosity=-1,
)

DEFAULT_CAMPAIGN_LIGHTGBM_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=200,
    n_estimators=300,
    n_jobs=-1,
    verbosity=-1,
)


TIME_UNIT_SCALE = {"s": 1.0, "ms": 1e3, "us": 1e6, "ns": 1e9}


def infer_time_unit(ts_series):
    valid = pd.to_numeric(pd.Series(ts_series), errors="coerce").dropna()
    if valid.empty:
        raise ValueError("timestamp has no numeric values after coercion.")

    span_raw = float(valid.max() - valid.min())
    best_unit = None
    best_score = None
    best_days = None

    for unit, denom in TIME_UNIT_SCALE.items():
        days = span_raw / denom / 86400.0
        score = abs(days - 30.0) + (0 if days <= 365 else (days - 365) * 10)
        if best_score is None or score < best_score:
            best_unit = unit
            best_score = score
            best_days = days

    return best_unit, best_days


def normalize_time_to_seconds(ts_series, unit_hint=None, missing_sentinels=(-1,)):
    ser = pd.Series(ts_series)
    dtype_str = str(ser.dtype)
    if pd.api.types.is_datetime64_any_dtype(ser) or dtype_str.startswith("datetime64[ns"):
        ser_dt = pd.to_datetime(ser, utc=True, errors="coerce")
        out = pd.Series(np.nan, index=ser.index, dtype=np.float64)
        valid = ser_dt.notna()
        if valid.any():
            out.loc[valid] = ser_dt.loc[valid].astype("int64").astype(np.float64) / 1e9
        return out, "datetime64[ns]"

    numeric = pd.to_numeric(ser, errors="coerce")
    if missing_sentinels:
        numeric = numeric.mask(numeric.isin(list(missing_sentinels)))
    valid = numeric.dropna()
    if valid.empty:
        return pd.Series(np.nan, index=ser.index, dtype=np.float64), unit_hint or "unknown"

    inferred_unit = unit_hint or infer_time_unit(valid)[0]
    scale = TIME_UNIT_SCALE[inferred_unit]
    return numeric.astype(np.float64) / scale, inferred_unit


def build_conversion_event_columns(df_in, conv_ts_col=None):
    work = df_in.copy()
    work["ts_sec"], timestamp_unit = normalize_time_to_seconds(work["timestamp"])
    if work["ts_sec"].dropna().empty:
        raise ValueError("timestamp could not be normalized to seconds.")

    conversion_time_unit = None
    if conv_ts_col is not None:
        work["conv_ts_sec"], conversion_time_unit = normalize_time_to_seconds(work[conv_ts_col])
    else:
        work["conv_ts_sec"] = np.nan

    conv_flag = (pd.to_numeric(work["conversion"], errors="coerce").fillna(0) > 0)
    actual_from_col = work["conv_ts_sec"].where(conv_flag)
    actual_from_ts = work["ts_sec"].where(conv_flag)
    use_conversion_timestamp = actual_from_col.notna() & (actual_from_col >= actual_from_ts)
    work["actual_conv_ts_sec"] = actual_from_col.where(use_conversion_timestamp, actual_from_ts)

    work = work.sort_values(["uid", "ts_sec"]).reset_index(drop=True)
    conv_flag = (pd.to_numeric(work["conversion"], errors="coerce").fillna(0) > 0)
    conv_event = work["actual_conv_ts_sec"].where(conv_flag, np.nan)
    work["next_conv_ts_sec"] = conv_event.groupby(work["uid"]).shift(-1).groupby(work["uid"]).bfill()
    work["prev_conv_ts_sec"] = conv_event.groupby(work["uid"]).ffill()

    meta = {
        "timestamp_unit": timestamp_unit,
        "conversion_time_unit": conversion_time_unit,
        "actual_conversion_time_source": conv_ts_col if conv_ts_col is not None else "timestamp_on_conversion_rows",
    }
    return work, meta


def get_outcome_time_column(df_in, mode="forward"):
    if mode == "forward":
        for col in ["next_conv_ts_sec", "conv_ts_sec"]:
            if col in df_in.columns:
                return pd.to_numeric(df_in[col], errors="coerce")
    elif mode == "placebo":
        for col in ["prev_conv_ts_sec"]:
            if col in df_in.columns:
                return pd.to_numeric(df_in[col], errors="coerce")
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return pd.Series(np.nan, index=df_in.index, dtype=np.float64)


def get_outcome_eligibility_mask(df_in, horizon_hours=24.0, mode="forward"):
    horizon_seconds = horizon_hours * 3600.0
    ts_sec = pd.to_numeric(df_in["ts_sec"], errors="coerce")
    if mode == "forward":
        upper = df_in.groupby("uid")["ts_sec"].transform("max")
        mask = ts_sec <= (upper - horizon_seconds)
    elif mode == "placebo":
        lower = df_in.groupby("uid")["ts_sec"].transform("min")
        mask = ts_sec >= (lower + horizon_seconds)
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return mask.fillna(False).to_numpy()


class ConstantProbabilityModel:
    def __init__(self, value):
        self.value = float(np.clip(value, 1e-6, 1 - 1e-6))

    def predict_proba(self, x_test):
        p = np.full(len(x_test), self.value, dtype=float)
        return np.column_stack([1 - p, p])


def get_cat_cols(frame):
    out = []
    if "campaign" in frame.columns:
        out.append("campaign")
    out += [c for c in frame.columns if str(c).lower().startswith("cat")]
    return [c for c in out if c in frame.columns]


def prepare_lightgbm_frame(frame):
    out = frame.copy()
    cat_cols = get_cat_cols(out)
    for col in cat_cols:
        if not isinstance(out[col].dtype, CategoricalDtype):
            out[col] = out[col].astype("category")
    return out, cat_cols


def prepare_numeric_frame(frame):
    out = frame.copy()
    for col in out.columns:
        if out[col].dtype == "object" or isinstance(out[col].dtype, CategoricalDtype):
            out[col] = out[col].astype("category").cat.codes.replace(-1, np.nan)
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.fillna(0.0).astype(np.float32)


def fit_binary_model(x_train, y_train, learner="lightgbm", lightgbm_params=None):
    y_train = np.asarray(y_train).astype(int)
    if y_train.size == 0:
        raise ValueError("Cannot fit on an empty training sample.")
    if np.unique(y_train).size == 1:
        return ConstantProbabilityModel(float(y_train[0]))

    lightgbm_params = lightgbm_params or {}

    if learner == "lightgbm":
        x_use, cat_cols = prepare_lightgbm_frame(x_train)
        model = lgb.LGBMClassifier(**lightgbm_params)
        model.fit(x_use, y_train, categorical_feature=cat_cols)
        return model

    if learner == "sgd_logit":
        x_use = prepare_numeric_frame(x_train)
        model = SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-5,
            max_iter=25,
            tol=1e-3,
            random_state=42,
        )
        model.fit(x_use, y_train)
        return model

    raise ValueError(f"Unknown learner: {learner}")


def predict_binary_model(model, x_test, learner="lightgbm"):
    if isinstance(model, ConstantProbabilityModel):
        pred = model.predict_proba(x_test)[:, 1]
        return np.clip(pred, 1e-6, 1 - 1e-6)

    if learner == "lightgbm":
        x_use, _ = prepare_lightgbm_frame(x_test)
        pred = model.predict_proba(x_use)[:, 1]
        return np.clip(np.asarray(pred, dtype=float), 1e-6, 1 - 1e-6)

    if learner == "sgd_logit":
        x_use = prepare_numeric_frame(x_test)
        pred = model.predict_proba(x_use)[:, 1]
        return np.clip(np.asarray(pred, dtype=float), 1e-6, 1 - 1e-6)

    raise ValueError(f"Unknown learner: {learner}")


def build_forward_outcome(df_in, horizon_hours=24.0):
    horizon_seconds = horizon_hours * 3600.0
    event_time = get_outcome_time_column(df_in, mode="forward")
    ts_sec = pd.to_numeric(df_in["ts_sec"], errors="coerce")
    y = (
        event_time.notna()
        & (event_time > ts_sec)
        & (event_time <= ts_sec + horizon_seconds)
    ).astype(int)
    return y.to_numpy()


def build_placebo_outcome(df_in, horizon_hours=24.0):
    horizon_seconds = horizon_hours * 3600.0
    event_time = get_outcome_time_column(df_in, mode="placebo")
    ts_sec = pd.to_numeric(df_in["ts_sec"], errors="coerce")
    y = (
        event_time.notna()
        & (event_time < ts_sec)
        & (event_time >= ts_sec - horizon_seconds)
    ).astype(int)
    return y.to_numpy()


def cluster_se_from_if(if_values, group_values):
    tmp = pd.DataFrame({"group": group_values, "if": if_values})
    cluster_sum = tmp.groupby("group", sort=False)["if"].sum().to_numpy(dtype=np.float64)
    group_count = len(cluster_sum)
    n_eff = len(if_values)
    if group_count <= 1 or n_eff <= 1:
        return np.nan, group_count
    centered = cluster_sum - cluster_sum.mean()
    var_theta = (group_count / (group_count - 1.0)) * np.sum(centered ** 2) / (n_eff ** 2)
    return float(np.sqrt(max(var_theta, 0.0))), group_count


In [ ]:
# ====== Step A/B: Build cleaned impression-level benchmark sample with cache ======
import numpy as np
import pandas as pd

HOURS_AHEAD = 24.0
H = HOURS_AHEAD * 3600.0

print("Project dir:", PROJECT_DIR)
print("Raw data path:", DATA_PATH)

if "df" not in globals():
    print("Reading raw TSV from:", DATA_PATH)
    df = pd.read_csv(DATA_PATH, sep="\t")
else:
    print("Using existing raw df from notebook memory.")

required = ["uid", "timestamp", "campaign", "click", "conversion"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

raw_click = pd.to_numeric(df["click"], errors="coerce").fillna(0)
raw_conversion = pd.to_numeric(df["conversion"], errors="coerce").fillna(0)
raw_click_mean = float((raw_click > 0).mean())
raw_conversion_mean = float((raw_conversion > 0).mean())

STEP_AB_CACHE_VERSION = "2026-03-29-outcome-v2"
cached_meta = load_json(STEP_AB_META_PATH, default={}) or {}
used_conversion_timestamp = cached_meta.get("used_conversion_timestamp")
time_unit = cached_meta.get("time_unit")
conversion_time_unit = cached_meta.get("conversion_time_unit")

cache_ok = DF_CLEAN_PATH.exists() and cached_meta.get("cache_version") == STEP_AB_CACHE_VERSION
if cache_ok:
    print("Loading cached df_clean from:", DF_CLEAN_PATH)
    df_clean = pd.read_parquet(DF_CLEAN_PATH)
    required_cached = {"uid", "ts_sec", "click", "next_conv_ts_sec", "prev_conv_ts_sec"}
    if not required_cached.issubset(df_clean.columns):
        print("[Cache] Step A/B cache is missing v2 outcome columns. Rebuilding...")
        cache_ok = False
else:
    print("No compatible cached df_clean found. Running full Step A/B...")

if not cache_ok:
    conv_ts_col = None
    for cand in ["conversion_timestamp", "conv_timestamp", "conv_ts", "conversion_ts", "T_i"]:
        if cand in df.columns:
            conv_ts_col = cand
            break

    df_clean, time_meta = build_conversion_event_columns(df, conv_ts_col=conv_ts_col)
    df_clean = df_clean.dropna(subset=["uid", "ts_sec"]).copy()
    time_unit = time_meta["timestamp_unit"]
    conversion_time_unit = time_meta["conversion_time_unit"]
    used_conversion_timestamp = time_meta["actual_conversion_time_source"]

    print(f"[Time] timestamp unit: {time_unit}")
    if conv_ts_col is not None:
        print(f"[Time] {conv_ts_col} unit: {conversion_time_unit}")
    else:
        print("[Outcome] No conversion timestamp column found. Falling back to conversion-row timestamps.")

    df_clean["click"] = pd.to_numeric(df_clean["click"], errors="coerce").fillna(0).astype(int)
    df_clean["click"] = (df_clean["click"] > 0).astype(int)

    df_clean["conversion"] = pd.to_numeric(df_clean["conversion"], errors="coerce").fillna(0).astype(int)
    df_clean["conversion"] = (df_clean["conversion"] > 0).astype(int)

    protected = {
        "uid", "timestamp", "ts_sec", "campaign", "click", "conversion",
        "conv_ts_sec", "actual_conv_ts_sec", "next_conv_ts_sec", "prev_conv_ts_sec",
    }
    cat_cols = [c for c in df_clean.columns if str(c).lower().startswith("cat")]
    for c in cat_cols:
        df_clean.loc[df_clean[c].astype(str) == "-1", c] = np.nan

    num_cols = [
        c for c in df_clean.columns
        if c not in protected and pd.api.types.is_numeric_dtype(df_clean[c])
    ]
    for c in num_cols:
        df_clean.loc[df_clean[c] == -1, c] = np.nan

    df_clean = df_clean[df_clean["conversion"] == 0].copy()
    df_clean = df_clean.sort_values(["uid", "ts_sec"]).reset_index(drop=True)
    df_clean.to_parquet(DF_CLEAN_PATH, index=False)
    print("Saved df_clean to:", DF_CLEAN_PATH)

D = (pd.to_numeric(df_clean["click"], errors="coerce").fillna(0).astype(int) > 0).astype(int).to_numpy()
Y_forward_24h = build_forward_outcome(df_clean, horizon_hours=24.0).astype(int)
Y_placebo_24h = build_placebo_outcome(df_clean, horizon_hours=24.0).astype(int)
eligible_forward_24h = get_outcome_eligibility_mask(df_clean, horizon_hours=24.0, mode="forward")
eligible_placebo_24h = get_outcome_eligibility_mask(df_clean, horizon_hours=24.0, mode="placebo")
forward_event = get_outcome_time_column(df_clean, mode="forward")
placebo_event = get_outcome_time_column(df_clean, mode="placebo")
forward_valid = forward_event.notna()
placebo_valid = placebo_event.notna()
forward_24h_mean = float(Y_forward_24h[eligible_forward_24h].mean()) if eligible_forward_24h.any() else np.nan
placebo_24h_mean = float(Y_placebo_24h[eligible_placebo_24h].mean()) if eligible_placebo_24h.any() else np.nan
forward_event_after_rate = float((forward_event[forward_valid] > df_clean.loc[forward_valid, "ts_sec"]).mean()) if forward_valid.any() else np.nan
placebo_event_before_rate = float((placebo_event[placebo_valid] < df_clean.loc[placebo_valid, "ts_sec"]).mean()) if placebo_valid.any() else np.nan

if not used_conversion_timestamp:
    if "conv_ts_sec" in df_clean.columns:
        used_conversion_timestamp = "conversion_timestamp-derived"
    elif "next_conv_ts_sec" in df_clean.columns:
        used_conversion_timestamp = "next_conv_ts_sec"
    else:
        used_conversion_timestamp = "unknown"

step_ab_meta = {
    "raw_rows": int(df.shape[0]),
    "raw_cols": int(df.shape[1]),
    "raw_click_mean": raw_click_mean,
    "raw_conversion_mean": raw_conversion_mean,
    "cleaned_rows": int(df_clean.shape[0]),
    "cleaned_cols": int(df_clean.shape[1]),
    "cleaned_click_mean": float(D.mean()),
    "forward_24h_mean": forward_24h_mean,
    "placebo_24h_mean": placebo_24h_mean,
    "forward_24h_eligible_rows": int(eligible_forward_24h.sum()),
    "placebo_24h_eligible_rows": int(eligible_placebo_24h.sum()),
    "forward_event_after_rate": forward_event_after_rate,
    "placebo_event_before_rate": placebo_event_before_rate,
    "time_unit": time_unit,
    "conversion_time_unit": conversion_time_unit,
    "hours_ahead": float(HOURS_AHEAD),
    "used_conversion_timestamp": used_conversion_timestamp,
    "cache_version": STEP_AB_CACHE_VERSION,
}

save_json(step_ab_meta, STEP_AB_META_PATH)
save_df(pd.DataFrame([step_ab_meta]), PAPER_OUTPUT_DIR / "table_step_ab_metadata.csv")

print("[Clean] df_clean shape:", df_clean.shape)
print("[Clean] click mean:", float(D.mean()))
print("[Outcome] forward 24h eligible rows:", int(eligible_forward_24h.sum()))
print("[Outcome] placebo 24h eligible rows:", int(eligible_placebo_24h.sum()))
print("[Outcome] forward event > ts rate:", forward_event_after_rate)
print("[Outcome] placebo event < ts rate:", placebo_event_before_rate)
print("[Outcome] forward 24h mean:", forward_24h_mean)
print("[Outcome] placebo 24h mean:", placebo_24h_mean)
print("Saved metadata to:", STEP_AB_META_PATH)

display(pd.Series(step_ab_meta, name="value").to_frame())


In [ ]:
# ====== Step C: Build X_it (pre-treatment state features) ======
import numpy as np
import pandas as pd
from collections import deque, defaultdict

df_clean = df_clean.sort_values(["uid", "ts_sec"]).reset_index(drop=True)


def build_state_features(df_in):
    uid = df_in["uid"].to_numpy()
    t = df_in["ts_sec"].to_numpy().astype(np.float64)
    click = df_in["click"].to_numpy().astype(np.int8)
    camp = df_in["campaign"].astype(str).to_numpy()

    n = len(df_in)
    print("Step C on rows:", n)

    W1 = 3600.0
    W24 = 86400.0
    W7 = 7 * 86400.0
    W30 = 30 * 86400.0

    imp_1h = np.zeros(n, dtype=np.int32); clk_1h = np.zeros(n, dtype=np.int32)
    imp_24h = np.zeros(n, dtype=np.int32); clk_24h = np.zeros(n, dtype=np.int32)
    imp_7d = np.zeros(n, dtype=np.int32); clk_7d = np.zeros(n, dtype=np.int32)
    imp_30d = np.zeros(n, dtype=np.int32); clk_30d = np.zeros(n, dtype=np.int32)

    imp_24h_c = np.zeros(n, dtype=np.int32); clk_24h_c = np.zeros(n, dtype=np.int32)
    imp_7d_c = np.zeros(n, dtype=np.int32); clk_7d_c = np.zeros(n, dtype=np.int32)

    tslc = np.full(n, 999999.0, dtype=np.float64)
    hour_of_day = ((t // 3600) % 24).astype(np.int16)
    day_index = (t // 86400).astype(np.int32)

    q1 = deque(); q24 = deque(); q7 = deque(); q30 = deque()
    s1 = s24 = s7 = s30 = 0

    q24_c = defaultdict(deque)
    q7_c = defaultdict(deque)
    s24_c = defaultdict(int)
    s7_c = defaultdict(int)

    def purge(q, cutoff, s):
        while q and q[0][0] <= cutoff:
            _, cf = q.popleft()
            s -= cf
        return s

    def purge_c(dq, cutoff, running_sum_dict, camp_key):
        q = dq.get(camp_key)
        if not q:
            return
        s = running_sum_dict[camp_key]
        while q and q[0][0] <= cutoff:
            _, cf = q.popleft()
            s -= cf
        if len(q) == 0:
            dq.pop(camp_key, None)
            running_sum_dict.pop(camp_key, None)
        else:
            running_sum_dict[camp_key] = s

    cur = None
    last_click_time = None

    for i in range(n):
        if uid[i] != cur:
            cur = uid[i]
            q1.clear(); q24.clear(); q7.clear(); q30.clear()
            s1 = s24 = s7 = s30 = 0
            q24_c.clear(); q7_c.clear(); s24_c.clear(); s7_c.clear()
            last_click_time = None

        ti = t[i]
        ci = camp[i]
        cf = int(click[i] > 0)

        s1 = purge(q1, ti - W1, s1)
        s24 = purge(q24, ti - W24, s24)
        s7 = purge(q7, ti - W7, s7)
        s30 = purge(q30, ti - W30, s30)

        imp_1h[i] = len(q1); clk_1h[i] = s1
        imp_24h[i] = len(q24); clk_24h[i] = s24
        imp_7d[i] = len(q7); clk_7d[i] = s7
        imp_30d[i] = len(q30); clk_30d[i] = s30

        purge_c(q24_c, ti - W24, s24_c, ci)
        purge_c(q7_c, ti - W7, s7_c, ci)

        qci24 = q24_c.get(ci, deque())
        qci7 = q7_c.get(ci, deque())
        imp_24h_c[i] = len(qci24); clk_24h_c[i] = s24_c.get(ci, 0)
        imp_7d_c[i] = len(qci7); clk_7d_c[i] = s7_c.get(ci, 0)

        tslc[i] = 999999.0 if last_click_time is None else (ti - last_click_time)

        q1.append((ti, cf)); s1 += cf
        q24.append((ti, cf)); s24 += cf
        q7.append((ti, cf)); s7 += cf
        q30.append((ti, cf)); s30 += cf

        dq24 = q24_c.setdefault(ci, deque()); dq24.append((ti, cf)); s24_c[ci] = s24_c.get(ci, 0) + cf
        dq7 = q7_c.setdefault(ci, deque()); dq7.append((ti, cf)); s7_c[ci] = s7_c.get(ci, 0) + cf

        if cf == 1:
            last_click_time = ti

    X_out = pd.DataFrame({
        "imp_cnt_1h_pre": imp_1h, "clk_cnt_1h_pre": clk_1h,
        "imp_cnt_24h_pre": imp_24h, "clk_cnt_24h_pre": clk_24h,
        "imp_cnt_7d_pre": imp_7d, "clk_cnt_7d_pre": clk_7d,
        "imp_cnt_30d_pre": imp_30d, "clk_cnt_30d_pre": clk_30d,
        "imp_cnt_24h_pre_same_camp": imp_24h_c, "clk_cnt_24h_pre_same_camp": clk_24h_c,
        "imp_cnt_7d_pre_same_camp": imp_7d_c, "clk_cnt_7d_pre_same_camp": clk_7d_c,
        "time_since_last_click_sec": tslc,
        "hour_of_day": hour_of_day,
        "day_index": day_index,
        "campaign": pd.Series(camp).astype("category"),
    })

    cap = 999999.0
    X_out["no_prior_click"] = (X_out["time_since_last_click_sec"] >= cap).astype(int)
    X_out["log_time_since_last_click"] = np.log1p(np.minimum(X_out["time_since_last_click_sec"], cap))

    cat_cols = [c for c in df_in.columns if str(c).lower().startswith("cat")]
    for c in sorted(cat_cols):
        X_out[c] = df_in[c].astype("object").where(df_in[c].notna(), "MISSING").astype("category")

    return X_out


X_df = build_state_features(df_clean)

leakage_checks = {
    "negative_recency_rows": int((X_df["time_since_last_click_sec"] < 0).sum()),
    "click_count_exceeds_impressions_24h": int((X_df["clk_cnt_24h_pre"] > X_df["imp_cnt_24h_pre"]).sum()),
    "same_campaign_click_count_exceeds_impressions_24h": int((X_df["clk_cnt_24h_pre_same_camp"] > X_df["imp_cnt_24h_pre_same_camp"]).sum()),
}
missing_top10 = X_df.isna().mean().sort_values(ascending=False).head(10)

print("X_df shape:", X_df.shape)
print("Leakage checks:", leakage_checks)
print("Top missingness shares:")
display(missing_top10.to_frame("missing_rate"))
display(X_df.head(3))


## 7. RAMP Layer 2: Diagnostic Tests

In [ ]:
# ====== Step D0: Raw Outcome Diagnostics ======
import numpy as np
import pandas as pd

RAW_OUTCOME_HOURS = 24.0
raw_diag_eligible_mask = get_outcome_eligibility_mask(
    df_clean,
    horizon_hours=RAW_OUTCOME_HOURS,
    mode="forward",
)
if raw_diag_eligible_mask.sum() == 0:
    raise ValueError("No eligible observations found for Step D0 raw outcome diagnostics.")

raw_diag_df = (
    df_clean.loc[raw_diag_eligible_mask]
    .sort_values(["uid", "ts_sec"])
    .reset_index(drop=True)
    .copy()
)

raw_diag_df["campaign"] = raw_diag_df["campaign"].astype(str)
raw_diag_df["D"] = (pd.to_numeric(raw_diag_df["click"], errors="coerce").fillna(0) > 0).astype(int)
raw_diag_df["Y"] = build_forward_outcome(raw_diag_df, horizon_hours=RAW_OUTCOME_HOURS).astype(int)

raw_diag_ts_sec = pd.to_numeric(raw_diag_df["ts_sec"], errors="coerce")
raw_diag_df["hour_of_day"] = ((raw_diag_ts_sec // 3600) % 24).astype("Int64")
raw_diag_df["day_index"] = (raw_diag_ts_sec // 86400).astype("Int64")
raw_diag_df["user_sample_rows"] = raw_diag_df.groupby("uid")["uid"].transform("size").astype(int)

click_ts = pd.Series(
    np.where(raw_diag_df["D"] == 1, raw_diag_ts_sec, np.nan),
    index=raw_diag_df.index,
    dtype=float,
)
last_click_seen = click_ts.groupby(raw_diag_df["uid"]).ffill()
prev_click_ts = last_click_seen.groupby(raw_diag_df["uid"]).shift(1)
raw_diag_df["time_since_last_click_sec"] = raw_diag_ts_sec - prev_click_ts
raw_diag_df["no_prior_click"] = prev_click_ts.isna().astype(int)


def _safe_rank_qcut(series, q, labels):
    ser = pd.Series(series)
    fill_value = float(ser.dropna().median()) if ser.notna().any() else 0.0
    ranked = ser.fillna(fill_value).rank(method="first")
    bins = min(q, ranked.nunique())
    if bins <= 1:
        return pd.Series([labels[0]] * len(ser), index=ser.index, dtype="object")
    use_labels = labels[:bins]
    return pd.qcut(ranked, q=bins, labels=use_labels)


def _safe_ratio(num, den):
    num = float(num) if pd.notna(num) else np.nan
    den = float(den) if pd.notna(den) else np.nan
    if not np.isfinite(num) or not np.isfinite(den) or abs(den) <= 1e-12:
        return np.nan
    return float(num / den)


def _aggregate(frame, group_col, extra_named_aggs=None):
    extra_named_aggs = extra_named_aggs or {}
    agg_map = {
        "n_obs": ("Y", "size"),
        "y_sum": ("Y", "sum"),
        "y_mean": ("Y", "mean"),
        "d_mean": ("D", "mean"),
    }
    agg_map.update(extra_named_aggs)
    out = (
        frame.groupby(group_col, dropna=False)
             .agg(**agg_map)
             .reset_index()
    )
    return out


raw_outcome_overall_df = pd.DataFrame(
    [
        {
            "horizon_hours": RAW_OUTCOME_HOURS,
            "sample_scope": "eligible_forward_24h",
            "n_obs": int(raw_diag_df.shape[0]),
            "n_users": int(raw_diag_df["uid"].nunique()),
            "n_campaigns": int(raw_diag_df["campaign"].nunique()),
            "y_mean": float(raw_diag_df["Y"].mean()),
            "d_mean": float(raw_diag_df["D"].mean()),
            "y_mean_d1": float(raw_diag_df.loc[raw_diag_df["D"] == 1, "Y"].mean())
            if (raw_diag_df["D"] == 1).any()
            else np.nan,
            "y_mean_d0": float(raw_diag_df.loc[raw_diag_df["D"] == 0, "Y"].mean())
            if (raw_diag_df["D"] == 0).any()
            else np.nan,
            "raw_difference_d1_minus_d0": float(
                raw_diag_df.loc[raw_diag_df["D"] == 1, "Y"].mean()
                - raw_diag_df.loc[raw_diag_df["D"] == 0, "Y"].mean()
            )
            if (raw_diag_df["D"] == 1).any() and (raw_diag_df["D"] == 0).any()
            else np.nan,
        }
    ]
)

campaign_counts = (
    raw_diag_df.groupby("campaign", as_index=False)
    .agg(campaign_exposure_count=("campaign", "size"))
    .sort_values(["campaign_exposure_count", "campaign"], ascending=[True, True])
    .reset_index(drop=True)
)
campaign_counts["decile"] = _safe_rank_qcut(
    campaign_counts["campaign_exposure_count"],
    10,
    list(range(1, 11)),
).astype(int)
raw_diag_df = raw_diag_df.merge(
    campaign_counts[["campaign", "decile"]],
    on="campaign",
    how="left",
)
raw_outcome_by_campaign_decile_df = _aggregate(
    raw_diag_df,
    "decile",
    extra_named_aggs={"n_campaigns": ("campaign", "nunique")},
)
raw_outcome_by_campaign_decile_df = raw_outcome_by_campaign_decile_df.sort_values("decile").reset_index(drop=True)

user_activity = (
    raw_diag_df[["uid", "user_sample_rows"]]
    .drop_duplicates()
    .sort_values(["user_sample_rows", "uid"], ascending=[True, True])
    .reset_index(drop=True)
    .rename(columns={"user_sample_rows": "user_activity_rows"})
)
user_activity["decile"] = _safe_rank_qcut(
    user_activity["user_activity_rows"],
    10,
    list(range(1, 11)),
).astype(int)
raw_diag_df = raw_diag_df.merge(
    user_activity[["uid", "decile"]],
    on="uid",
    how="left",
    suffixes=("", "_user_activity"),
)
raw_diag_df = raw_diag_df.rename(columns={"decile_user_activity": "user_activity_decile"})
raw_outcome_by_user_activity_decile_df = _aggregate(
    raw_diag_df,
    "user_activity_decile",
    extra_named_aggs={"n_users": ("uid", "nunique")},
)
raw_outcome_by_user_activity_decile_df = raw_outcome_by_user_activity_decile_df.sort_values("user_activity_decile").reset_index(drop=True)

raw_outcome_by_hour_df = (
    _aggregate(raw_diag_df, "hour_of_day")
    .sort_values("hour_of_day")
    .reset_index(drop=True)
)

raw_outcome_by_day_index_df = (
    _aggregate(raw_diag_df, "day_index")
    .sort_values("day_index")
    .reset_index(drop=True)
)

recency_days = raw_diag_df["time_since_last_click_sec"] / 86400.0
recency_bucket = pd.cut(
    recency_days,
    bins=[-np.inf, 1.0, 7.0, 30.0, np.inf],
    labels=["<=1d", "1-7d", "7-30d", ">30d"],
).astype(object)
recency_bucket.loc[raw_diag_df["no_prior_click"] == 1] = "no_prior_click"
recency_order = ["no_prior_click", "<=1d", "1-7d", "7-30d", ">30d"]
raw_diag_df["recency_bucket"] = pd.Categorical(recency_bucket, categories=recency_order, ordered=True)
raw_outcome_by_recency_bucket_df = (
    _aggregate(raw_diag_df, "recency_bucket")
    .rename(columns={"recency_bucket": "bucket"})
    .sort_values("bucket")
    .reset_index(drop=True)
)

top_50_campaigns = set(campaign_counts.sort_values("campaign_exposure_count", ascending=False).head(50)["campaign"])
top_100_campaigns = set(campaign_counts.sort_values("campaign_exposure_count", ascending=False).head(100)["campaign"])
top_vs_tail_rows = []
for group_name, campaign_set in [
    ("top_50_campaigns", top_50_campaigns),
    ("top_100_campaigns", top_100_campaigns),
    ("tail_campaigns_excluding_top_100", set(campaign_counts["campaign"]) - top_100_campaigns),
]:
    sub = raw_diag_df.loc[raw_diag_df["campaign"].isin(campaign_set)].copy()
    top_vs_tail_rows.append(
        {
            "group": group_name,
            "n_obs": int(sub.shape[0]),
            "n_campaigns": int(sub["campaign"].nunique()) if not sub.empty else 0,
            "y_sum": int(sub["Y"].sum()) if not sub.empty else 0,
            "y_mean": float(sub["Y"].mean()) if not sub.empty else np.nan,
            "d_mean": float(sub["D"].mean()) if not sub.empty else np.nan,
        }
    )
raw_outcome_top_vs_tail_campaigns_df = pd.DataFrame(top_vs_tail_rows)

save_df(raw_outcome_overall_df, PAPER_OUTPUT_DIR / "table_raw_outcome_overall.csv")
save_json(raw_outcome_overall_df.iloc[0].to_dict(), PAPER_OUTPUT_DIR / "summary_raw_outcome_overall.json")
save_df(raw_outcome_by_campaign_decile_df, PAPER_OUTPUT_DIR / "table_raw_outcome_by_campaign_decile.csv")
save_df(raw_outcome_by_user_activity_decile_df, PAPER_OUTPUT_DIR / "table_raw_outcome_by_user_activity_decile.csv")
save_df(raw_outcome_by_hour_df, PAPER_OUTPUT_DIR / "table_raw_outcome_by_hour.csv")
save_df(raw_outcome_by_day_index_df, PAPER_OUTPUT_DIR / "table_raw_outcome_by_day_index.csv")
save_df(raw_outcome_by_recency_bucket_df, PAPER_OUTPUT_DIR / "table_raw_outcome_by_recency_bucket.csv")
save_df(raw_outcome_top_vs_tail_campaigns_df, PAPER_OUTPUT_DIR / "table_raw_outcome_top_vs_tail_campaigns.csv")

bucket_frames = [
    raw_outcome_by_campaign_decile_df.assign(bucket_type="campaign_decile", bucket_value=lambda x: x["decile"].astype(str)),
    raw_outcome_by_user_activity_decile_df.assign(bucket_type="user_activity_decile", bucket_value=lambda x: x["user_activity_decile"].astype(str)),
    raw_outcome_by_hour_df.assign(bucket_type="hour_of_day", bucket_value=lambda x: x["hour_of_day"].astype(str)),
    raw_outcome_by_day_index_df.assign(bucket_type="day_index", bucket_value=lambda x: x["day_index"].astype(str)),
    raw_outcome_by_recency_bucket_df.assign(bucket_type="recency_bucket", bucket_value=lambda x: x["bucket"].astype(str)),
    raw_outcome_top_vs_tail_campaigns_df.assign(bucket_type="campaign_scale_group", bucket_value=lambda x: x["group"].astype(str)),
]
combined_bucket_df = pd.concat(bucket_frames, axis=0, ignore_index=True, sort=False)
combined_bucket_df["positive_outcomes"] = pd.to_numeric(combined_bucket_df["y_sum"], errors="coerce").fillna(0).astype(int)

overall_y_mean = float(raw_outcome_overall_df.loc[0, "y_mean"])
sparsity_label = "not_assessed"
if np.isfinite(overall_y_mean):
    if overall_y_mean < 0.001:
        sparsity_label = "extremely_sparse"
    elif overall_y_mean < 0.01:
        sparsity_label = "sparse"
    else:
        sparsity_label = "moderate_or_dense"

min_bucket_threshold = max(1e-6, overall_y_mean * 0.25) if np.isfinite(overall_y_mean) else 1e-6
near_zero_bucket_df = (
    combined_bucket_df.loc[
        (combined_bucket_df["n_obs"] >= 100)
        & (
            (combined_bucket_df["y_mean"] <= min_bucket_threshold)
            | (combined_bucket_df["positive_outcomes"] <= 5)
        ),
        ["bucket_type", "bucket_value", "n_obs", "y_mean", "d_mean", "positive_outcomes"],
    ]
    .sort_values(["y_mean", "n_obs"], ascending=[True, False])
    .head(20)
    .reset_index(drop=True)
)

top_campaign_decile_y = raw_outcome_by_campaign_decile_df.loc[
    raw_outcome_by_campaign_decile_df["decile"] == raw_outcome_by_campaign_decile_df["decile"].max(),
    "y_mean",
].mean()
bottom_campaign_decile_y = raw_outcome_by_campaign_decile_df.loc[
    raw_outcome_by_campaign_decile_df["decile"] == raw_outcome_by_campaign_decile_df["decile"].min(),
    "y_mean",
].mean()
top_user_decile_y = raw_outcome_by_user_activity_decile_df.loc[
    raw_outcome_by_user_activity_decile_df["user_activity_decile"] == raw_outcome_by_user_activity_decile_df["user_activity_decile"].max(),
    "y_mean",
].mean()
bottom_user_decile_y = raw_outcome_by_user_activity_decile_df.loc[
    raw_outcome_by_user_activity_decile_df["user_activity_decile"] == raw_outcome_by_user_activity_decile_df["user_activity_decile"].min(),
    "y_mean",
].mean()
top50_y = raw_outcome_top_vs_tail_campaigns_df.loc[
    raw_outcome_top_vs_tail_campaigns_df["group"] == "top_50_campaigns",
    "y_mean",
].mean()
tail_y = raw_outcome_top_vs_tail_campaigns_df.loc[
    raw_outcome_top_vs_tail_campaigns_df["group"] == "tail_campaigns_excluding_top_100",
    "y_mean",
].mean()

summary_raw_outcome_diagnostics = {
    "horizon_hours": RAW_OUTCOME_HOURS,
    "sample_scope": "eligible_forward_24h",
    "overall_y_mean": overall_y_mean,
    "overall_d_mean": float(raw_outcome_overall_df.loc[0, "d_mean"]),
    "overall_y_mean_d1": float(raw_outcome_overall_df.loc[0, "y_mean_d1"]),
    "overall_y_mean_d0": float(raw_outcome_overall_df.loc[0, "y_mean_d0"]),
    "raw_difference_d1_minus_d0": float(raw_outcome_overall_df.loc[0, "raw_difference_d1_minus_d0"]),
    "outcome_sparsity_label": sparsity_label,
    "max_bucket_y_mean": float(combined_bucket_df["y_mean"].max()),
    "min_bucket_y_mean": float(combined_bucket_df["y_mean"].min()),
    "campaign_top_to_bottom_decile_y_ratio": _safe_ratio(top_campaign_decile_y, bottom_campaign_decile_y),
    "user_activity_top_to_bottom_decile_y_ratio": _safe_ratio(top_user_decile_y, bottom_user_decile_y),
    "top50_vs_tail_y_ratio": _safe_ratio(top50_y, tail_y),
    "top_campaigns_y_mean_higher_than_tail": bool(top50_y > tail_y) if pd.notna(top50_y) and pd.notna(tail_y) else None,
    "active_users_y_mean_higher_than_inactive_tail": bool(top_user_decile_y > bottom_user_decile_y)
    if pd.notna(top_user_decile_y) and pd.notna(bottom_user_decile_y)
    else None,
    "negative_benchmark_may_partly_reflect_tail_noise": bool(
        _safe_ratio(top50_y, tail_y) > 1.5
    ) if pd.notna(top50_y) and pd.notna(tail_y) else None,
    "near_zero_positive_buckets": near_zero_bucket_df.to_dict(orient="records"),
}

save_json(summary_raw_outcome_diagnostics, PAPER_OUTPUT_DIR / "summary_raw_outcome_diagnostics.json")

display(raw_outcome_overall_df)
display(raw_outcome_by_campaign_decile_df)
display(raw_outcome_by_user_activity_decile_df)
display(raw_outcome_by_recency_bucket_df)
display(raw_outcome_top_vs_tail_campaigns_df)
print("Saved Step D0 diagnostics to:", PAPER_OUTPUT_DIR)

In [ ]:
# ====== Step D: PLR Double Machine Learning with cross-fitting by user + corrected USER-CLUSTERED SE ======
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold

df_clean = df_clean.sort_values(["uid", "ts_sec"]).reset_index(drop=True)
X_df = X_df.reset_index(drop=True).copy()
assert len(X_df) == len(df_clean), f"X_df length {len(X_df)} != df_clean length {len(df_clean)}"

STEPD_PAYLOADS = globals().get("STEPD_PAYLOADS", {})
STEPD_PLR_CACHE = globals().get("STEPD_PLR_CACHE", {})
STEPD_BENCHMARK_CACHE = globals().get("STEPD_BENCHMARK_CACHE", {})


def get_outcome_vector(df_in, horizon_hours=24.0, mode="forward"):
    if mode == "forward":
        return build_forward_outcome(df_in, horizon_hours=horizon_hours)
    if mode == "placebo":
        return build_placebo_outcome(df_in, horizon_hours=horizon_hours)
    raise ValueError(f"Unknown mode: {mode}")


def _safe_scalar(value):
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    return value


def _safe_ratio(num, den):
    num = float(num) if pd.notna(num) else np.nan
    den = float(den) if pd.notna(den) else np.nan
    if not np.isfinite(num) or not np.isfinite(den) or abs(den) <= 1e-12:
        return np.nan
    return float(num / den)


def _normalize_score(series, higher_is_better=True):
    ser = pd.to_numeric(pd.Series(series), errors="coerce")
    finite = ser[np.isfinite(ser)]
    out = pd.Series(np.nan, index=ser.index, dtype=float)
    if finite.empty:
        return out.fillna(0.0)
    lo = float(finite.min())
    hi = float(finite.max())
    if abs(hi - lo) <= 1e-12:
        out.loc[ser.index] = 1.0
        return out.fillna(0.0)
    scaled = (ser - lo) / (hi - lo)
    out.loc[scaled.index] = scaled
    if not higher_is_better:
        out = 1.0 - out
    return out.fillna(0.0)


def build_stepd_tag(restriction_name, horizon_hours, trim_eps, mode="forward", learner="lightgbm"):
    horizon_label = str(horizon_hours).replace(".", "p")
    trim_label = str(trim_eps).replace(".", "p")
    return f"{restriction_name}__h{horizon_label}__trim{trim_label}__{mode}__{learner}"


def estimate_crossfit_propensity(X, D, groups, learner="lightgbm", n_splits=5):
    X = X.reset_index(drop=True).copy()
    D = np.asarray(D).astype(int)
    groups = np.asarray(groups)
    unique_groups = int(pd.Series(groups).nunique())
    folds = min(n_splits, unique_groups)
    if folds < 2:
        raise ValueError("Need at least two user groups to estimate cross-fit propensity.")
    splitter = GroupKFold(n_splits=folds)
    phat = np.zeros(len(X), dtype=np.float64)
    for tr_idx, te_idx in splitter.split(X, D, groups=groups):
        model = fit_binary_model(
            X.iloc[tr_idx],
            D[tr_idx],
            learner=learner,
            lightgbm_params=DEFAULT_BENCHMARK_LIGHTGBM_PARAMS,
        )
        phat[te_idx] = predict_binary_model(model, X.iloc[te_idx], learner=learner)
    return np.clip(phat, 1e-6, 1 - 1e-6)


def apply_sample_restrictions(
    df_clean,
    X_df,
    *,
    min_prior_impressions_7d=None,
    min_prior_clicks_7d=None,
    active_user_only=False,
    top_k_campaigns=None,
    min_campaign_exposures=None,
    exclude_extreme_propensity=None,
    exclude_near_conversion_clicks=None,
    horizon_hours=24,
    learner="lightgbm",
    n_splits=5,
):
    df_local = df_clean.reset_index(drop=True).copy()
    X_local = X_df.reset_index(drop=True).copy()
    if len(df_local) != len(X_local):
        raise ValueError("df_clean and X_df must be aligned before applying restrictions.")

    base_campaign_counts = df_local["campaign"].astype(str).value_counts()
    D_local = (pd.to_numeric(df_local["click"], errors="coerce").fillna(0) > 0).astype(int).to_numpy()
    mask = np.ones(len(df_local), dtype=bool)
    trace = []

    def _apply(rule_name, rule_mask, description):
        nonlocal mask
        rule_mask = np.asarray(rule_mask, dtype=bool)
        if len(rule_mask) != len(mask):
            raise ValueError(f"Restriction {rule_name} has length mismatch.")
        mask = mask & rule_mask
        trace.append(
            {
                "rule_name": rule_name,
                "description": description,
                "n_after": int(mask.sum()),
                "keep_rate_after": float(mask.mean()),
            }
        )

    if active_user_only:
        if "imp_cnt_7d_pre" in X_local.columns:
            active_mask = pd.to_numeric(X_local["imp_cnt_7d_pre"], errors="coerce").fillna(0) >= 1
            active_desc = "Require at least one prior impression in the past 7d."
        else:
            user_total_rows = df_local.groupby("uid")["uid"].transform("size")
            active_mask = user_total_rows >= 2
            active_desc = "Fallback active-user rule: require at least two rows in the cleaned sample."
        _apply("active_user_only", active_mask, active_desc)

    if min_prior_impressions_7d is not None:
        if "imp_cnt_7d_pre" not in X_local.columns:
            raise KeyError("imp_cnt_7d_pre is required for min_prior_impressions_7d restriction.")
        imp_mask = pd.to_numeric(X_local["imp_cnt_7d_pre"], errors="coerce").fillna(0) >= float(min_prior_impressions_7d)
        _apply(
            "min_prior_impressions_7d",
            imp_mask,
            f"Require imp_cnt_7d_pre >= {min_prior_impressions_7d}.",
        )

    if min_prior_clicks_7d is not None:
        if "clk_cnt_7d_pre" not in X_local.columns:
            raise KeyError("clk_cnt_7d_pre is required for min_prior_clicks_7d restriction.")
        clk_mask = pd.to_numeric(X_local["clk_cnt_7d_pre"], errors="coerce").fillna(0) >= float(min_prior_clicks_7d)
        _apply(
            "min_prior_clicks_7d",
            clk_mask,
            f"Require clk_cnt_7d_pre >= {min_prior_clicks_7d}.",
        )

    if min_campaign_exposures is not None:
        allowed = set(base_campaign_counts.loc[base_campaign_counts >= int(min_campaign_exposures)].index.astype(str))
        campaign_mask = df_local["campaign"].astype(str).isin(allowed)
        _apply(
            "min_campaign_exposures",
            campaign_mask,
            f"Keep campaigns with at least {int(min_campaign_exposures)} exposures in the baseline cleaned sample.",
        )

    if top_k_campaigns is not None:
        top_campaigns = set(base_campaign_counts.head(int(top_k_campaigns)).index.astype(str))
        top_mask = df_local["campaign"].astype(str).isin(top_campaigns)
        _apply(
            "top_k_campaigns",
            top_mask,
            f"Keep only the top {int(top_k_campaigns)} campaigns by exposure count in the baseline cleaned sample.",
        )

    near_conv_hours = None if exclude_near_conversion_clicks is None else float(exclude_near_conversion_clicks)
    if near_conv_hours is not None:
        if "next_conv_ts_sec" not in df_local.columns:
            raise KeyError("next_conv_ts_sec is required for near-conversion exclusion.")
        conv_gap_sec = (
            pd.to_numeric(df_local["next_conv_ts_sec"], errors="coerce")
            - pd.to_numeric(df_local["ts_sec"], errors="coerce")
        )
        near_conv_mask = ~(
            conv_gap_sec.notna()
            & (conv_gap_sec > 0)
            & (conv_gap_sec <= near_conv_hours * 3600.0)
        )
        _apply(
            "exclude_near_conversion_clicks",
            near_conv_mask,
            f"Exclude impressions occurring within {near_conv_hours} hour(s) before the next observed conversion timestamp.",
        )

    overlap_keep_rate = np.nan
    if exclude_extreme_propensity is not None:
        eps = float(exclude_extreme_propensity)
        idx = np.flatnonzero(mask)
        if idx.size == 0:
            raise ValueError("No rows remain before extreme-propensity restriction.")
        phat = estimate_crossfit_propensity(
            X_local.iloc[idx].reset_index(drop=True),
            D_local[idx],
            df_local.iloc[idx]["uid"].to_numpy(),
            learner=learner,
            n_splits=n_splits,
        )
        overlap_mask_sub = (phat > eps) & (phat < 1 - eps)
        overlap_mask = np.zeros(len(df_local), dtype=bool)
        overlap_mask[idx] = overlap_mask_sub
        overlap_keep_rate = float(overlap_mask_sub.mean()) if overlap_mask_sub.size else np.nan
        _apply(
            "exclude_extreme_propensity",
            overlap_mask | (~mask),
            f"Pre-trim observations with cross-fit propensity outside ({eps}, {1 - eps}).",
        )

    restricted_df = df_local.loc[mask].reset_index(drop=True).copy()
    restricted_X = X_local.loc[mask].reset_index(drop=True).copy()

    manifest = {
        "min_prior_impressions_7d": _safe_scalar(min_prior_impressions_7d),
        "min_prior_clicks_7d": _safe_scalar(min_prior_clicks_7d),
        "active_user_only": bool(active_user_only),
        "top_k_campaigns": _safe_scalar(top_k_campaigns),
        "min_campaign_exposures": _safe_scalar(min_campaign_exposures),
        "exclude_extreme_propensity": _safe_scalar(exclude_extreme_propensity),
        "exclude_near_conversion_clicks_hours": _safe_scalar(near_conv_hours),
        "overlap_keep_rate_pretrim": overlap_keep_rate,
        "horizon_hours": float(horizon_hours),
        "n_input": int(len(df_local)),
        "n_output": int(mask.sum()),
        "keep_rate": float(mask.mean()),
        "n_users_input": int(df_local["uid"].nunique()),
        "n_users_output": int(restricted_df["uid"].nunique()) if not restricted_df.empty else 0,
        "n_campaigns_input": int(df_local["campaign"].astype(str).nunique()),
        "n_campaigns_output": int(restricted_df["campaign"].astype(str).nunique()) if not restricted_df.empty else 0,
        "applied_rules": trace,
    }

    return {
        "mask": mask,
        "index": np.flatnonzero(mask).astype(np.int64),
        "df": restricted_df,
        "X": restricted_X,
        "manifest": manifest,
    }


def run_plr_dml(
    X=None,
    df_in=None,
    outcome_hours=24.0,
    trim_eps=0.01,
    learner="lightgbm",
    outcome_mode="forward",
    n_splits=5,
    cache_key=None,
):
    if cache_key is not None and cache_key in STEPD_PLR_CACHE:
        return STEPD_PLR_CACHE[cache_key]

    if X is None:
        if "X_df" not in globals():
            raise ValueError("X not provided and global X_df is not available.")
        X = X_df
    if df_in is None:
        if "df_clean" not in globals():
            raise ValueError("df_in not provided and global df_clean is not available.")
        df_in = df_clean

    if len(X) != len(df_in):
        raise ValueError(f"X length {len(X)} does not match df_in length {len(df_in)}.")

    df_in = df_in.reset_index(drop=True).copy()
    X = X.reset_index(drop=True).copy()
    eligible_mask = get_outcome_eligibility_mask(df_in, horizon_hours=outcome_hours, mode=outcome_mode)
    if eligible_mask.sum() == 0:
        raise ValueError(f"No eligible rows left for outcome_hours={outcome_hours}, mode={outcome_mode}.")

    df_local = df_in.loc[eligible_mask].reset_index(drop=True).copy()
    X_local = X.loc[eligible_mask].reset_index(drop=True).copy()

    n = len(df_local)
    groups = df_local["uid"].to_numpy()
    unique_groups = int(pd.Series(groups).nunique())
    folds = min(n_splits, unique_groups)
    if folds < 2:
        raise ValueError("Impression benchmark requires at least two user groups for GroupKFold.")

    D_local = pd.to_numeric(df_local["click"], errors="coerce").fillna(0).astype(int).to_numpy()
    D_local = (D_local > 0).astype(int)
    Y_local = get_outcome_vector(df_local, horizon_hours=outcome_hours, mode=outcome_mode).astype(int)

    print(f"[DML] outcome_hours={outcome_hours}, trim_eps={trim_eps}, learner={learner}, mode={outcome_mode}")
    print("eligible rows:", int(eligible_mask.sum()), "of", len(df_in))
    print("n:", n, "D mean:", float(D_local.mean()), "Y mean:", float(Y_local.mean()))
    if np.unique(D_local).size < 2:
        raise ValueError("Treatment D has only one class. Cannot estimate ATE.")

    splitter = GroupKFold(n_splits=folds)
    mhat = np.zeros(n, dtype=np.float64)
    ghat = np.zeros(n, dtype=np.float64)
    auc_m, auc_g, ll_m, ll_g = [], [], [], []

    for fold, (tr_idx, te_idx) in enumerate(splitter.split(X_local, Y_local, groups=groups), start=1):
        X_tr, X_te = X_local.iloc[tr_idx], X_local.iloc[te_idx]
        D_tr, D_te = D_local[tr_idx], D_local[te_idx]
        Y_tr, Y_te = Y_local[tr_idx], Y_local[te_idx]

        pm_model = fit_binary_model(
            X_tr,
            D_tr,
            learner=learner,
            lightgbm_params=DEFAULT_BENCHMARK_LIGHTGBM_PARAMS,
        )
        pg_model = fit_binary_model(
            X_tr,
            Y_tr,
            learner=learner,
            lightgbm_params=DEFAULT_BENCHMARK_LIGHTGBM_PARAMS,
        )

        pm = predict_binary_model(pm_model, X_te, learner=learner)
        pg = predict_binary_model(pg_model, X_te, learner=learner)

        mhat[te_idx] = pm
        ghat[te_idx] = pg

        if np.unique(D_te).size == 2:
            auc_m.append(float(roc_auc_score(D_te, pm)))
            ll_m.append(float(log_loss(D_te, pm)))
        if np.unique(Y_te).size == 2:
            auc_g.append(float(roc_auc_score(Y_te, pg)))
            ll_g.append(float(log_loss(Y_te, pg)))

        print(f"Fold {fold}/{folds}: mhat_mean={pm.mean():.4f}, ghat_mean={pg.mean():.4f}")

    keep = (mhat > trim_eps) & (mhat < 1 - trim_eps)
    if keep.sum() == 0:
        raise ValueError("No observations left after trimming.")

    Yk = Y_local[keep].astype(np.float64)
    Dk = D_local[keep].astype(np.float64)
    mk = np.clip(mhat[keep], trim_eps, 1 - trim_eps).astype(np.float64)
    gk = ghat[keep].astype(np.float64)

    D_tilde = Dk - mk
    Y_tilde = Yk - gk
    den = np.mean(D_tilde ** 2)
    if den <= 1e-12:
        raise ValueError("Denominator too small after trimming (overlap problem).")

    theta_hat = float(np.mean(D_tilde * Y_tilde) / den)
    resid = Y_tilde - theta_hat * D_tilde
    influence = (D_tilde * resid) / den

    se_cluster, group_count = cluster_se_from_if(influence, df_local.loc[keep, "uid"].to_numpy())
    ci_lo = theta_hat - 1.96 * se_cluster if np.isfinite(se_cluster) else np.nan
    ci_hi = theta_hat + 1.96 * se_cluster if np.isfinite(se_cluster) else np.nan

    result = {
        "theta_hat": theta_hat,
        "se_cluster": se_cluster,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "t_stat": theta_hat / se_cluster if np.isfinite(se_cluster) and se_cluster > 0 else np.nan,
        "keep_rate": float(keep.mean()),
        "n_eff": int(keep.sum()),
        "clusters": int(group_count),
        "propensity_mean": float(np.mean(mhat)),
        "propensity_min": float(np.min(mhat)),
        "propensity_max": float(np.max(mhat)),
        "propensity_p01": float(np.quantile(mhat, 0.01)),
        "propensity_p99": float(np.quantile(mhat, 0.99)),
        "propensity_auc": float(np.mean(auc_m)) if auc_m else np.nan,
        "propensity_logloss": float(np.mean(ll_m)) if ll_m else np.nan,
        "outcome_auc": float(np.mean(auc_g)) if auc_g else np.nan,
        "outcome_logloss": float(np.mean(ll_g)) if ll_g else np.nan,
        "auc_m_list": auc_m,
        "auc_g_list": auc_g,
        "ll_m_list": ll_m,
        "ll_g_list": ll_g,
        "mhat": mhat,
        "ghat": ghat,
        "keep": keep,
        "Y": Y_local,
        "D": D_local,
        "row_index": np.flatnonzero(eligible_mask).astype(np.int64),
        "eligible_rows": int(eligible_mask.sum()),
        "eligible_rate": float(np.mean(eligible_mask)),
        "y_mean": float(np.mean(Y_local)),
        "d_mean": float(np.mean(D_local)),
        "outcome_hours": float(outcome_hours),
        "outcome_mode": outcome_mode,
    }

    print("\nNuisance diagnostics (mean over folds):")
    print(" Propensity AUC:", result["propensity_auc"], " LogLoss:", result["propensity_logloss"])
    print(" Outcome AUC:   ", result["outcome_auc"], " LogLoss:", result["outcome_logloss"])

    print(f"\nOverlap keep rate (eps={trim_eps}): {result['keep_rate']:.3f}")
    print(pd.Series(mhat).describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]))

    print("\n=== PLR-DML ATE ===")
    print("theta_hat:", theta_hat)
    print("\n=== Corrected Clustered SE (uid) ===")
    print("se_cluster:", se_cluster)
    print("t-stat:", result["t_stat"])
    print("clusters:", group_count, " n_eff:", int(keep.sum()))
    print("95% CI:", (ci_lo, ci_hi))

    if cache_key is not None:
        STEPD_PLR_CACHE[cache_key] = result
    return result


def run_step_d_benchmark(
    df_in,
    X_in,
    *,
    outcome_hours,
    trim_eps,
    learner="lightgbm",
    mode="forward",
    tag="",
    restriction_name="baseline",
    restriction_manifest=None,
    n_splits=5,
    run_placebo=True,
    include_data=False,
):
    benchmark_key = tag or build_stepd_tag(restriction_name, outcome_hours, trim_eps, mode=mode, learner=learner)
    cached = STEPD_BENCHMARK_CACHE.get(benchmark_key)
    if cached is not None and (not include_data or ("df_input" in cached and "X_input" in cached)):
        return cached

    forward_res = run_plr_dml(
        X=X_in,
        df_in=df_in,
        outcome_hours=outcome_hours,
        trim_eps=trim_eps,
        learner=learner,
        outcome_mode=mode,
        n_splits=n_splits,
        cache_key=f"{benchmark_key}__forward",
    )

    placebo_res = None
    if run_placebo and mode == "forward":
        placebo_res = run_plr_dml(
            X=X_in,
            df_in=df_in,
            outcome_hours=outcome_hours,
            trim_eps=trim_eps,
            learner=learner,
            outcome_mode="placebo",
            n_splits=n_splits,
            cache_key=f"{benchmark_key}__placebo",
        )

    summary = {
        "tag": benchmark_key,
        "restriction_name": restriction_name,
        "horizon_hours": float(outcome_hours),
        "trim_eps": float(trim_eps),
        "learner": learner,
        "mode": mode,
        "sample_rows": int(len(df_in)),
        "sample_users": int(pd.Series(df_in["uid"]).nunique()),
        "sample_campaigns": int(df_in["campaign"].astype(str).nunique()),
        "n": int(forward_res["eligible_rows"]),
        "n_eff": int(forward_res["n_eff"]),
        "clusters": int(forward_res["clusters"]),
        "d_mean": float(forward_res["d_mean"]),
        "y_mean": float(forward_res["y_mean"]),
        "theta_hat": float(forward_res["theta_hat"]),
        "se_cluster": float(forward_res["se_cluster"]),
        "ci_low": float(forward_res["ci_lo"]),
        "ci_high": float(forward_res["ci_hi"]),
        "t_stat": float(forward_res["t_stat"]),
        "keep_rate": float(forward_res["keep_rate"]),
        "eligible_rate": float(forward_res["eligible_rate"]),
        "propensity_mean": float(forward_res["propensity_mean"]),
        "propensity_min": float(forward_res["propensity_min"]),
        "propensity_max": float(forward_res["propensity_max"]),
        "propensity_p01": float(forward_res["propensity_p01"]),
        "propensity_p99": float(forward_res["propensity_p99"]),
        "propensity_auc": float(forward_res["propensity_auc"]),
        "propensity_logloss": float(forward_res["propensity_logloss"]),
        "outcome_auc": float(forward_res["outcome_auc"]),
        "outcome_logloss": float(forward_res["outcome_logloss"]),
        "placebo_theta_hat": np.nan,
        "placebo_se": np.nan,
        "placebo_ci_low": np.nan,
        "placebo_ci_high": np.nan,
        "placebo_t_stat": np.nan,
        "placebo_keep_rate": np.nan,
        "placebo_n": np.nan,
        "placebo_y_mean": np.nan,
    }

    if restriction_manifest is not None:
        summary.update(
            {
                "sample_restriction_keep_rate": float(restriction_manifest.get("keep_rate", np.nan)),
                "sample_restriction_n_output": int(restriction_manifest.get("n_output", len(df_in))),
                "sample_restriction_n_users_output": int(restriction_manifest.get("n_users_output", pd.Series(df_in["uid"]).nunique())),
                "sample_restriction_n_campaigns_output": int(
                    restriction_manifest.get("n_campaigns_output", df_in["campaign"].astype(str).nunique())
                ),
            }
        )

    if placebo_res is not None:
        summary.update(
            {
                "placebo_theta_hat": float(placebo_res["theta_hat"]),
                "placebo_se": float(placebo_res["se_cluster"]),
                "placebo_ci_low": float(placebo_res["ci_lo"]),
                "placebo_ci_high": float(placebo_res["ci_hi"]),
                "placebo_t_stat": float(placebo_res["t_stat"]),
                "placebo_keep_rate": float(placebo_res["keep_rate"]),
                "placebo_n": int(placebo_res["eligible_rows"]),
                "placebo_y_mean": float(placebo_res["y_mean"]),
            }
        )

    benchmark = {
        "tag": benchmark_key,
        "summary": summary,
        "forward_result": forward_res,
        "placebo_result": placebo_res,
        "restriction_manifest": restriction_manifest,
    }
    if include_data:
        benchmark["df_input"] = df_in.reset_index(drop=True).copy()
        benchmark["X_input"] = X_in.reset_index(drop=True).copy()

    STEPD_BENCHMARK_CACHE[benchmark_key] = benchmark
    return benchmark


def register_stepd_payload(payload_key, benchmark, label=None):
    if "df_input" not in benchmark or "X_input" not in benchmark:
        raise ValueError("Benchmark must be run with include_data=True before registering a Step D payload.")

    forward_res = benchmark["forward_result"]
    eligible_idx = np.asarray(forward_res["row_index"], dtype=np.int64)
    df_eligible = benchmark["df_input"].iloc[eligible_idx].reset_index(drop=True).copy()
    X_eligible = benchmark["X_input"].iloc[eligible_idx].reset_index(drop=True).copy()
    trim_eps = float(benchmark["summary"].get("trim_eps", 0.01))

    payload = {
        "key": payload_key,
        "label": label or benchmark["summary"].get("restriction_name", payload_key),
        "summary": dict(benchmark["summary"]),
        "manifest": dict(benchmark.get("restriction_manifest") or {}),
        "df": df_eligible,
        "X": X_eligible,
        "D": np.asarray(forward_res["D"]).astype(int),
        "Y": np.asarray(forward_res["Y"]).astype(int),
        "keep": np.asarray(forward_res["keep"]).astype(bool),
        "mhat": np.clip(np.asarray(forward_res["mhat"], dtype=float), max(trim_eps, 1e-6), 1 - max(trim_eps, 1e-6)),
        "ghat": np.asarray(forward_res["ghat"], dtype=float),
        "Y_placebo_same_sample": build_placebo_outcome(
            df_eligible,
            horizon_hours=float(benchmark["summary"].get("horizon_hours", 24.0)),
        ).astype(int),
    }
    STEPD_PAYLOADS[payload_key] = payload
    return payload


def get_stepd_payload(source=None):
    requested = source or globals().get("STEPD_DOWNSTREAM_SOURCE", "auto")
    if requested == "auto":
        requested = "final" if "final" in STEPD_PAYLOADS else "baseline"
    if requested == "final" and "final" in STEPD_PAYLOADS:
        return STEPD_PAYLOADS["final"]
    if requested in STEPD_PAYLOADS:
        return STEPD_PAYLOADS[requested]
    if "baseline" in STEPD_PAYLOADS:
        return STEPD_PAYLOADS["baseline"]
    raise KeyError(f"Requested Step D payload '{requested}' is not available.")


baseline_restriction = apply_sample_restrictions(df_clean, X_df, horizon_hours=24.0)
baseline_stepd_benchmark_24h = run_step_d_benchmark(
    baseline_restriction["df"],
    baseline_restriction["X"],
    outcome_hours=24.0,
    trim_eps=0.01,
    learner="lightgbm",
    mode="forward",
    tag="baseline__h24__trim0p01__forward__lightgbm",
    restriction_name="baseline",
    restriction_manifest=baseline_restriction["manifest"],
    n_splits=5,
    run_placebo=True,
    include_data=True,
)

main_result_24h = baseline_stepd_benchmark_24h["forward_result"]
stepd_baseline_summary_24h = baseline_stepd_benchmark_24h["summary"]
baseline_stepd_payload = register_stepd_payload(
    "baseline",
    baseline_stepd_benchmark_24h,
    label="Baseline Step D benchmark (24h, trim 0.01)",
)
STEPD_DOWNSTREAM_SOURCE = globals().get("STEPD_DOWNSTREAM_SOURCE", "baseline")

theta_hat = float(main_result_24h["theta_hat"])
se_cluster = float(main_result_24h["se_cluster"])
ci_low_24h = float(main_result_24h["ci_lo"])
ci_high_24h = float(main_result_24h["ci_hi"])
t_stat_24h = float(main_result_24h["t_stat"])
analysis_row_idx = np.asarray(main_result_24h["row_index"], dtype=np.int64)
df_main_24h = baseline_stepd_payload["df"].copy()
X_main_24h = baseline_stepd_payload["X"].copy()
keep = baseline_stepd_payload["keep"].copy()
mhat = baseline_stepd_payload["mhat"].copy()
ghat = baseline_stepd_payload["ghat"].copy()
Y = baseline_stepd_payload["Y"].copy()
D = baseline_stepd_payload["D"].copy()
Y_placebo_24h = baseline_stepd_payload["Y_placebo_same_sample"].copy()
auc_m_list = list(main_result_24h.get("auc_m_list", []))
auc_g_list = list(main_result_24h.get("auc_g_list", []))
ll_m_list = list(main_result_24h.get("ll_m_list", []))
ll_g_list = list(main_result_24h.get("ll_g_list", []))

diagnostics_main_24h = {
    k: v
    for k, v in main_result_24h.items()
    if k not in {"mhat", "ghat", "keep", "Y", "D", "auc_m_list", "auc_g_list", "ll_m_list", "ll_g_list", "row_index"}
}
diagnostics_main_24h["restriction_name"] = "baseline"
diagnostics_main_24h["placebo_theta_hat"] = float(stepd_baseline_summary_24h.get("placebo_theta_hat", np.nan))
diagnostics_main_24h["placebo_ci_low"] = float(stepd_baseline_summary_24h.get("placebo_ci_low", np.nan))
diagnostics_main_24h["placebo_ci_high"] = float(stepd_baseline_summary_24h.get("placebo_ci_high", np.nan))
diagnostics_main_24h["sample_rows_before_horizon_filter"] = int(stepd_baseline_summary_24h.get("sample_rows", len(df_clean)))
save_json(diagnostics_main_24h, MAIN_24H_JSON_PATH)
display(pd.Series(diagnostics_main_24h, name="value").to_frame())
print("Saved main 24h summary to:", MAIN_24H_JSON_PATH)

## 7. RAMP Layer 2: Diagnostic Tests

In [ ]:
# ====== Step D1: Sample Restriction Grid ======
import json
import numpy as np
import pandas as pd


def evaluate_restriction_spec(spec_name, spec):
    restricted = apply_sample_restrictions(
        df_clean,
        X_df,
        horizon_hours=24.0,
        **spec.get("restriction_kwargs", {}),
    )
    benchmark = run_step_d_benchmark(
        restricted["df"],
        restricted["X"],
        outcome_hours=24.0,
        trim_eps=float(spec.get("trim_eps", 0.01)),
        learner=spec.get("learner", "lightgbm"),
        mode=spec.get("mode", "forward"),
        tag=build_stepd_tag(
            spec_name,
            24.0,
            float(spec.get("trim_eps", 0.01)),
            mode=spec.get("mode", "forward"),
            learner=spec.get("learner", "lightgbm"),
        ),
        restriction_name=spec_name,
        restriction_manifest=restricted["manifest"],
        n_splits=5,
        run_placebo=True,
        include_data=False,
    )
    row = dict(benchmark["summary"])
    row.update(
        {
            "restriction_family": spec.get("restriction_family", "other"),
            "decision_relevant": bool(spec.get("decision_relevant", False)),
            "candidate_for_preferred": bool(spec.get("candidate_for_preferred", False)),
            "stability_group": spec.get("stability_group", spec_name),
            "notes": spec.get("notes", ""),
            "restriction_params_json": json.dumps(spec.get("restriction_kwargs", {}), sort_keys=True),
        }
    )
    return row


stepd_restriction_spec_catalog = {
    "baseline": {
        "restriction_family": "baseline",
        "trim_eps": 0.01,
        "restriction_kwargs": {},
        "notes": "Original cleaned sample with no additional restriction.",
    },
    "prior_imp_7d_ge_1": {
        "restriction_family": "prior_activity",
        "trim_eps": 0.01,
        "restriction_kwargs": {"min_prior_impressions_7d": 1},
        "notes": "Require at least one prior impression in the past 7d.",
    },
    "prior_imp_7d_ge_2": {
        "restriction_family": "prior_activity",
        "trim_eps": 0.01,
        "restriction_kwargs": {"min_prior_impressions_7d": 2},
        "notes": "Require at least two prior impressions in the past 7d.",
    },
    "prior_click_7d_ge_1": {
        "restriction_family": "prior_activity",
        "trim_eps": 0.01,
        "restriction_kwargs": {"min_prior_clicks_7d": 1},
        "notes": "Require at least one prior click in the past 7d.",
    },
    "baseline_trim_0p05": {
        "restriction_family": "overlap_trim",
        "trim_eps": 0.05,
        "restriction_kwargs": {},
        "notes": "Baseline sample with stricter overlap trimming at 0.05.",
        "stability_group": "baseline_trim",
    },
    "baseline_trim_0p10": {
        "restriction_family": "overlap_trim",
        "trim_eps": 0.10,
        "restriction_kwargs": {},
        "notes": "Baseline sample with stricter overlap trimming at 0.10.",
        "stability_group": "baseline_trim",
    },
    "top_100_campaigns": {
        "restriction_family": "campaign_relevance",
        "trim_eps": 0.01,
        "restriction_kwargs": {"top_k_campaigns": 100},
        "notes": "Restrict to the top 100 campaigns by baseline exposure count.",
        "decision_relevant": True,
    },
    "top_50_campaigns": {
        "restriction_family": "campaign_relevance",
        "trim_eps": 0.01,
        "restriction_kwargs": {"top_k_campaigns": 50},
        "notes": "Restrict to the top 50 campaigns by baseline exposure count.",
        "decision_relevant": True,
    },
    "exclude_near_conversion_1h": {
        "restriction_family": "near_conversion",
        "trim_eps": 0.01,
        "restriction_kwargs": {"exclude_near_conversion_clicks": 1.0},
        "notes": "Exclude impressions within 1 hour before the next observed conversion.",
        "decision_relevant": True,
    },
    "active_only_trim0p05": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.05,
        "restriction_kwargs": {"active_user_only": True},
        "notes": "Active users only, with trim 0.05.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_only",
    },
    "active_top100_trim0p01": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.01,
        "restriction_kwargs": {"active_user_only": True, "top_k_campaigns": 100},
        "notes": "Active users only plus the top 100 campaigns.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_top100",
    },
    "active_top100_trim0p05": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.05,
        "restriction_kwargs": {"active_user_only": True, "top_k_campaigns": 100},
        "notes": "Active users only plus the top 100 campaigns, with trim 0.05.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_top100",
    },
    "active_top50_trim0p01": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.01,
        "restriction_kwargs": {"active_user_only": True, "top_k_campaigns": 50},
        "notes": "Active users only plus the top 50 campaigns.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_top50",
    },
    "active_top50_trim0p05": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.05,
        "restriction_kwargs": {"active_user_only": True, "top_k_campaigns": 50},
        "notes": "Active users only plus the top 50 campaigns, with trim 0.05.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_top50",
    },
    "active_top100_nearconv1h_trim0p05": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.05,
        "restriction_kwargs": {
            "active_user_only": True,
            "top_k_campaigns": 100,
            "exclude_near_conversion_clicks": 1.0,
        },
        "notes": "Active users, top 100 campaigns, and a 1h near-conversion exclusion, with trim 0.05.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_top100_nearconv1h",
    },
    "active_top50_nearconv1h_trim0p05": {
        "restriction_family": "combined_decision_relevant",
        "trim_eps": 0.05,
        "restriction_kwargs": {
            "active_user_only": True,
            "top_k_campaigns": 50,
            "exclude_near_conversion_clicks": 1.0,
        },
        "notes": "Active users, top 50 campaigns, and a 1h near-conversion exclusion, with trim 0.05.",
        "decision_relevant": True,
        "candidate_for_preferred": True,
        "stability_group": "active_top50_nearconv1h",
    },
}

prior_activity_names = ["baseline", "prior_imp_7d_ge_1", "prior_imp_7d_ge_2", "prior_click_7d_ge_1"]
overlap_trim_names = ["baseline", "baseline_trim_0p05", "baseline_trim_0p10"]
campaign_relevance_names = ["baseline", "top_100_campaigns", "top_50_campaigns"]
near_conversion_names = ["baseline", "exclude_near_conversion_1h"]
combined_candidate_names = [
    "active_only_trim0p05",
    "active_top100_trim0p01",
    "active_top100_trim0p05",
    "active_top50_trim0p01",
    "active_top50_trim0p05",
    "active_top100_nearconv1h_trim0p05",
    "active_top50_nearconv1h_trim0p05",
]

prior_activity_rows = [evaluate_restriction_spec(name, stepd_restriction_spec_catalog[name]) for name in prior_activity_names]
restriction_prior_activity_df = pd.DataFrame(prior_activity_rows)

overlap_trim_rows = [evaluate_restriction_spec(name, stepd_restriction_spec_catalog[name]) for name in overlap_trim_names]
restriction_overlap_trim_df = pd.DataFrame(overlap_trim_rows)
restriction_overlap_trim_df.loc[
    restriction_overlap_trim_df["restriction_name"] == "baseline",
    "trim_eps",
] = 0.01

campaign_relevance_rows = [evaluate_restriction_spec(name, stepd_restriction_spec_catalog[name]) for name in campaign_relevance_names]
restriction_campaign_relevance_df = pd.DataFrame(campaign_relevance_rows)

near_conversion_rows = [evaluate_restriction_spec(name, stepd_restriction_spec_catalog[name]) for name in near_conversion_names]
restriction_near_conversion_exclusion_df = pd.DataFrame(near_conversion_rows)

combined_candidate_rows = [evaluate_restriction_spec(name, stepd_restriction_spec_catalog[name]) for name in combined_candidate_names]
restriction_combined_candidate_df = pd.DataFrame(combined_candidate_rows)

stepD_restriction_grid_df = (
    pd.concat(
        [
            restriction_prior_activity_df,
            restriction_overlap_trim_df,
            restriction_campaign_relevance_df,
            restriction_near_conversion_exclusion_df,
            restriction_combined_candidate_df,
        ],
        axis=0,
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(subset=["restriction_name", "horizon_hours", "trim_eps"])
    .sort_values(["restriction_family", "restriction_name", "trim_eps"])
    .reset_index(drop=True)
)

save_df(restriction_prior_activity_df, PAPER_OUTPUT_DIR / "table_restriction_prior_activity.csv")
save_df(restriction_overlap_trim_df, PAPER_OUTPUT_DIR / "table_restriction_overlap_trim_grid.csv")
save_df(restriction_campaign_relevance_df, PAPER_OUTPUT_DIR / "table_restriction_campaign_relevance.csv")
save_df(restriction_near_conversion_exclusion_df, PAPER_OUTPUT_DIR / "table_restriction_near_conversion_exclusion.csv")
save_df(stepD_restriction_grid_df, PAPER_OUTPUT_DIR / "table_stepD_restriction_grid.csv")

baseline_row = stepD_restriction_grid_df.loc[stepD_restriction_grid_df["restriction_name"] == "baseline"].iloc[0]
baseline_sample_rows = float(baseline_row["sample_rows"])
baseline_y_mean = float(baseline_row["y_mean"])
baseline_keep_rate = float(baseline_row["keep_rate"])

candidate_df = stepD_restriction_grid_df.loc[stepD_restriction_grid_df["candidate_for_preferred"]].copy()
if candidate_df.empty:
    preferred_restriction_name = "baseline"
else:
    candidate_df["sample_fraction_vs_baseline"] = candidate_df["sample_rows"] / baseline_sample_rows if baseline_sample_rows > 0 else np.nan
    candidate_df["density_ratio_vs_baseline"] = candidate_df["y_mean"] / baseline_y_mean if baseline_y_mean > 0 else np.nan
    candidate_df["placebo_abs_t"] = candidate_df["placebo_t_stat"].abs()
    candidate_df["sample_ok"] = candidate_df["sample_fraction_vs_baseline"] >= 0.20
    candidate_df["score_placebo"] = _normalize_score(-candidate_df["placebo_abs_t"], higher_is_better=True)
    candidate_df["score_density"] = _normalize_score(candidate_df["density_ratio_vs_baseline"], higher_is_better=True)
    candidate_df["score_overlap"] = _normalize_score(candidate_df["keep_rate"], higher_is_better=True)
    candidate_df["score_sample"] = _normalize_score(candidate_df["sample_fraction_vs_baseline"], higher_is_better=True)
    candidate_df["selection_score"] = (
        0.35 * candidate_df["score_placebo"]
        + 0.30 * candidate_df["score_density"]
        + 0.20 * candidate_df["score_overlap"]
        + 0.15 * candidate_df["score_sample"]
    )
    candidate_df = candidate_df.sort_values(
        ["sample_ok", "selection_score", "keep_rate", "y_mean"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    preferred_restriction_name = str(candidate_df.loc[0, "restriction_name"])

theta_stability_df = (
    stepD_restriction_grid_df.groupby("stability_group", as_index=False)
    .agg(
        n_specs=("theta_hat", "size"),
        theta_min=("theta_hat", "min"),
        theta_max=("theta_hat", "max"),
    )
)
theta_stability_df["theta_span"] = theta_stability_df["theta_max"] - theta_stability_df["theta_min"]
theta_stability_df = theta_stability_df.sort_values(["theta_span", "n_specs"], ascending=[True, False]).reset_index(drop=True)

improved_y_mean_rows = (
    stepD_restriction_grid_df.loc[
        stepD_restriction_grid_df["y_mean"] > baseline_y_mean,
        ["restriction_name", "y_mean", "sample_rows", "keep_rate"],
    ]
    .sort_values("y_mean", ascending=False)
    .head(5)
    .to_dict(orient="records")
)
placebo_best_rows = (
    stepD_restriction_grid_df.assign(placebo_abs_t=stepD_restriction_grid_df["placebo_t_stat"].abs())
    .sort_values(["placebo_abs_t", "keep_rate"], ascending=[True, False])
    .head(5)[["restriction_name", "placebo_theta_hat", "placebo_t_stat", "keep_rate", "y_mean"]]
    .to_dict(orient="records")
)
overlap_best_rows = (
    stepD_restriction_grid_df.loc[
        stepD_restriction_grid_df["keep_rate"] >= max(0.50, baseline_keep_rate - 0.10),
        ["restriction_name", "keep_rate", "y_mean", "placebo_theta_hat", "placebo_t_stat"],
    ]
    .sort_values(["keep_rate", "y_mean"], ascending=[False, False])
    .head(5)
    .to_dict(orient="records")
)

stepd_preferred_restriction_spec = stepd_restriction_spec_catalog[preferred_restriction_name]
summary_stepD_restriction_grid = {
    "baseline_restriction_name": "baseline",
    "baseline_y_mean": baseline_y_mean,
    "baseline_keep_rate": baseline_keep_rate,
    "baseline_placebo_theta_hat": float(baseline_row["placebo_theta_hat"]),
    "preferred_restriction_name": preferred_restriction_name,
    "preferred_trim_eps": float(stepd_preferred_restriction_spec.get("trim_eps", 0.01)),
    "preferred_restriction_kwargs": stepd_preferred_restriction_spec.get("restriction_kwargs", {}),
    "preferred_restriction_notes": stepd_preferred_restriction_spec.get("notes", ""),
    "y_mean_improving_restrictions": improved_y_mean_rows,
    "placebo_best_restrictions": placebo_best_rows,
    "overlap_preserving_restrictions": overlap_best_rows,
    "theta_stability_by_group": theta_stability_df.to_dict(orient="records"),
    "selection_rule": "Preferred restriction ranks high on placebo closeness to zero, denser outcome incidence, and retained overlap/sample share. Theta sign is not part of the score.",
}
save_json(summary_stepD_restriction_grid, PAPER_OUTPUT_DIR / "summary_stepD_restriction_grid.json")

display(restriction_prior_activity_df)
display(restriction_overlap_trim_df)
display(restriction_campaign_relevance_df)
display(restriction_near_conversion_exclusion_df)
display(stepD_restriction_grid_df)
print("Preferred Step D restriction candidate:", preferred_restriction_name)

## 7. RAMP Layer 2: Diagnostic Tests

In [ ]:
# ====== Step D2: Horizon Comparison Grid ======
import numpy as np
import pandas as pd

HORIZON_GRID_HOURS = [1.0, 6.0, 24.0, 168.0]

preferred_restriction_name = summary_stepD_restriction_grid.get("preferred_restriction_name", "active_top50_trim0p05")
preferred_restriction_spec = stepd_restriction_spec_catalog.get(
    preferred_restriction_name,
    stepd_restriction_spec_catalog["baseline"],
)

stepD_horizon_grid_baseline_rows = []
baseline_restricted = apply_sample_restrictions(df_clean, X_df, horizon_hours=24.0, **stepd_restriction_spec_catalog["baseline"]["restriction_kwargs"])
for horizon_hours in HORIZON_GRID_HOURS:
    benchmark = run_step_d_benchmark(
        baseline_restricted["df"],
        baseline_restricted["X"],
        outcome_hours=horizon_hours,
        trim_eps=0.01,
        learner="lightgbm",
        mode="forward",
        tag=build_stepd_tag("baseline", horizon_hours, 0.01, mode="forward", learner="lightgbm"),
        restriction_name="baseline",
        restriction_manifest=baseline_restricted["manifest"],
        n_splits=5,
        run_placebo=True,
        include_data=False,
    )
    row = dict(benchmark["summary"])
    row["sample_variant"] = "baseline"
    stepD_horizon_grid_baseline_rows.append(row)

stepD_horizon_grid_baseline_df = pd.DataFrame(stepD_horizon_grid_baseline_rows).sort_values("horizon_hours").reset_index(drop=True)
save_df(stepD_horizon_grid_baseline_df, PAPER_OUTPUT_DIR / "table_stepD_horizon_grid_baseline.csv")

stepD_horizon_grid_preferred_rows = []
preferred_restricted = apply_sample_restrictions(
    df_clean,
    X_df,
    horizon_hours=24.0,
    **preferred_restriction_spec.get("restriction_kwargs", {}),
)
preferred_trim_eps = float(preferred_restriction_spec.get("trim_eps", 0.01))

for horizon_hours in HORIZON_GRID_HOURS:
    benchmark = run_step_d_benchmark(
        preferred_restricted["df"],
        preferred_restricted["X"],
        outcome_hours=horizon_hours,
        trim_eps=preferred_trim_eps,
        learner="lightgbm",
        mode="forward",
        tag=build_stepd_tag(preferred_restriction_name, horizon_hours, preferred_trim_eps, mode="forward", learner="lightgbm"),
        restriction_name=preferred_restriction_name,
        restriction_manifest=preferred_restricted["manifest"],
        n_splits=5,
        run_placebo=True,
        include_data=False,
    )
    row = dict(benchmark["summary"])
    row["sample_variant"] = "preferred"
    stepD_horizon_grid_preferred_rows.append(row)

stepD_horizon_grid_preferred_df = pd.DataFrame(stepD_horizon_grid_preferred_rows).sort_values("horizon_hours").reset_index(drop=True)
save_df(stepD_horizon_grid_preferred_df, PAPER_OUTPUT_DIR / "table_stepD_horizon_grid_preferred.csv")

horizon_compare_df = stepD_horizon_grid_preferred_df.merge(
    stepD_horizon_grid_baseline_df[["horizon_hours", "theta_hat", "keep_rate", "placebo_t_stat"]].rename(
        columns={
            "theta_hat": "baseline_theta_hat",
            "keep_rate": "baseline_keep_rate",
            "placebo_t_stat": "baseline_placebo_t_stat",
        }
    ),
    on="horizon_hours",
    how="left",
)

horizon_compare_df["placebo_abs_t"] = horizon_compare_df["placebo_t_stat"].abs()
horizon_compare_df["theta_gap_vs_baseline"] = (horizon_compare_df["theta_hat"] - horizon_compare_df["baseline_theta_hat"]).abs()
horizon_compare_df["density_score"] = _normalize_score(horizon_compare_df["y_mean"], higher_is_better=True)
horizon_compare_df["placebo_score"] = _normalize_score(-horizon_compare_df["placebo_abs_t"], higher_is_better=True)
horizon_compare_df["overlap_score"] = _normalize_score(horizon_compare_df["keep_rate"], higher_is_better=True)
horizon_compare_df["stability_score"] = _normalize_score(-horizon_compare_df["theta_gap_vs_baseline"], higher_is_better=True)
horizon_compare_df["business_score"] = horizon_compare_df["horizon_hours"].map(
    {1.0: 0.35, 6.0: 0.65, 24.0: 1.0, 168.0: 0.70}
).fillna(0.50)
horizon_compare_df["selection_score"] = (
    0.30 * horizon_compare_df["density_score"]
    + 0.25 * horizon_compare_df["placebo_score"]
    + 0.15 * horizon_compare_df["overlap_score"]
    + 0.15 * horizon_compare_df["stability_score"]
    + 0.15 * horizon_compare_df["business_score"]
)
horizon_compare_df = horizon_compare_df.sort_values(
    ["selection_score", "business_score", "y_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

least_sparse_horizon = float(stepD_horizon_grid_preferred_df.sort_values("y_mean", ascending=False).iloc[0]["horizon_hours"])
best_placebo_horizon = float(
    stepD_horizon_grid_preferred_df.assign(placebo_abs_t=stepD_horizon_grid_preferred_df["placebo_t_stat"].abs())
    .sort_values(["placebo_abs_t", "keep_rate"], ascending=[True, False])
    .iloc[0]["horizon_hours"]
)
best_overlap_horizon = float(stepD_horizon_grid_preferred_df.sort_values("keep_rate", ascending=False).iloc[0]["horizon_hours"])
recommended_main_horizon_hours = float(horizon_compare_df.iloc[0]["horizon_hours"])

summary_stepD_horizon_selection = {
    "preferred_restriction_name": preferred_restriction_name,
    "preferred_trim_eps": preferred_trim_eps,
    "least_sparse_horizon_hours": least_sparse_horizon,
    "best_placebo_horizon_hours": best_placebo_horizon,
    "best_overlap_horizon_hours": best_overlap_horizon,
    "recommended_main_horizon_hours": recommended_main_horizon_hours,
    "selection_rule": "The recommended horizon prioritizes denser outcomes, placebo closeness to zero, retained overlap, stability relative to the baseline sample, and decision relevance. Theta sign is not used for selection.",
    "horizon_score_table": horizon_compare_df[
        [
            "horizon_hours",
            "selection_score",
            "y_mean",
            "keep_rate",
            "placebo_theta_hat",
            "placebo_t_stat",
            "theta_gap_vs_baseline",
            "business_score",
        ]
    ].to_dict(orient="records"),
}
save_json(summary_stepD_horizon_selection, PAPER_OUTPUT_DIR / "summary_stepD_horizon_selection.json")

display(stepD_horizon_grid_baseline_df)
display(stepD_horizon_grid_preferred_df)
display(horizon_compare_df[["horizon_hours", "selection_score", "y_mean", "keep_rate", "placebo_t_stat", "theta_gap_vs_baseline"]])
print("Recommended Step D main horizon:", recommended_main_horizon_hours)

## 6. RAMP Layer 1: Selection-Adjusted Benchmark

In [ ]:
# ====== Step D3: Select Preferred Step D Spec ======
import numpy as np
import pandas as pd

final_restriction_name = summary_stepD_restriction_grid.get("preferred_restriction_name", "baseline")
final_restriction_spec = stepd_restriction_spec_catalog.get(final_restriction_name, stepd_restriction_spec_catalog["baseline"])
final_horizon_hours = float(summary_stepD_horizon_selection.get("recommended_main_horizon_hours", 24.0))
final_trim_eps = float(final_restriction_spec.get("trim_eps", 0.01))
final_learner = str(final_restriction_spec.get("learner", "lightgbm"))
final_mode = str(final_restriction_spec.get("mode", "forward"))

final_restricted = apply_sample_restrictions(
    df_clean,
    X_df,
    horizon_hours=final_horizon_hours,
    **final_restriction_spec.get("restriction_kwargs", {}),
)
final_benchmark = run_step_d_benchmark(
    final_restricted["df"],
    final_restricted["X"],
    outcome_hours=final_horizon_hours,
    trim_eps=final_trim_eps,
    learner=final_learner,
    mode=final_mode,
    tag="final_selected_stepd_spec",
    restriction_name=final_restriction_name,
    restriction_manifest=final_restricted["manifest"],
    n_splits=5,
    run_placebo=True,
    include_data=True,
)
final_payload = register_stepd_payload(
    "final",
    final_benchmark,
    label=f"{final_restriction_name}, {final_horizon_hours:g}h, trim {final_trim_eps:.2f}",
)

stepD_final_spec = {
    "restriction_name": final_restriction_name,
    "restriction_kwargs": final_restriction_spec.get("restriction_kwargs", {}),
    "restriction_notes": final_restriction_spec.get("notes", ""),
    "horizon_hours": final_horizon_hours,
    "trim_eps": final_trim_eps,
    "learner": final_learner,
    "mode": final_mode,
    "selection_basis": {
        "restriction_summary": summary_stepD_restriction_grid,
        "horizon_summary": summary_stepD_horizon_selection,
    },
    "diagnostics_snapshot": {
        "sample_rows": int(final_benchmark["summary"]["sample_rows"]),
        "n": int(final_benchmark["summary"]["n"]),
        "n_eff": int(final_benchmark["summary"]["n_eff"]),
        "y_mean": float(final_benchmark["summary"]["y_mean"]),
        "d_mean": float(final_benchmark["summary"]["d_mean"]),
        "theta_hat": float(final_benchmark["summary"]["theta_hat"]),
        "se_cluster": float(final_benchmark["summary"]["se_cluster"]),
        "ci_low": float(final_benchmark["summary"]["ci_low"]),
        "ci_high": float(final_benchmark["summary"]["ci_high"]),
        "keep_rate": float(final_benchmark["summary"]["keep_rate"]),
        "placebo_theta_hat": float(final_benchmark["summary"]["placebo_theta_hat"]),
        "placebo_ci_low": float(final_benchmark["summary"]["placebo_ci_low"]),
        "placebo_ci_high": float(final_benchmark["summary"]["placebo_ci_high"]),
    },
    "downstream_payload_key": "final",
}
save_json(stepD_final_spec, PAPER_OUTPUT_DIR / "stepD_final_spec.json")

STEPD_DOWNSTREAM_SOURCE = "final"

stepD_final_spec_df = pd.DataFrame(
    [
        {
            "restriction_name": stepD_final_spec["restriction_name"],
            "horizon_hours": stepD_final_spec["horizon_hours"],
            "trim_eps": stepD_final_spec["trim_eps"],
            "learner": stepD_final_spec["learner"],
            "mode": stepD_final_spec["mode"],
            "theta_hat": stepD_final_spec["diagnostics_snapshot"]["theta_hat"],
            "placebo_theta_hat": stepD_final_spec["diagnostics_snapshot"]["placebo_theta_hat"],
            "keep_rate": stepD_final_spec["diagnostics_snapshot"]["keep_rate"],
        }
    ]
)

display(stepD_final_spec_df)
print("Saved final Step D spec to:", PAPER_OUTPUT_DIR / "stepD_final_spec.json")
print("Downstream Step G/H source is now set to:", STEPD_DOWNSTREAM_SOURCE)
print("Rerun Step G/H after Step D spec is finalized. Set STEPD_DOWNSTREAM_SOURCE='baseline' to use the original benchmark.")

## 6. RAMP Layer 1: Selection-Adjusted Benchmark

This standardized layer aligns the currently selected Step D objects into a single `dml_analysis_df` for downstream DML-based analysis. It prefers `stepD_final_spec.json` when available and otherwise falls back to the current baseline Step D payload. If Step D spec changes, rerun Step G2 and Step H2.


In [ ]:
# ====== Step D4: Build standardized Step D analysis frame ======
from pathlib import Path
import json
import numpy as np
import pandas as pd

assert "get_stepd_payload" in globals(), "get_stepd_payload not found. Run Step D first."
assert "apply_sample_restrictions" in globals(), "apply_sample_restrictions not found. Run Step D first."
assert "run_step_d_benchmark" in globals(), "run_step_d_benchmark not found. Run Step D first."
assert "register_stepd_payload" in globals(), "register_stepd_payload not found. Run Step D first."


def dml_to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): dml_to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [dml_to_builtin(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, pd.Series):
        return [dml_to_builtin(v) for v in obj.tolist()]
    if isinstance(obj, np.ndarray):
        return [dml_to_builtin(v) for v in obj.tolist()]
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    return obj


def _load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)


def _resolve_stepd_payload_for_d4():
    spec_path = Path(PAPER_OUTPUT_DIR) / "stepD_final_spec.json"
    spec_obj = _load_json_if_exists(spec_path)
    if spec_obj is None and "stepD_final_spec" in globals() and isinstance(stepD_final_spec, dict):
        spec_obj = dict(stepD_final_spec)

    if spec_obj is not None:
        preferred_key = str(spec_obj.get("downstream_payload_key") or "final")
        if preferred_key in globals().get("STEPD_PAYLOADS", {}):
            payload = get_stepd_payload(preferred_key)
            return payload, spec_obj, "preferred_payload_in_memory"
        if preferred_key == "final" and "final" in globals().get("STEPD_PAYLOADS", {}):
            payload = get_stepd_payload("final")
            return payload, spec_obj, "preferred_payload_in_memory"

        restriction_name = str(spec_obj.get("restriction_name", "baseline"))
        restriction_kwargs = dict(spec_obj.get("restriction_kwargs") or {})
        outcome_hours = float(spec_obj.get("horizon_hours", 24.0))
        trim_eps = float(spec_obj.get("trim_eps", 0.01))
        learner = str(spec_obj.get("learner", "lightgbm"))
        outcome_mode = str(spec_obj.get("mode", "forward"))

        rebuilt = apply_sample_restrictions(
            df_clean,
            X_df,
            horizon_hours=outcome_hours,
            **restriction_kwargs,
        )
        rebuilt_benchmark = run_step_d_benchmark(
            rebuilt["df"],
            rebuilt["X"],
            outcome_hours=outcome_hours,
            trim_eps=trim_eps,
            learner=learner,
            mode=outcome_mode,
            tag=f"{build_stepd_tag(restriction_name, outcome_hours, trim_eps, mode=outcome_mode, learner=learner)}__d4_reload",
            restriction_name=restriction_name,
            restriction_manifest=rebuilt["manifest"],
            n_splits=5,
            run_placebo=True,
            include_data=True,
        )
        payload = register_stepd_payload(
            preferred_key,
            rebuilt_benchmark,
            label=f"Reloaded Step D preferred spec ({restriction_name}, {outcome_hours:g}h, trim {trim_eps:.2f})",
        )
        return payload, spec_obj, "reconstructed_from_stepD_final_spec_json"

    payload = get_stepd_payload(globals().get("STEPD_DOWNSTREAM_SOURCE", "baseline"))
    return payload, None, "current_notebook_payload"


selected_stepd_payload, selected_stepd_spec, selected_stepd_resolution = _resolve_stepd_payload_for_d4()
selected_stepd_summary = dict(selected_stepd_payload.get("summary") or {})

dml_analysis_base_df = selected_stepd_payload["df"].copy().reset_index(drop=True)
dml_analysis_X_df = selected_stepd_payload.get("X", pd.DataFrame(index=dml_analysis_base_df.index)).copy().reset_index(drop=True)
dml_analysis_arrays = {
    "D": np.asarray(selected_stepd_payload["D"]).astype(int),
    "Y": np.asarray(selected_stepd_payload["Y"]).astype(int),
    "mhat": np.asarray(selected_stepd_payload["mhat"], dtype=float),
    "ghat": np.asarray(selected_stepd_payload["ghat"], dtype=float),
    "keep": np.asarray(selected_stepd_payload["keep"]).astype(bool),
}

expected_len = len(dml_analysis_base_df)
if len(dml_analysis_X_df) != expected_len:
    raise ValueError(f"Step D4 alignment error: X length {len(dml_analysis_X_df)} != df length {expected_len}.")
for arr_name, arr_value in dml_analysis_arrays.items():
    if len(arr_value) != expected_len:
        raise ValueError(f"Step D4 alignment error: {arr_name} length {len(arr_value)} != df length {expected_len}.")

dml_analysis_df_full = dml_analysis_base_df.copy().reset_index(drop=True)
for col_name, arr_value in dml_analysis_arrays.items():
    dml_analysis_df_full[col_name] = arr_value

added_feature_cols = []
for feature_col in dml_analysis_X_df.columns:
    if feature_col not in dml_analysis_df_full.columns:
        dml_analysis_df_full[feature_col] = dml_analysis_X_df[feature_col].to_numpy()
        added_feature_cols.append(feature_col)

dml_analysis_df_full["D_tilde"] = pd.to_numeric(dml_analysis_df_full["D"], errors="coerce") - pd.to_numeric(dml_analysis_df_full["mhat"], errors="coerce")
dml_analysis_df_full["Y_tilde"] = pd.to_numeric(dml_analysis_df_full["Y"], errors="coerce") - pd.to_numeric(dml_analysis_df_full["ghat"], errors="coerce")

dml_analysis_metadata = {
    "stepd_payload_key": str(selected_stepd_payload.get("key", "unknown")),
    "stepd_payload_label": str(selected_stepd_payload.get("label", "unknown")),
    "stepd_resolution": str(selected_stepd_resolution),
    "restriction_name": str(selected_stepd_summary.get("restriction_name", "baseline")),
    "outcome_hours": float(selected_stepd_summary.get("horizon_hours", 24.0)),
    "trim_eps": float(selected_stepd_summary.get("trim_eps", 0.01)),
    "outcome_mode": str(selected_stepd_summary.get("mode", "forward")),
    "keep_rate": float(selected_stepd_summary.get("keep_rate", np.nan)) if pd.notna(selected_stepd_summary.get("keep_rate", np.nan)) else np.nan,
    "spec_from_json": bool(selected_stepd_spec is not None),
    "selected_stepd_spec_path": str(Path(PAPER_OUTPUT_DIR) / "stepD_final_spec.json"),
}

for meta_col, meta_value in dml_analysis_metadata.items():
    dml_analysis_df_full[meta_col] = meta_value

dml_analysis_df = dml_analysis_df_full.loc[dml_analysis_df_full["keep"]].copy().reset_index(drop=True)

required_cols = ["uid", "campaign", "ts_sec", "D", "Y", "mhat", "ghat", "keep", "D_tilde", "Y_tilde"]
missing_required_cols = [c for c in required_cols if c not in dml_analysis_df.columns]
if missing_required_cols:
    raise KeyError(f"Step D4 is missing required columns: {missing_required_cols}")
if dml_analysis_df.empty:
    raise ValueError("Step D4 produced an empty keep==True analysis frame.")

dml_analysis_frame_summary_df = pd.DataFrame(
    [
        {
            "stepd_payload_key": dml_analysis_metadata["stepd_payload_key"],
            "stepd_payload_label": dml_analysis_metadata["stepd_payload_label"],
            "stepd_resolution": dml_analysis_metadata["stepd_resolution"],
            "restriction_name": dml_analysis_metadata["restriction_name"],
            "outcome_hours": dml_analysis_metadata["outcome_hours"],
            "trim_eps": dml_analysis_metadata["trim_eps"],
            "outcome_mode": dml_analysis_metadata["outcome_mode"],
            "n_rows_eligible": int(len(dml_analysis_df_full)),
            "n_rows_keep": int(len(dml_analysis_df)),
            "n_users_keep": int(dml_analysis_df["uid"].nunique()),
            "n_campaigns_keep": int(dml_analysis_df["campaign"].astype(str).nunique()),
            "d_mean_keep": float(pd.to_numeric(dml_analysis_df["D"], errors="coerce").mean()),
            "y_mean_keep": float(pd.to_numeric(dml_analysis_df["Y"], errors="coerce").mean()),
            "mhat_mean_keep": float(pd.to_numeric(dml_analysis_df["mhat"], errors="coerce").mean()),
            "ghat_mean_keep": float(pd.to_numeric(dml_analysis_df["ghat"], errors="coerce").mean()),
            "d_tilde_sq_mean_keep": float(np.mean(np.square(pd.to_numeric(dml_analysis_df["D_tilde"], errors="coerce")))),
            "added_feature_columns": int(len(added_feature_cols)),
        }
    ]
)

summary_dml_analysis_frame = {
    "title": "Standardized Step D analysis frame",
    **dml_analysis_metadata,
    "n_rows_eligible": int(len(dml_analysis_df_full)),
    "n_rows_keep": int(len(dml_analysis_df)),
    "n_users_keep": int(dml_analysis_df["uid"].nunique()),
    "n_campaigns_keep": int(dml_analysis_df["campaign"].astype(str).nunique()),
    "required_columns": required_cols,
    "added_feature_columns": added_feature_cols,
    "all_columns": list(dml_analysis_df.columns),
}

save_df(dml_analysis_frame_summary_df, Path(PAPER_OUTPUT_DIR) / "table_dml_analysis_frame_summary.csv")
save_json(dml_to_builtin(summary_dml_analysis_frame), Path(PAPER_OUTPUT_DIR) / "summary_dml_analysis_frame.json")

display(dml_analysis_frame_summary_df)
print("dml_analysis_df shape:", dml_analysis_df.shape)
print("Using Step D payload:", dml_analysis_metadata["stepd_payload_label"])
print("If Step D spec changes, rerun Step G2 and Step H2.")


## 7. RAMP Layer 2: Diagnostic Tests

This cell operationalizes the robustness section promised in the paper:

1. alternative outcome horizons (`1h`, `6h`, `24h`, `7d`)
2. overlap trimming sensitivity (`0.01`, `0.05`)
3. placebo outcome test
4. alternative nuisance learner baseline (`sgd_logit`)


In [ ]:
# ====== Step E: Robustness table + overlap figure ======
import matplotlib.pyplot as plt

robust_rows = []

for horizon_hours in [1.0, 6.0, 24.0, 24.0 * 7]:
    for trim_eps in [0.01, 0.05]:
        res = run_plr_dml(
            X=X_df,
            df_in=df_clean,
            outcome_hours=horizon_hours,
            trim_eps=trim_eps,
            learner="lightgbm",
            outcome_mode="forward",
            n_splits=5,
        )
        robust_rows.append({
            "spec": "main",
            "learner": "lightgbm",
            "outcome_mode": "forward",
            "horizon_hours": horizon_hours,
            "trim_eps": trim_eps,
            "theta_hat": res["theta_hat"],
            "se_cluster": res["se_cluster"],
            "ci_lo": res["ci_lo"],
            "ci_hi": res["ci_hi"],
            "keep_rate": res["keep_rate"],
            "eligible_rows": res["eligible_rows"],
            "eligible_rate": res["eligible_rate"],
            "y_mean": res["y_mean"],
            "d_mean": res["d_mean"],
            "propensity_mean": res["propensity_mean"],
            "propensity_min": res["propensity_min"],
            "propensity_max": res["propensity_max"],
            "propensity_p01": res["propensity_p01"],
            "propensity_p99": res["propensity_p99"],
            "propensity_auc": res["propensity_auc"],
            "propensity_logloss": res["propensity_logloss"],
            "outcome_auc": res["outcome_auc"],
            "outcome_logloss": res["outcome_logloss"],
        })

placebo_res = run_plr_dml(
    X=X_df,
    df_in=df_clean,
    outcome_hours=24.0,
    trim_eps=0.01,
    learner="lightgbm",
    outcome_mode="placebo",
    n_splits=5,
)
robust_rows.append({
    "spec": "placebo",
    "learner": "lightgbm",
    "outcome_mode": "placebo",
    "horizon_hours": 24.0,
    "trim_eps": 0.01,
    "theta_hat": placebo_res["theta_hat"],
    "se_cluster": placebo_res["se_cluster"],
    "ci_lo": placebo_res["ci_lo"],
    "ci_hi": placebo_res["ci_hi"],
    "keep_rate": placebo_res["keep_rate"],
    "eligible_rows": placebo_res["eligible_rows"],
    "eligible_rate": placebo_res["eligible_rate"],
    "y_mean": placebo_res["y_mean"],
    "d_mean": placebo_res["d_mean"],
    "propensity_mean": placebo_res["propensity_mean"],
    "propensity_min": placebo_res["propensity_min"],
    "propensity_max": placebo_res["propensity_max"],
    "propensity_p01": placebo_res["propensity_p01"],
    "propensity_p99": placebo_res["propensity_p99"],
    "propensity_auc": placebo_res["propensity_auc"],
    "propensity_logloss": placebo_res["propensity_logloss"],
    "outcome_auc": placebo_res["outcome_auc"],
    "outcome_logloss": placebo_res["outcome_logloss"],
})

sgd_res = run_plr_dml(
    X=X_df,
    df_in=df_clean,
    outcome_hours=24.0,
    trim_eps=0.01,
    learner="sgd_logit",
    outcome_mode="forward",
    n_splits=5,
)
robust_rows.append({
    "spec": "alt_learner",
    "learner": "sgd_logit",
    "outcome_mode": "forward",
    "horizon_hours": 24.0,
    "trim_eps": 0.01,
    "theta_hat": sgd_res["theta_hat"],
    "se_cluster": sgd_res["se_cluster"],
    "ci_lo": sgd_res["ci_lo"],
    "ci_hi": sgd_res["ci_hi"],
    "keep_rate": sgd_res["keep_rate"],
    "eligible_rows": sgd_res["eligible_rows"],
    "eligible_rate": sgd_res["eligible_rate"],
    "y_mean": sgd_res["y_mean"],
    "d_mean": sgd_res["d_mean"],
    "propensity_mean": sgd_res["propensity_mean"],
    "propensity_min": sgd_res["propensity_min"],
    "propensity_max": sgd_res["propensity_max"],
    "propensity_p01": sgd_res["propensity_p01"],
    "propensity_p99": sgd_res["propensity_p99"],
    "propensity_auc": sgd_res["propensity_auc"],
    "propensity_logloss": sgd_res["propensity_logloss"],
    "outcome_auc": sgd_res["outcome_auc"],
    "outcome_logloss": sgd_res["outcome_logloss"],
})

robustness_table = pd.DataFrame(robust_rows).sort_values(["spec", "horizon_hours", "trim_eps"]).reset_index(drop=True)
save_df(robustness_table, ROBUSTNESS_TABLE_PATH)
display(robustness_table)

main_horizon_slice = robustness_table[
    (robustness_table["spec"] == "main")
    & np.isclose(robustness_table["trim_eps"], 0.01)
].sort_values("horizon_hours").reset_index(drop=True)
analysis_validation_issues = []
if main_horizon_slice.empty:
    analysis_validation_issues.append("No main horizon robustness rows were produced.")
else:
    y_vals = main_horizon_slice["y_mean"].to_numpy(dtype=float)
    theta_vals = main_horizon_slice["theta_hat"].to_numpy(dtype=float)
    if np.all(~np.isfinite(y_vals)):
        analysis_validation_issues.append("All horizon outcome means are missing.")
    elif np.nanmax(np.abs(y_vals)) == 0.0:
        analysis_validation_issues.append("All forward outcome means are exactly zero across horizons.")
    if np.unique(np.round(y_vals[np.isfinite(y_vals)], 12)).size <= 1:
        analysis_validation_issues.append("Forward outcome means do not vary across horizons.")
    if np.unique(np.round(theta_vals[np.isfinite(theta_vals)], 12)).size <= 1:
        analysis_validation_issues.append("ATE estimates do not vary across horizons.")
if float(step_ab_meta.get("forward_event_after_rate", np.nan)) == 0.0:
    analysis_validation_issues.append("No forward event_time values occur after impression timestamps in Step A/B diagnostics.")
if float(step_ab_meta.get("forward_24h_mean", np.nan)) == 0.0:
    analysis_validation_issues.append("Step A/B forward 24h outcome mean is zero.")

analysis_validation = {
    "ready_for_writing": len(analysis_validation_issues) == 0,
    "issues": analysis_validation_issues,
}
print("[Validation] ready_for_writing:", analysis_validation["ready_for_writing"])
for issue in analysis_validation_issues:
    print("[Validation]", issue)

keep = main_result_24h["keep"]
mhat = main_result_24h["mhat"]
D = main_result_24h["D"]

plt.figure(figsize=(10, 6))
plt.hist(mhat[keep & (D == 1)], bins=60, alpha=0.5, label="Clicked impression", density=True)
plt.hist(mhat[keep & (D == 0)], bins=60, alpha=0.5, label="Unclicked impression", density=True)
plt.title("Propensity overlap after cross-fitting")
plt.xlabel("Estimated click propensity")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(OVERLAP_FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()

print("Saved robustness table to:", ROBUSTNESS_TABLE_PATH)
print("Saved overlap figure to:", OVERLAP_FIG_PATH)


## 12. Reproduce Paper Tables


In [ ]:
# ====== Step E.1: Paper-ready interpretation of Step D/E results ======
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def pick_row(frame, spec, horizon_hours, trim_eps):
    mask = (
        (frame["spec"] == spec)
        & np.isclose(frame["horizon_hours"], horizon_hours)
        & np.isclose(frame["trim_eps"], trim_eps)
    )
    rows = frame.loc[mask]
    if rows.empty:
        raise KeyError(f"Missing robustness row for spec={spec}, horizon={horizon_hours}, trim={trim_eps}")
    return rows.iloc[0]


main_h1 = pick_row(robustness_table, "main", 1.0, 0.01)
main_h6 = pick_row(robustness_table, "main", 6.0, 0.01)
main_h24 = pick_row(robustness_table, "main", 24.0, 0.01)
main_h7d = pick_row(robustness_table, "main", 24.0 * 7, 0.01)
main_h24_trim05 = pick_row(robustness_table, "main", 24.0, 0.05)

trim_change = abs(main_h24["theta_hat"] - main_h24_trim05["theta_hat"])
trim_language = "nearly unchanged" if trim_change <= 0.002 else f"changed by {trim_change:.5f}"

if analysis_validation.get("ready_for_writing", False):
    paper_ready_writeup = f"""## Paper-ready interpretation of Step D/E results

The cleaned estimation sample contains {step_ab_meta['cleaned_rows']:,} impression-level rows and {step_ab_meta['cleaned_cols']:,} columns, with click mean {step_ab_meta['cleaned_click_mean']:.4f} and raw conversion mean {step_ab_meta['raw_conversion_mean']:.4f}. Timestamps were normalized using `{step_ab_meta['time_unit']}`, and the 24-hour forward outcome was built from `{step_ab_meta['used_conversion_timestamp']}`.

The main 24-hour DML estimate is theta_hat = {main_result_24h['theta_hat']:.5f}, SE = {main_result_24h['se_cluster']:.6f}, 95% CI = [{main_result_24h['ci_lo']:.5f}, {main_result_24h['ci_hi']:.5f}], with keep rate {main_result_24h['keep_rate']:.6f}. Horizon robustness at 1h, 6h, 24h, and 7d shows estimates of {main_h1['theta_hat']:.5f}, {main_h6['theta_hat']:.5f}, {main_h24['theta_hat']:.5f}, and {main_h7d['theta_hat']:.5f}, and the 24-hour estimate is {trim_language} under trim 0.05.

These results support the sample description, main ATE table, overlap diagnostics, nuisance diagnostics, and horizon/trim robustness tables. Segment-level heterogeneity and campaign-level reallocations still belong in later sections rather than this summary.
"""
else:
    issue_lines = "\n".join([f"- {x}" for x in analysis_validation.get("issues", [])])
    paper_ready_writeup = f"""## Step D/E Diagnostic Warning

The current Step D/E outputs are **not ready for paper writing**. The notebook detected one or more internal consistency problems in the outcome or horizon construction:

{issue_lines}

Treat the current 24-hour ATE table, horizon robustness table, and any narrative derived from them as draft diagnostics only. Re-run the patched notebook and inspect Step A/B and Step E validation output before using these numbers in the paper.
"""

sample_outcome_table = pd.DataFrame([
    {
        "cleaned_rows": int(step_ab_meta["cleaned_rows"]),
        "cleaned_cols": int(step_ab_meta["cleaned_cols"]),
        "click_mean": float(step_ab_meta["cleaned_click_mean"]),
        "raw_conversion_mean": float(step_ab_meta["raw_conversion_mean"]),
        "forward_24h_mean": float(step_ab_meta["forward_24h_mean"]),
        "placebo_24h_mean": float(step_ab_meta["placebo_24h_mean"]),
        "time_unit": step_ab_meta["time_unit"],
        "outcome_source": step_ab_meta["used_conversion_timestamp"],
        "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
    }
])

main_ate_table = pd.DataFrame([
    {
        "theta_hat": float(main_result_24h["theta_hat"]),
        "se_cluster": float(main_result_24h["se_cluster"]),
        "ci_lo": float(main_result_24h["ci_lo"]),
        "ci_hi": float(main_result_24h["ci_hi"]),
        "t_stat": float(main_result_24h["t_stat"]),
        "n_eff": int(main_result_24h["n_eff"]),
        "clusters": int(main_result_24h["clusters"]),
        "keep_rate": float(main_result_24h["keep_rate"]),
        "eligible_rows": int(main_result_24h.get("eligible_rows", np.nan)),
        "y_mean": float(main_result_24h.get("y_mean", np.nan)),
        "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
    }
])

overlap_table = pd.DataFrame([
    {
        "propensity_mean": float(main_result_24h["propensity_mean"]),
        "propensity_min": float(main_result_24h["propensity_min"]),
        "propensity_max": float(main_result_24h["propensity_max"]),
        "propensity_p01": float(main_result_24h["propensity_p01"]),
        "propensity_p99": float(main_result_24h["propensity_p99"]),
        "keep_rate": float(main_result_24h["keep_rate"]),
        "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
    }
])

nuisance_table = pd.DataFrame([
    {
        "propensity_auc": float(main_result_24h["propensity_auc"]),
        "propensity_logloss": float(main_result_24h["propensity_logloss"]),
        "outcome_auc": float(main_result_24h["outcome_auc"]),
        "outcome_logloss": float(main_result_24h["outcome_logloss"]),
        "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
    }
])

horizon_table = (
    robustness_table[
        (robustness_table["spec"] == "main")
        & np.isclose(robustness_table["trim_eps"], 0.01)
    ][["horizon_hours", "theta_hat", "se_cluster", "ci_lo", "ci_hi", "keep_rate", "eligible_rows", "y_mean"]]
    .sort_values("horizon_hours")
    .reset_index(drop=True)
)
horizon_table["result_status"] = "valid" if analysis_validation.get("ready_for_writing", False) else "suspect"

trim_table = (
    robustness_table[
        (robustness_table["spec"] == "main")
        & np.isclose(robustness_table["horizon_hours"], 24.0)
    ][["trim_eps", "theta_hat", "se_cluster", "ci_lo", "ci_hi", "keep_rate", "eligible_rows", "y_mean"]]
    .sort_values("trim_eps")
    .reset_index(drop=True)
)
trim_table["result_status"] = "valid" if analysis_validation.get("ready_for_writing", False) else "suspect"

save_df(sample_outcome_table, PAPER_OUTPUT_DIR / "table_sample_outcome_summary.csv")
save_df(main_ate_table, PAPER_OUTPUT_DIR / "table_main_ate_24h.csv")
save_df(overlap_table, PAPER_OUTPUT_DIR / "table_overlap_diagnostics_24h.csv")
save_df(nuisance_table, PAPER_OUTPUT_DIR / "table_nuisance_diagnostics_24h.csv")
save_df(horizon_table, PAPER_OUTPUT_DIR / "table_horizon_robustness.csv")
save_df(trim_table, PAPER_OUTPUT_DIR / "table_trim_robustness_24h.csv")
save_json(analysis_validation, PAPER_OUTPUT_DIR / "summary_step_de_validation.json")

display(Markdown(paper_ready_writeup))
write_text(paper_ready_writeup, PAPER_WRITEUP_PATH)
print("Saved paper-ready interpretation to:", PAPER_WRITEUP_PATH)
print("Saved sample summary table to:", PAPER_OUTPUT_DIR / "table_sample_outcome_summary.csv")
print("Saved main ATE table to:", PAPER_OUTPUT_DIR / "table_main_ate_24h.csv")
print("Saved overlap diagnostics table to:", PAPER_OUTPUT_DIR / "table_overlap_diagnostics_24h.csv")
print("Saved nuisance diagnostics table to:", PAPER_OUTPUT_DIR / "table_nuisance_diagnostics_24h.csv")
print("Saved horizon robustness table to:", PAPER_OUTPUT_DIR / "table_horizon_robustness.csv")
print("Saved trim robustness table to:", PAPER_OUTPUT_DIR / "table_trim_robustness_24h.csv")
print("Saved Step D/E validation summary to:", PAPER_OUTPUT_DIR / "summary_step_de_validation.json")


## Step F: Paper-ready summary tables and exports


In [ ]:
# ====== Step F0: Paper output setup ======
from pathlib import Path
import json
import numpy as np
import pandas as pd

PAPER_OUT = PAPER_TABLE_DIR
PAPER_OUT.mkdir(parents=True, exist_ok=True)


def save_table(df, name):
    path = PAPER_OUT / f"{name}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


def save_named_json(obj, name):
    path = PAPER_OUT / f"{name}.json"
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print("Saved:", path)
    return path


def pct(x):
    return float(x) * 100.0


print("Paper output dir:", PAPER_OUT.resolve())


### Step F1: Sample construction and outcome summary


In [ ]:
# ====== Step F1: Sample construction + outcome summary ======
assert "df_clean" in globals(), "df_clean not found. Run Step A/B first."
assert "df_main_24h" in globals(), "df_main_24h not found. Run Step D first."

D_main = np.asarray(D).astype(int) if "D" in globals() else np.asarray(main_result_24h.get("D"))
Y_main = np.asarray(Y).astype(int) if "Y" in globals() else np.asarray(main_result_24h.get("Y"))
assert Y_main is not None, "Y not found. Run Step D first."

if "Y_placebo_24h" in globals():
    Y_placebo_main = np.asarray(Y_placebo_24h).astype(int)
elif "build_placebo_outcome" in globals():
    Y_placebo_main = build_placebo_outcome(df_main_24h, horizon_hours=24.0).astype(int)
else:
    tmp = df_main_24h[["uid", "ts_sec", "conversion"]].copy().sort_values(["uid", "ts_sec"]).reset_index(drop=True)

    def prior_conv_24h(g):
        t = g["ts_sec"].to_numpy()
        y = g["conversion"].astype(int).to_numpy()
        out = np.zeros(len(g), dtype=int)
        left = 0
        running = 0
        for i in range(len(g)):
            while left < i and t[left] < t[i] - 24 * 3600:
                running -= y[left]
                left += 1
            out[i] = 1 if running > 0 else 0
            running += y[i]
        return pd.Series(out, index=g.index)

    Y_placebo_main = (
        tmp.groupby("uid", group_keys=False)
           .apply(prior_conv_24h)
           .astype(int)
           .to_numpy()
    )

Y_placebo_24h = Y_placebo_main

sample_summary = {
    "n_rows_clean": int(step_ab_meta.get("cleaned_rows", df_clean.shape[0])) if "step_ab_meta" in globals() else int(df_clean.shape[0]),
    "n_rows_analysis_24h": int(df_main_24h.shape[0]),
    "n_cols_clean": int(step_ab_meta.get("cleaned_cols", df_clean.shape[1])) if "step_ab_meta" in globals() else int(df_clean.shape[1]),
    "n_users": int(df_clean["uid"].nunique()),
    "n_users_analysis_24h": int(df_main_24h["uid"].nunique()),
    "n_campaigns": int(df_clean["campaign"].astype(str).nunique()),
    "click_mean": float(np.mean(D_main)),
    "raw_conversion_mean": float(step_ab_meta.get("raw_conversion_mean", df_clean["conversion"].mean())) if "step_ab_meta" in globals() else float(df_clean["conversion"].mean()),
    "y_24h_mean": float(np.mean(Y_main)),
    "y_placebo_24h_mean": float(np.mean(Y_placebo_main)),
    "time_unit": step_ab_meta.get("time_unit", np.nan) if "step_ab_meta" in globals() else np.nan,
    "outcome_source": step_ab_meta.get("used_conversion_timestamp", np.nan) if "step_ab_meta" in globals() else np.nan,
    "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
}

sample_summary_df = pd.DataFrame([
    {"metric": "Cleaned estimation rows", "value": sample_summary["n_rows_clean"]},
    {"metric": "24h analysis rows", "value": sample_summary["n_rows_analysis_24h"]},
    {"metric": "Number of columns", "value": sample_summary["n_cols_clean"]},
    {"metric": "Unique users", "value": sample_summary["n_users"]},
    {"metric": "24h analysis users", "value": sample_summary["n_users_analysis_24h"]},
    {"metric": "Unique campaigns", "value": sample_summary["n_campaigns"]},
    {"metric": "Click mean", "value": sample_summary["click_mean"]},
    {"metric": "Raw conversion mean", "value": sample_summary["raw_conversion_mean"]},
    {"metric": "Forward 24h outcome mean", "value": sample_summary["y_24h_mean"]},
    {"metric": "Placebo 24h outcome mean", "value": sample_summary["y_placebo_24h_mean"]},
    {"metric": "Timestamp unit", "value": sample_summary["time_unit"]},
    {"metric": "Outcome timestamp source", "value": sample_summary["outcome_source"]},
    {"metric": "Result status", "value": sample_summary["result_status"]},
])

display(sample_summary_df)
save_table(sample_summary_df, "table_sample_construction")
save_named_json(sample_summary, "summary_sample_construction")


### Step F2: Main 24h ATE table


In [ ]:
# ====== Step F2: Main 24h ATE result ======
required_vars = ["main_result_24h", "df_main_24h"]
for v in required_vars:
    assert v in globals(), f"{v} not found. Run Step D first."

theta_hat = float(main_result_24h["theta_hat"])
se_cluster = float(main_result_24h["se_cluster"])
ci_low_24h = float(main_result_24h["ci_lo"])
ci_high_24h = float(main_result_24h["ci_hi"])
t_stat_24h = float(main_result_24h["t_stat"])
keep = main_result_24h["keep"]
mhat = main_result_24h["mhat"]
ghat = main_result_24h["ghat"]
Y = main_result_24h["Y"]
D = main_result_24h["D"]

n_eff_24h = int(np.sum(keep))
n_clusters_24h = int(df_main_24h.loc[keep, "uid"].nunique())
keep_rate_24h = float(np.mean(keep))

main_ate_df = pd.DataFrame([{
    "horizon_hours": 24,
    "ate": theta_hat,
    "se_cluster": se_cluster,
    "ci_low": ci_low_24h,
    "ci_high": ci_high_24h,
    "t_stat": t_stat_24h,
    "n_eff": n_eff_24h,
    "n_clusters": n_clusters_24h,
    "keep_rate": keep_rate_24h,
    "eligible_rows": int(main_result_24h.get("eligible_rows", len(df_main_24h))),
    "y_mean": float(np.mean(Y)),
    "d_mean": float(np.mean(D)),
    "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
}])

display(main_ate_df)
save_table(main_ate_df, "table_main_ate_24h")


### Step F3: Overlap and nuisance diagnostics


In [ ]:
# ====== Step F3: Overlap + nuisance diagnostics ======
assert "mhat" in globals(), "mhat not found. Run Step D first."
assert "keep" in globals(), "keep not found. Run Step D first."

overlap_stats = pd.Series(mhat).describe(percentiles=[0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99])
overlap_df = pd.DataFrame([
    {"metric": "Propensity mean", "value": float(np.mean(mhat))},
    {"metric": "Propensity min", "value": float(np.min(mhat))},
    {"metric": "Propensity max", "value": float(np.max(mhat))},
    {"metric": "Propensity p01", "value": float(overlap_stats["1%"])},
    {"metric": "Propensity p05", "value": float(overlap_stats["5%"])},
    {"metric": "Propensity p10", "value": float(overlap_stats["10%"])},
    {"metric": "Propensity median", "value": float(overlap_stats["50%"])},
    {"metric": "Propensity p90", "value": float(overlap_stats["90%"])},
    {"metric": "Propensity p95", "value": float(overlap_stats["95%"])},
    {"metric": "Propensity p99", "value": float(overlap_stats["99%"])},
    {"metric": "Trim keep rate", "value": float(np.mean(keep))},
])

prop_auc = float(np.mean(auc_m_list)) if "auc_m_list" in globals() and len(auc_m_list) > 0 else float(main_result_24h.get("propensity_auc", np.nan))
prop_ll = float(np.mean(ll_m_list)) if "ll_m_list" in globals() and len(ll_m_list) > 0 else float(main_result_24h.get("propensity_logloss", np.nan))
out_auc = float(np.mean(auc_g_list)) if "auc_g_list" in globals() and len(auc_g_list) > 0 else float(main_result_24h.get("outcome_auc", np.nan))
out_ll = float(np.mean(ll_g_list)) if "ll_g_list" in globals() and len(ll_g_list) > 0 else float(main_result_24h.get("outcome_logloss", np.nan))

nuisance_df = pd.DataFrame([
    {"model": "Propensity model", "metric": "AUC", "value": prop_auc},
    {"model": "Propensity model", "metric": "LogLoss", "value": prop_ll},
    {"model": "Outcome model", "metric": "AUC", "value": out_auc},
    {"model": "Outcome model", "metric": "LogLoss", "value": out_ll},
])

display(overlap_df)
display(nuisance_df)
save_table(overlap_df, "table_overlap_diagnostics_24h")
save_table(nuisance_df, "table_nuisance_diagnostics_24h")


### Step F4: Horizon robustness


In [ ]:
# ====== Step F4: Horizon robustness ======
assert "run_plr_dml" in globals(), "run_plr_dml not found. Make sure Step D is defined."

horizon_list = [1.0, 6.0, 24.0]
horizon_rows = []
existing = robustness_table if "robustness_table" in globals() else pd.DataFrame()

for h in horizon_list:
    row = existing[
        (existing.get("spec") == "main")
        & np.isclose(existing.get("horizon_hours"), h)
        & np.isclose(existing.get("trim_eps"), 0.01)
    ] if not existing.empty else pd.DataFrame()

    if not row.empty:
        res = row.iloc[0].to_dict()
    else:
        print(f"Running horizon robustness fallback for {h}h ...")
        res = run_plr_dml(outcome_hours=h, trim_eps=0.01, learner="lightgbm", outcome_mode="forward", n_splits=5)

    theta = float(res.get("theta_hat", np.nan))
    se = float(res.get("se_cluster", np.nan))
    ci_low = float(res.get("ci_lo", res.get("ci_low", theta - 1.96 * se if np.isfinite(theta) and np.isfinite(se) else np.nan)))
    ci_high = float(res.get("ci_hi", res.get("ci_high", theta + 1.96 * se if np.isfinite(theta) and np.isfinite(se) else np.nan)))

    horizon_rows.append({
        "horizon_hours": h,
        "ate": theta,
        "se_cluster": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "keep_rate": float(res.get("keep_rate", np.nan)),
        "n_eff": int(res.get("n_eff", np.nan)) if pd.notna(res.get("n_eff", np.nan)) else np.nan,
        "eligible_rows": int(res.get("eligible_rows", np.nan)) if pd.notna(res.get("eligible_rows", np.nan)) else np.nan,
        "y_mean": float(res.get("y_mean", np.nan)),
        "propensity_auc": float(res.get("propensity_auc", np.nan)),
        "outcome_auc": float(res.get("outcome_auc", np.nan)),
        "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
    })

horizon_df = pd.DataFrame(horizon_rows)
display(horizon_df)
save_table(horizon_df, "table_horizon_robustness")


### Step F5: Trim robustness


In [ ]:
# ====== Step F5: Trim robustness ======
assert "run_plr_dml" in globals(), "run_plr_dml not found. Make sure Step D is defined."

trim_list = [0.00, 0.01, 0.05]
trim_rows = []
existing = robustness_table if "robustness_table" in globals() else pd.DataFrame()

for eps in trim_list:
    row = existing[
        (existing.get("spec") == "main")
        & np.isclose(existing.get("horizon_hours"), 24.0)
        & np.isclose(existing.get("trim_eps"), eps)
    ] if not existing.empty else pd.DataFrame()

    if not row.empty:
        res = row.iloc[0].to_dict()
    else:
        print(f"Running trim robustness fallback for eps={eps} ...")
        res = run_plr_dml(outcome_hours=24.0, trim_eps=eps, learner="lightgbm", outcome_mode="forward", n_splits=5)

    theta = float(res.get("theta_hat", np.nan))
    se = float(res.get("se_cluster", np.nan))
    ci_low = float(res.get("ci_lo", res.get("ci_low", theta - 1.96 * se if np.isfinite(theta) and np.isfinite(se) else np.nan)))
    ci_high = float(res.get("ci_hi", res.get("ci_high", theta + 1.96 * se if np.isfinite(theta) and np.isfinite(se) else np.nan)))

    trim_rows.append({
        "trim_eps": eps,
        "ate": theta,
        "se_cluster": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "keep_rate": float(res.get("keep_rate", np.nan)),
        "n_eff": int(res.get("n_eff", np.nan)) if pd.notna(res.get("n_eff", np.nan)) else np.nan,
        "eligible_rows": int(res.get("eligible_rows", np.nan)) if pd.notna(res.get("eligible_rows", np.nan)) else np.nan,
        "y_mean": float(res.get("y_mean", np.nan)),
        "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
    })

trim_df = pd.DataFrame(trim_rows)
display(trim_df)
save_table(trim_df, "table_trim_robustness_24h")


### Step F6: Summary numbers for writing


In [ ]:
# ====== Step F6: Summary numbers for writing ======
summary_numbers = {
    "n_rows_clean": int(df_clean.shape[0]),
    "n_rows_analysis_24h": int(df_main_24h.shape[0]) if "df_main_24h" in globals() else np.nan,
    "n_users": int(df_clean["uid"].nunique()),
    "n_users_analysis_24h": int(df_main_24h["uid"].nunique()) if "df_main_24h" in globals() else np.nan,
    "n_campaigns": int(df_clean["campaign"].astype(str).nunique()),
    "click_mean": float(np.mean(D)),
    "raw_conversion_mean": float(step_ab_meta.get("raw_conversion_mean", df_clean["conversion"].mean())) if "step_ab_meta" in globals() else float(df_clean["conversion"].mean()),
    "y_24h_mean": float(np.mean(Y)),
    "y_placebo_24h_mean": float(np.mean(Y_placebo_24h)) if "Y_placebo_24h" in globals() else np.nan,
    "theta_hat_24h": float(theta_hat),
    "se_cluster_24h": float(se_cluster),
    "ci_low_24h": float(ci_low_24h),
    "ci_high_24h": float(ci_high_24h),
    "t_stat_24h": float(t_stat_24h),
    "n_eff_24h": int(n_eff_24h),
    "n_clusters_24h": int(n_clusters_24h),
    "keep_rate_24h": float(keep_rate_24h),
    "propensity_mean": float(np.mean(mhat)),
    "propensity_min": float(np.min(mhat)),
    "propensity_max": float(np.max(mhat)),
    "propensity_p01": float(overlap_stats["1%"]),
    "propensity_p99": float(overlap_stats["99%"]),
    "propensity_auc": float(prop_auc),
    "propensity_logloss": float(prop_ll),
    "outcome_auc": float(out_auc),
    "outcome_logloss": float(out_ll),
    "result_status": "valid" if analysis_validation.get("ready_for_writing", False) else "suspect",
}

display(pd.DataFrame(list(summary_numbers.items()), columns=["metric", "value"]))
save_named_json(summary_numbers, "summary_numbers_main")


## 8. RAMP Layer 3: Campaign-Level Aggregation

This section uses the cross-fitted nuisance predictions from Step D to estimate segment-level PLR effects without retraining the full DML benchmark inside each subgroup. The goal is to answer a first-pass "who is persuadable?" question across observable user states.


In [ ]:
# ====== Step G.1: Build heterogeneity analysis dataframe + helpers ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "get_stepd_payload" in globals(), "get_stepd_payload not found. Run Step D first."
assert "save_table" in globals(), "save_table not found. Run Step F0 first."

STEPD_DOWNSTREAM_SOURCE = globals().get(
    "STEPD_DOWNSTREAM_SOURCE",
    "final" if "final" in globals().get("STEPD_PAYLOADS", {}) else "baseline",
)
stepd_payload = get_stepd_payload(STEPD_DOWNSTREAM_SOURCE)
stepd_payload_summary = dict(stepd_payload["summary"])

selected_df = stepd_payload["df"].copy().reset_index(drop=True)
selected_X = stepd_payload["X"].copy().reset_index(drop=True)
selected_D = np.asarray(stepd_payload["D"]).astype(int)
selected_Y = np.asarray(stepd_payload["Y"]).astype(int)
selected_keep = np.asarray(stepd_payload["keep"]).astype(bool)
selected_mhat = np.asarray(stepd_payload["mhat"], dtype=float)
selected_ghat = np.asarray(stepd_payload["ghat"], dtype=float)

if len(selected_df) != len(selected_X):
    raise ValueError("Selected Step D df and X frames are not aligned.")
if len(selected_df) != len(selected_D) or len(selected_df) != len(selected_Y):
    raise ValueError("Selected Step D local sample is not aligned with D/Y vectors.")

trim_eps_selected = float(stepd_payload_summary.get("trim_eps", 0.01))
mhat_clip = np.clip(selected_mhat, max(trim_eps_selected, 1e-6), 1 - max(trim_eps_selected, 1e-6))

hetero_df = selected_df.copy().reset_index(drop=True)
hetero_df["D"] = selected_D
hetero_df["Y"] = selected_Y
hetero_df["mhat"] = mhat_clip
hetero_df["ghat"] = selected_ghat
hetero_df["keep"] = selected_keep

for c in selected_X.columns:
    if c not in hetero_df.columns:
        hetero_df[c] = selected_X[c].to_numpy()

hetero_df = hetero_df.loc[hetero_df["keep"]].copy().reset_index(drop=True)
hetero_df["D_tilde"] = hetero_df["D"] - hetero_df["mhat"]
hetero_df["Y_tilde"] = hetero_df["Y"] - hetero_df["ghat"]

HETERO_FIG_DIR = PAPER_OUT if "PAPER_OUT" in globals() else PAPER_OUTPUT_DIR
HETERO_FIG_DIR.mkdir(parents=True, exist_ok=True)

SEGMENT_COLS = [
    "segment_name", "segment_value", "n_eff", "n_users",
    "d_mean", "y_mean", "ate", "se_cluster", "ci_low", "ci_high", "t_stat", "status"
]

def _safe_rank_qcut(series, q, labels):
    series = pd.Series(series)
    fill_value = float(series.dropna().median()) if series.notna().any() else 0.0
    ranked = series.fillna(fill_value).rank(method="first")
    return pd.qcut(ranked, q=q, labels=labels)

def segment_plr(df_seg, segment_name, segment_value, min_obs=200, min_users=50):
    df_seg = df_seg.copy()
    n_eff = int(df_seg.shape[0])
    n_users = int(df_seg["uid"].nunique()) if n_eff else 0

    result = {
        "segment_name": segment_name,
        "segment_value": str(segment_value),
        "n_eff": n_eff,
        "n_users": n_users,
        "d_mean": float(df_seg["D"].mean()) if n_eff else np.nan,
        "y_mean": float(df_seg["Y"].mean()) if n_eff else np.nan,
        "ate": np.nan,
        "se_cluster": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "t_stat": np.nan,
        "status": "too_small",
    }

    if n_eff < min_obs or n_users < min_users:
        return result

    D_tilde = df_seg["D_tilde"].to_numpy(dtype=float)
    Y_tilde = df_seg["Y_tilde"].to_numpy(dtype=float)
    den = float(np.mean(D_tilde ** 2))
    if den <= 1e-12:
        result["status"] = "weak_overlap"
        return result

    theta = float(np.mean(D_tilde * Y_tilde) / den)
    resid = Y_tilde - theta * D_tilde
    influence = (D_tilde * resid) / den

    if "cluster_se_from_if" in globals():
        se, _ = cluster_se_from_if(influence, df_seg["uid"].to_numpy())
        se = float(se)
    else:
        tmp_if = pd.DataFrame({"uid": df_seg["uid"].to_numpy(), "if_value": influence})
        scores = tmp_if.groupby("uid")["if_value"].sum().to_numpy(dtype=float)
        se = float(np.sqrt(np.dot(scores, scores) / (n_eff ** 2))) if n_eff > 0 else np.nan

    result.update({
        "ate": theta,
        "se_cluster": se,
        "ci_low": float(theta - 1.96 * se) if np.isfinite(se) else np.nan,
        "ci_high": float(theta + 1.96 * se) if np.isfinite(se) else np.nan,
        "t_stat": float(theta / se) if np.isfinite(se) and se > 0 else np.nan,
        "status": "ok",
    })
    return result

def plot_segment_effects(df_plot, title, filename):
    if df_plot.empty:
        return None
    df_plot = df_plot.loc[df_plot["status"] == "ok"].copy().reset_index(drop=True)
    if df_plot.empty:
        return None

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.errorbar(
        x=df_plot["ate"],
        y=np.arange(len(df_plot)),
        xerr=[df_plot["ate"] - df_plot["ci_low"], df_plot["ci_high"] - df_plot["ate"]],
        fmt="o",
    )
    ax.set_yticks(np.arange(len(df_plot)))
    ax.set_yticklabels(df_plot["segment_value"].astype(str))
    ax.axvline(0.0, linestyle="--", color="black", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Estimated segment-level effect")
    ax.set_ylabel("")
    fig.tight_layout()
    out_path = HETERO_FIG_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)
    return out_path

print("Step G using Step D payload:", stepd_payload.get("label", STEPD_DOWNSTREAM_SOURCE))
print("hetero_df shape:", hetero_df.shape)
display(hetero_df.head())

In [ ]:
# ====== Step G.2: Estimate segment-level heterogeneity tables ======
prop_labels = ["Q1_low", "Q2", "Q3", "Q4", "Q5_high"]
hetero_df["propensity_quintile"] = _safe_rank_qcut(hetero_df["mhat"], 5, prop_labels)

prop_rows = []
for grp in prop_labels:
    sub = hetero_df.loc[hetero_df["propensity_quintile"].astype(str) == grp]
    if sub.empty:
        continue
    prop_rows.append(segment_plr(sub, "propensity_quintile", grp))
hetero_propensity_df = pd.DataFrame(prop_rows, columns=SEGMENT_COLS)
display(hetero_propensity_df)
save_table(hetero_propensity_df, "table_heterogeneity_propensity_quintile")

recency_candidates = [
    "days_since_click",
    "time_since_last_click_sec",
    "log_days_since_click",
]
recency_col = next((c for c in recency_candidates if c in hetero_df.columns), None)

if recency_col is not None or "no_prior_click" in hetero_df.columns:
    if recency_col == "time_since_last_click_sec":
        recency_days = pd.to_numeric(hetero_df[recency_col], errors="coerce") / 86400.0
    elif recency_col == "log_days_since_click":
        recency_days = np.expm1(pd.to_numeric(hetero_df[recency_col], errors="coerce"))
    elif recency_col is not None:
        recency_days = pd.to_numeric(hetero_df[recency_col], errors="coerce")
    else:
        recency_days = pd.Series(np.inf, index=hetero_df.index, dtype=float)

    recency_bucket = pd.cut(
        recency_days,
        bins=[-np.inf, 1.0, 7.0, 30.0, np.inf],
        labels=["<=1d", "1-7d", "7-30d", ">30d"],
    ).astype(object)

    if "no_prior_click" in hetero_df.columns:
        no_prior = hetero_df["no_prior_click"].fillna(0).astype(int) == 1
        recency_bucket.loc[no_prior] = "no_prior_click"

    recency_order = ["no_prior_click", "<=1d", "1-7d", "7-30d", ">30d"]
    hetero_df["recency_bucket"] = pd.Categorical(recency_bucket, categories=recency_order, ordered=True)

    recency_rows = []
    for grp in recency_order:
        sub = hetero_df.loc[hetero_df["recency_bucket"].astype(str) == grp]
        if sub.empty:
            continue
        recency_rows.append(segment_plr(sub, "recency_bucket", grp))
    hetero_recency_df = pd.DataFrame(recency_rows, columns=SEGMENT_COLS)
else:
    hetero_recency_df = pd.DataFrame(columns=SEGMENT_COLS)
    print("Skipping recency heterogeneity: no recency column found.")

display(hetero_recency_df)
save_table(hetero_recency_df, "table_heterogeneity_recency")

intensity_candidates = [
    "imp_cnt_24h_pre",
    "imp_hist_7",
    "imp_hist_3",
    "n_impressions",
]
intensity_col = next((c for c in intensity_candidates if c in hetero_df.columns), None)

if intensity_col is not None:
    hetero_df["impression_intensity_tercile"] = _safe_rank_qcut(
        pd.to_numeric(hetero_df[intensity_col], errors="coerce").fillna(0.0),
        3,
        ["Low", "Medium", "High"],
    )
    intensity_rows = []
    for grp in ["Low", "Medium", "High"]:
        sub = hetero_df.loc[hetero_df["impression_intensity_tercile"].astype(str) == grp]
        if sub.empty:
            continue
        intensity_rows.append(segment_plr(sub, "impression_intensity_tercile", grp))
    hetero_intensity_df = pd.DataFrame(intensity_rows, columns=SEGMENT_COLS)
else:
    hetero_intensity_df = pd.DataFrame(columns=SEGMENT_COLS)
    print("Skipping intensity heterogeneity: no impression-intensity column found.")

display(hetero_intensity_df)
save_table(hetero_intensity_df, "table_heterogeneity_impression_intensity")

hetero_frames = [df for df in [hetero_propensity_df, hetero_recency_df, hetero_intensity_df] if not df.empty]
hetero_all_df = pd.concat(hetero_frames, axis=0, ignore_index=True) if hetero_frames else pd.DataFrame(columns=SEGMENT_COLS)
display(hetero_all_df.head(30))
save_table(hetero_all_df, "table_heterogeneity_all_segments")

heterogeneity_inputs = {
    "recency_col": recency_col,
    "intensity_col": intensity_col,
    "n_overlap_rows": int(hetero_df.shape[0]),
    "n_overlap_users": int(hetero_df["uid"].nunique()),
}
if "save_named_json" in globals():
    save_named_json(heterogeneity_inputs, "summary_heterogeneity_inputs")

plot_segment_effects(hetero_propensity_df, "Heterogeneity by Propensity Quintile", "figure_heterogeneity_propensity_quintile.png")
plot_segment_effects(hetero_recency_df, "Heterogeneity by Prior Click Recency", "figure_heterogeneity_recency.png")
plot_segment_effects(hetero_intensity_df, "Heterogeneity by Impression Intensity", "figure_heterogeneity_impression_intensity.png")


## 8. RAMP Layer 3: Campaign-Level Aggregation

This section uses the cross-fitted residualized benchmark from Step D to estimate segment-level effects across interpretable observable states. The estimands below are selection-adjusted heterogeneity estimates, not raw subgroup comparisons. If Step D spec changes, rerun Step G2 and Step H2.


In [ ]:
# ====== Step G2: selection-adjusted heterogeneity across observable states ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "dml_analysis_df" in globals(), "dml_analysis_df not found. Run Step D4 first."
assert "save_table" in globals(), "save_table not found. Run Step F0 first."

DML_HETERO_SEGMENT_COLS = [
    "segment_name",
    "segment_value",
    "n_eff",
    "n_users",
    "d_mean",
    "y_mean",
    "ate",
    "se_cluster",
    "ci_low",
    "ci_high",
    "t_stat",
    "status",
]

DML_HETERO_FIG_DIR = PAPER_OUT if "PAPER_OUT" in globals() else PAPER_OUTPUT_DIR
DML_HETERO_FIG_DIR.mkdir(parents=True, exist_ok=True)


def _ranked_qcut_with_fallback(series, primary_labels, fallback_labels=None):
    ser = pd.to_numeric(pd.Series(series), errors="coerce")
    fill_value = float(ser.dropna().median()) if ser.notna().any() else 0.0
    ranked = ser.fillna(fill_value).rank(method="first")
    attempts = [(len(primary_labels), primary_labels, f"{len(primary_labels)}_quantile")]
    if fallback_labels is not None:
        attempts.append((len(fallback_labels), fallback_labels, f"{len(fallback_labels)}_quantile_fallback"))
    for q, labels, scheme in attempts:
        if ranked.nunique() < q:
            continue
        try:
            bucket = pd.qcut(ranked, q=q, labels=labels)
            return pd.Series(bucket, index=ser.index), scheme
        except ValueError:
            continue
    single_label = primary_labels[0] if primary_labels else "all"
    return pd.Series([single_label] * len(ser), index=ser.index, dtype="object"), "single_bucket_fallback"


def estimate_segment_plr(
    df_seg,
    *,
    segment_name,
    segment_value,
    cluster_col="uid",
    min_obs=200,
    min_users=50,
    weak_denominator_tol=1e-8,
):
    df_seg = df_seg.copy()
    valid_mask = (
        pd.to_numeric(df_seg.get("D_tilde"), errors="coerce").notna()
        & pd.to_numeric(df_seg.get("Y_tilde"), errors="coerce").notna()
        & df_seg[cluster_col].notna()
    )
    df_seg = df_seg.loc[valid_mask].copy()

    n_eff = int(df_seg.shape[0])
    n_users = int(df_seg[cluster_col].nunique()) if n_eff else 0
    result = {
        "segment_name": str(segment_name),
        "segment_value": str(segment_value),
        "n_eff": n_eff,
        "n_users": n_users,
        "d_mean": float(pd.to_numeric(df_seg["D"], errors="coerce").mean()) if n_eff else np.nan,
        "y_mean": float(pd.to_numeric(df_seg["Y"], errors="coerce").mean()) if n_eff else np.nan,
        "ate": np.nan,
        "se_cluster": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "t_stat": np.nan,
        "status": "too_small",
    }
    if n_eff == 0:
        result["status"] = "no_finite_rows"
        return result
    if n_eff < int(min_obs) or n_users < int(min_users):
        result["status"] = "too_small"
        return result

    d_tilde = pd.to_numeric(df_seg["D_tilde"], errors="coerce").to_numpy(dtype=float)
    y_tilde = pd.to_numeric(df_seg["Y_tilde"], errors="coerce").to_numpy(dtype=float)
    den_sum = float(np.sum(np.square(d_tilde)))
    den_mean = float(np.mean(np.square(d_tilde))) if n_eff else np.nan
    if (not np.isfinite(den_sum)) or (not np.isfinite(den_mean)) or den_sum <= 0 or den_mean <= weak_denominator_tol:
        result["status"] = "weak_overlap"
        return result

    theta_hat_seg = float(np.sum(d_tilde * y_tilde) / den_sum)
    residual = y_tilde - theta_hat_seg * d_tilde
    influence = (d_tilde * residual) / den_mean

    if "cluster_se_from_if" in globals():
        se_cluster_seg, cluster_count = cluster_se_from_if(influence, df_seg[cluster_col].to_numpy())
        se_cluster_seg = float(se_cluster_seg) if pd.notna(se_cluster_seg) else np.nan
    else:
        tmp_if = pd.DataFrame({cluster_col: df_seg[cluster_col].to_numpy(), "if_value": influence})
        cluster_sum = tmp_if.groupby(cluster_col, sort=False)["if_value"].sum().to_numpy(dtype=float)
        cluster_count = len(cluster_sum)
        if cluster_count <= 1 or len(influence) <= 1:
            se_cluster_seg = np.nan
        else:
            centered = cluster_sum - cluster_sum.mean()
            var_theta = (cluster_count / (cluster_count - 1.0)) * np.sum(centered ** 2) / (len(influence) ** 2)
            se_cluster_seg = float(np.sqrt(max(var_theta, 0.0)))

    result.update(
        {
            "ate": theta_hat_seg,
            "se_cluster": se_cluster_seg,
            "ci_low": float(theta_hat_seg - 1.96 * se_cluster_seg) if np.isfinite(se_cluster_seg) else np.nan,
            "ci_high": float(theta_hat_seg + 1.96 * se_cluster_seg) if np.isfinite(se_cluster_seg) else np.nan,
            "t_stat": float(theta_hat_seg / se_cluster_seg) if np.isfinite(se_cluster_seg) and se_cluster_seg > 0 else np.nan,
            "status": "ok" if np.isfinite(se_cluster_seg) else "se_unavailable",
        }
    )
    if cluster_count <= 1:
        result["status"] = "too_few_clusters"
    return result


def _choose_recency_days(series_df):
    candidate_specs = [
        ("days_since_click", "days_since_click"),
        ("time_since_last_click_sec", "time_since_last_click_sec"),
        ("log_days_since_click", "log_days_since_click"),
        ("log_time_since_last_click", "log_time_since_last_click"),
    ]
    for col_name, mode in candidate_specs:
        if col_name not in series_df.columns:
            continue
        raw = pd.to_numeric(series_df[col_name], errors="coerce")
        if mode == "time_since_last_click_sec":
            recency_days = raw / 86400.0
        elif mode in {"log_days_since_click", "log_time_since_last_click"}:
            recency_days = np.expm1(raw)
            if mode == "log_time_since_last_click":
                recency_days = recency_days / 86400.0
        else:
            recency_days = raw
        return recency_days, col_name
    return None, None


def _choose_intensity_series(series_df):
    for col_name in ["imp_cnt_24h_pre", "imp_hist_7", "imp_hist_3", "n_impressions"]:
        if col_name in series_df.columns:
            return pd.to_numeric(series_df[col_name], errors="coerce").fillna(0.0), col_name
    return None, None


def _plot_dml_adjusted_effects(df_plot, title, filename):
    plot_df = df_plot.loc[df_plot["status"] == "ok"].copy().reset_index(drop=True)
    if plot_df.empty:
        print(f"Skipping figure {filename}: no estimable segments.")
        return None

    y_pos = np.arange(len(plot_df))
    y_labels = [f"{label} (n={int(n_eff):,})" for label, n_eff in zip(plot_df["segment_value"], plot_df["n_eff"])]
    x_err = np.vstack(
        [
            np.clip(plot_df["ate"] - plot_df["ci_low"], a_min=0, a_max=None),
            np.clip(plot_df["ci_high"] - plot_df["ate"], a_min=0, a_max=None),
        ]
    )

    fig, ax = plt.subplots(figsize=(8.0, 0.8 + 0.9 * len(plot_df)))
    ax.errorbar(plot_df["ate"], y_pos, xerr=x_err, fmt="o", color="#0B6E4F", ecolor="#6B7280", capsize=3)
    ax.axvline(0.0, linestyle="--", color="black", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("DML-adjusted segment effect")
    ax.set_ylabel("")
    ax.set_title(title)
    fig.tight_layout()
    out_path = DML_HETERO_FIG_DIR / filename
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)
    return out_path


heterogeneity_base_df = dml_analysis_df.copy().reset_index(drop=True)
heterogeneity_stepd_source = dml_analysis_metadata.get("stepd_payload_key", "unknown") if "dml_analysis_metadata" in globals() else "unknown"

propensity_bucket, propensity_bucket_scheme = _ranked_qcut_with_fallback(
    heterogeneity_base_df["mhat"],
    ["Q1_low", "Q2", "Q3", "Q4", "Q5_high"],
    ["T1_low", "T2", "T3_high"],
)
heterogeneity_base_df["propensity_state_bucket"] = propensity_bucket.astype(str)
if str(propensity_bucket.dtype) == "category":
    propensity_order = [str(v) for v in propensity_bucket.cat.categories]
else:
    propensity_order = list(pd.unique(heterogeneity_base_df["propensity_state_bucket"]))

propensity_rows = []
for bucket_label in propensity_order:
    sub = heterogeneity_base_df.loc[heterogeneity_base_df["propensity_state_bucket"] == bucket_label]
    propensity_rows.append(
        estimate_segment_plr(
            sub,
            segment_name="propensity_bucket",
            segment_value=bucket_label,
            cluster_col="uid",
        )
    )
dml_heterogeneity_propensity_df = pd.DataFrame(propensity_rows, columns=DML_HETERO_SEGMENT_COLS)
dml_heterogeneity_propensity_df["heterogeneity_family"] = "propensity"
dml_heterogeneity_propensity_df["segment_order"] = range(1, len(dml_heterogeneity_propensity_df) + 1)
dml_heterogeneity_propensity_df["source_variable"] = "mhat"
dml_heterogeneity_propensity_df["bucket_scheme"] = propensity_bucket_scheme
dml_heterogeneity_propensity_df["stepd_payload_key"] = heterogeneity_stepd_source
save_table(dml_heterogeneity_propensity_df, "table_heterogeneity_propensity")

recency_days, recency_source_col = _choose_recency_days(heterogeneity_base_df)
recency_order = ["no prior click", "<=1 day", "1-7 days", "7-30 days", "30+ days"]
if recency_days is not None:
    recency_bucket = pd.cut(
        recency_days,
        bins=[-np.inf, 1.0, 7.0, 30.0, np.inf],
        labels=recency_order[1:],
    ).astype(object)
    if "no_prior_click" in heterogeneity_base_df.columns:
        no_prior_mask = pd.to_numeric(heterogeneity_base_df["no_prior_click"], errors="coerce").fillna(0).astype(int) == 1
        recency_bucket.loc[no_prior_mask] = "no prior click"
    heterogeneity_base_df["recency_state_bucket"] = pd.Categorical(recency_bucket, categories=recency_order, ordered=True)
    recency_rows = []
    for bucket_label in recency_order:
        sub = heterogeneity_base_df.loc[heterogeneity_base_df["recency_state_bucket"].astype(str) == bucket_label]
        if sub.empty:
            continue
        recency_rows.append(
            estimate_segment_plr(
                sub,
                segment_name="recency_bucket",
                segment_value=bucket_label,
                cluster_col="uid",
            )
        )
    dml_heterogeneity_recency_df = pd.DataFrame(recency_rows, columns=DML_HETERO_SEGMENT_COLS)
else:
    dml_heterogeneity_recency_df = pd.DataFrame(columns=DML_HETERO_SEGMENT_COLS)
    print("Skipping recency-based DML-adjusted heterogeneity: no usable recency state column found.")

dml_heterogeneity_recency_df["heterogeneity_family"] = "recency"
dml_heterogeneity_recency_df["segment_order"] = range(1, len(dml_heterogeneity_recency_df) + 1)
dml_heterogeneity_recency_df["source_variable"] = recency_source_col
dml_heterogeneity_recency_df["bucket_scheme"] = "interpretable_recency_buckets"
dml_heterogeneity_recency_df["stepd_payload_key"] = heterogeneity_stepd_source
save_table(dml_heterogeneity_recency_df, "table_heterogeneity_recency")

intensity_series, intensity_source_col = _choose_intensity_series(heterogeneity_base_df)
if intensity_series is not None:
    intensity_bucket, intensity_bucket_scheme = _ranked_qcut_with_fallback(
        intensity_series,
        ["low", "medium", "high"],
        None,
    )
    heterogeneity_base_df["intensity_state_bucket"] = intensity_bucket.astype(str)
    if str(intensity_bucket.dtype) == "category":
        intensity_order = [str(v) for v in intensity_bucket.cat.categories]
    else:
        intensity_order = list(pd.unique(heterogeneity_base_df["intensity_state_bucket"]))
    intensity_rows = []
    for bucket_label in intensity_order:
        sub = heterogeneity_base_df.loc[heterogeneity_base_df["intensity_state_bucket"] == bucket_label]
        intensity_rows.append(
            estimate_segment_plr(
                sub,
                segment_name="intensity_bucket",
                segment_value=bucket_label,
                cluster_col="uid",
            )
        )
    dml_heterogeneity_intensity_df = pd.DataFrame(intensity_rows, columns=DML_HETERO_SEGMENT_COLS)
else:
    intensity_bucket_scheme = None
    dml_heterogeneity_intensity_df = pd.DataFrame(columns=DML_HETERO_SEGMENT_COLS)
    print("Skipping intensity-based DML-adjusted heterogeneity: no usable intensity state column found.")

dml_heterogeneity_intensity_df["heterogeneity_family"] = "intensity"
dml_heterogeneity_intensity_df["segment_order"] = range(1, len(dml_heterogeneity_intensity_df) + 1)
dml_heterogeneity_intensity_df["source_variable"] = intensity_source_col
dml_heterogeneity_intensity_df["bucket_scheme"] = intensity_bucket_scheme
dml_heterogeneity_intensity_df["stepd_payload_key"] = heterogeneity_stepd_source
save_table(dml_heterogeneity_intensity_df, "table_heterogeneity_intensity")

heterogeneity_frames = [
    dml_heterogeneity_propensity_df,
    dml_heterogeneity_recency_df,
    dml_heterogeneity_intensity_df,
]
heterogeneity_frames = [df for df in heterogeneity_frames if df is not None and not df.empty]
dml_heterogeneity_all_df = (
    pd.concat(heterogeneity_frames, axis=0, ignore_index=True, sort=False)
    if heterogeneity_frames
    else pd.DataFrame(columns=DML_HETERO_SEGMENT_COLS + ["heterogeneity_family", "segment_order", "source_variable", "bucket_scheme", "stepd_payload_key"])
)
save_table(dml_heterogeneity_all_df, "table_heterogeneity_all")

ok_segments_df = dml_heterogeneity_all_df.loc[dml_heterogeneity_all_df["status"] == "ok"].copy().reset_index(drop=True)
if not ok_segments_df.empty:
    largest_row = ok_segments_df.sort_values("ate", ascending=False).iloc[0]
    smallest_row = ok_segments_df.sort_values("ate", ascending=True).iloc[0]
else:
    largest_row = None
    smallest_row = None

family_stats_rows = []
for family_name, fam_df in dml_heterogeneity_all_df.groupby("heterogeneity_family", dropna=False):
    ok_df = fam_df.loc[fam_df["status"] == "ok"].copy()
    family_stats_rows.append(
        {
            "heterogeneity_family": str(family_name),
            "n_segments": int(len(fam_df)),
            "n_ok_segments": int(len(ok_df)),
            "ok_share": float(len(ok_df) / len(fam_df)) if len(fam_df) else np.nan,
            "effect_range": float(ok_df["ate"].max() - ok_df["ate"].min()) if len(ok_df) >= 2 else np.nan,
            "avg_ci_width": float((ok_df["ci_high"] - ok_df["ci_low"]).mean()) if not ok_df.empty else np.nan,
            "avg_abs_t": float(ok_df["t_stat"].abs().mean()) if not ok_df.empty else np.nan,
        }
    )
heterogeneity_family_stats_df = pd.DataFrame(family_stats_rows)

most_stable_row = None
stable_candidates = heterogeneity_family_stats_df.loc[
    heterogeneity_family_stats_df["n_ok_segments"] >= 1
].sort_values(["avg_ci_width", "ok_share"], ascending=[True, False])
if not stable_candidates.empty:
    most_stable_row = stable_candidates.iloc[0]

most_managerial_row = None
managerial_candidates = heterogeneity_family_stats_df.loc[
    heterogeneity_family_stats_df["n_ok_segments"] >= 2
].sort_values(["effect_range", "avg_abs_t"], ascending=[False, False])
if not managerial_candidates.empty:
    most_managerial_row = managerial_candidates.iloc[0]

summary_heterogeneity_managerial = {
    "title": "DML-adjusted heterogeneity across observable states",
    "stepd_payload_key": heterogeneity_stepd_source,
    "restriction_name": dml_analysis_metadata.get("restriction_name") if "dml_analysis_metadata" in globals() else None,
    "outcome_hours": dml_analysis_metadata.get("outcome_hours") if "dml_analysis_metadata" in globals() else None,
    "trim_eps": dml_analysis_metadata.get("trim_eps") if "dml_analysis_metadata" in globals() else None,
    "outcome_mode": dml_analysis_metadata.get("outcome_mode") if "dml_analysis_metadata" in globals() else None,
    "propensity_bucket_scheme": propensity_bucket_scheme,
    "recency_source_variable": recency_source_col,
    "intensity_source_variable": intensity_source_col,
    "largest_segment_effect": dml_to_builtin(largest_row.to_dict()) if largest_row is not None else None,
    "smallest_segment_effect": dml_to_builtin(smallest_row.to_dict()) if smallest_row is not None else None,
    "largest_minus_smallest": float(largest_row["ate"] - smallest_row["ate"]) if (largest_row is not None and smallest_row is not None) else None,
    "most_stable_mode": dml_to_builtin(most_stable_row.to_dict()) if most_stable_row is not None else None,
    "most_stable_mode_basis": "Smallest average 95% confidence-interval width among families with at least one estimable segment.",
    "most_managerial_mode": dml_to_builtin(most_managerial_row.to_dict()) if most_managerial_row is not None else None,
    "most_managerial_mode_basis": "Largest within-family spread in DML-adjusted segment effects among estimable observable-state families.",
    "writer_note": "Effect heterogeneity is systematic across observable user-state buckets in the cross-fitted residualized benchmark.",
}
save_named_json(dml_to_builtin(summary_heterogeneity_managerial), "summary_heterogeneity_managerial")

_plot_dml_adjusted_effects(
    dml_heterogeneity_propensity_df,
    "DML-adjusted heterogeneity by propensity state",
    "fig_heterogeneity_propensity.png",
)
_plot_dml_adjusted_effects(
    dml_heterogeneity_recency_df,
    "DML-adjusted heterogeneity by recency state",
    "fig_heterogeneity_recency.png",
)
_plot_dml_adjusted_effects(
    dml_heterogeneity_intensity_df,
    "DML-adjusted heterogeneity by intensity state",
    "fig_heterogeneity_intensity.png",
)

display(dml_heterogeneity_propensity_df)
display(dml_heterogeneity_recency_df)
display(dml_heterogeneity_intensity_df)


## 8. RAMP Layer 3: Campaign-Level Aggregation

This section builds a lightweight campaign-level diagnostic contribution proxy from the residualized Step D objects. It is intended as a first-pass campaign ranking exercise, not a final path-level attribution rule.


In [ ]:
# ====== Step H: Campaign-level diagnostic credit proxy and rank shifts ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "get_stepd_payload" in globals(), "get_stepd_payload not found. Run Step D first."
assert "save_table" in globals(), "save_table not found. Run Step F0 first."

STEPD_DOWNSTREAM_SOURCE = globals().get(
    "STEPD_DOWNSTREAM_SOURCE",
    "final" if "final" in globals().get("STEPD_PAYLOADS", {}) else "baseline",
)

stepd_payload_for_h = get_stepd_payload(STEPD_DOWNSTREAM_SOURCE)
selected_base_df = stepd_payload_for_h["df"].copy().reset_index(drop=True)

if "hetero_df" in globals():
    campaign_df = hetero_df.copy().reset_index(drop=True)
    print("Step H using hetero_df built from Step D payload:", STEPD_DOWNSTREAM_SOURCE)
else:
    stepd_payload = stepd_payload_for_h
    base_df_selected = stepd_payload["df"].copy().reset_index(drop=True)
    campaign_df = base_df_selected.copy().reset_index(drop=True)
    campaign_df["D"] = np.asarray(stepd_payload["D"]).astype(int)
    campaign_df["Y"] = np.asarray(stepd_payload["Y"]).astype(int)
    trim_eps_selected = float(stepd_payload["summary"].get("trim_eps", 0.01))
    campaign_df["mhat"] = np.clip(np.asarray(stepd_payload["mhat"], dtype=float), max(trim_eps_selected, 1e-6), 1 - max(trim_eps_selected, 1e-6))
    campaign_df["ghat"] = np.asarray(stepd_payload["ghat"], dtype=float)
    campaign_df["keep"] = np.asarray(stepd_payload["keep"]).astype(bool)
    campaign_df = campaign_df.loc[campaign_df["keep"]].copy().reset_index(drop=True)
    campaign_df["D_tilde"] = campaign_df["D"] - campaign_df["mhat"]
    campaign_df["Y_tilde"] = campaign_df["Y"] - campaign_df["ghat"]
    print("Step H using direct Step D payload:", stepd_payload.get("label", STEPD_DOWNSTREAM_SOURCE))

campaign_df["campaign"] = campaign_df["campaign"].astype(str)
campaign_df["diagnostic_score_proxy"] = campaign_df["D_tilde"] * campaign_df["Y_tilde"]

campaign_credit = (
    campaign_df.groupby("campaign", as_index=False)
              .agg(
                  n_obs=("campaign", "size"),
                  n_users=("uid", "nunique"),
                  n_clicks=("D", "sum"),
                  y_mean=("Y", "mean"),
                  score_sum=("diagnostic_score_proxy", "sum"),
                  score_mean=("diagnostic_score_proxy", "mean"),
              )
)
campaign_credit["incremental_proxy"] = campaign_credit["score_sum"]
campaign_credit["incremental_proxy_pos"] = campaign_credit["incremental_proxy"].clip(lower=0.0)
total_pos = float(campaign_credit["incremental_proxy_pos"].sum())
campaign_credit["diagnostic_share_proxy"] = (
    campaign_credit["incremental_proxy_pos"] / total_pos if total_pos > 0 else np.nan
)
campaign_credit["diagnostic_rank_proxy"] = campaign_credit["incremental_proxy"].rank(ascending=False, method="min")
campaign_credit = campaign_credit.sort_values("diagnostic_rank_proxy").reset_index(drop=True)
display(campaign_credit.head(20))
save_table(campaign_credit, "table_campaign_diagnostic_score_proxy")

base_df = selected_base_df.copy().reset_index(drop=True)
base_df["campaign"] = base_df["campaign"].astype(str)

heur_exposure = (
    base_df.groupby("campaign", as_index=False)
           .agg(exposure_count=("campaign", "size"))
)
heur_exposure["exposure_share"] = heur_exposure["exposure_count"] / heur_exposure["exposure_count"].sum()
heur_exposure["exposure_rank"] = heur_exposure["exposure_count"].rank(ascending=False, method="min")

heur_click = (
    base_df.groupby("campaign", as_index=False)
           .agg(click_count=("click", "sum"))
)
total_clicks = float(heur_click["click_count"].sum())
heur_click["click_share"] = heur_click["click_count"] / total_clicks if total_clicks > 0 else np.nan
heur_click["click_rank"] = heur_click["click_count"].rank(ascending=False, method="min")

COMPUTE_LAST_TOUCH_BASELINE = False
if COMPUTE_LAST_TOUCH_BASELINE and "df" in globals():
    raw_for_last_touch = df.copy()
    raw_for_last_touch["campaign"] = raw_for_last_touch["campaign"].astype(str)
    time_col = "ts_sec" if "ts_sec" in raw_for_last_touch.columns else "timestamp"
    raw_for_last_touch[time_col] = pd.to_numeric(raw_for_last_touch[time_col], errors="coerce")
    raw_for_last_touch = raw_for_last_touch.sort_values(["uid", time_col])

    last_touch_rows = []
    for uid, g in raw_for_last_touch.groupby("uid", sort=False):
        conv_times = g.loc[g["conversion"] == 1, time_col].dropna().to_numpy()
        if len(conv_times) == 0:
            continue
        touches = g.loc[g[time_col].notna(), [time_col, "campaign"]]
        for conv_time in conv_times:
            prior = touches.loc[touches[time_col] < conv_time]
            if prior.empty:
                continue
            last_touch_rows.append(prior.iloc[-1]["campaign"])

    if last_touch_rows:
        heur_last_touch = (
            pd.Series(last_touch_rows, name="campaign")
              .value_counts()
              .rename_axis("campaign")
              .reset_index(name="last_touch_credit")
        )
        heur_last_touch["last_touch_share"] = heur_last_touch["last_touch_credit"] / heur_last_touch["last_touch_credit"].sum()
        heur_last_touch["last_touch_rank"] = heur_last_touch["last_touch_credit"].rank(ascending=False, method="min")
    else:
        heur_last_touch = pd.DataFrame(columns=["campaign", "last_touch_credit", "last_touch_share", "last_touch_rank"])
else:
    heur_last_touch = pd.DataFrame(columns=["campaign", "last_touch_credit", "last_touch_share", "last_touch_rank"])
    print("Skipping last-touch baseline by default to keep Step H lightweight. Set COMPUTE_LAST_TOUCH_BASELINE=True to enable it.")

display(heur_exposure.head())
display(heur_click.head())
if not heur_last_touch.empty:
    display(heur_last_touch.head())

save_table(heur_exposure, "table_heuristic_exposure")
save_table(heur_click, "table_heuristic_click")
save_table(heur_last_touch, "table_heuristic_last_touch")

compare_df = campaign_credit.copy()
compare_df = compare_df.merge(
    heur_exposure[["campaign", "exposure_count", "exposure_share", "exposure_rank"]],
    on="campaign",
    how="left",
)
compare_df = compare_df.merge(
    heur_click[["campaign", "click_count", "click_share", "click_rank"]],
    on="campaign",
    how="left",
)
if not heur_last_touch.empty:
    compare_df = compare_df.merge(
        heur_last_touch[["campaign", "last_touch_credit", "last_touch_share", "last_touch_rank"]],
        on="campaign",
        how="left",
    )

compare_df["rank_shift_exposure"] = compare_df["exposure_rank"] - compare_df["diagnostic_rank_proxy"]
compare_df["rank_shift_click"] = compare_df["click_rank"] - compare_df["diagnostic_rank_proxy"]
compare_df["share_shift_exposure"] = compare_df["diagnostic_share_proxy"] - compare_df["exposure_share"]
compare_df["share_shift_click"] = compare_df["diagnostic_share_proxy"] - compare_df["click_share"]

if "last_touch_rank" in compare_df.columns:
    compare_df["rank_shift_last_touch"] = compare_df["last_touch_rank"] - compare_df["diagnostic_rank_proxy"]
if "last_touch_share" in compare_df.columns:
    compare_df["share_shift_last_touch"] = compare_df["diagnostic_share_proxy"] - compare_df["last_touch_share"]

compare_df["heuristic_winner_diagnostic_loser"] = (
    (compare_df["exposure_rank"] <= 10) & (compare_df["diagnostic_rank_proxy"] > 20)
)
compare_df["heuristic_loser_diagnostic_winner"] = (
    (compare_df["exposure_rank"] > 20) & (compare_df["diagnostic_rank_proxy"] <= 10)
)
compare_df = compare_df.sort_values("diagnostic_rank_proxy").reset_index(drop=True)

display(compare_df.head(30))
save_table(compare_df, "table_campaign_rank_shift")

def _safe_spearman(a, b):
    frame = pd.DataFrame({"a": a, "b": b}).dropna()
    if frame.shape[0] < 2:
        return np.nan
    return float(frame["a"].corr(frame["b"], method="spearman"))

summary_attr = {
    "corr_exposure_vs_diagnostic_rank": _safe_spearman(compare_df.get("exposure_rank"), compare_df.get("diagnostic_rank_proxy")),
    "corr_click_vs_diagnostic_rank": _safe_spearman(compare_df.get("click_rank"), compare_df.get("diagnostic_rank_proxy")),
    "mean_abs_rank_shift_exposure": float(compare_df["rank_shift_exposure"].abs().mean()) if not compare_df.empty else np.nan,
    "mean_abs_rank_shift_click": float(compare_df["rank_shift_click"].abs().mean()) if not compare_df.empty else np.nan,
    "mean_abs_share_shift_exposure": float(compare_df["share_shift_exposure"].abs().mean()) if not compare_df.empty else np.nan,
    "mean_abs_share_shift_click": float(compare_df["share_shift_click"].abs().mean()) if not compare_df.empty else np.nan,
    "n_heuristic_winner_diagnostic_loser": int(compare_df["heuristic_winner_diagnostic_loser"].sum()) if not compare_df.empty else 0,
    "n_heuristic_loser_diagnostic_winner": int(compare_df["heuristic_loser_diagnostic_winner"].sum()) if not compare_df.empty else 0,
    "stepd_source": STEPD_DOWNSTREAM_SOURCE,
}

if "rank_shift_last_touch" in compare_df.columns:
    summary_attr["corr_last_touch_vs_diagnostic_rank"] = _safe_spearman(compare_df.get("last_touch_rank"), compare_df.get("diagnostic_rank_proxy"))
    summary_attr["mean_abs_rank_shift_last_touch"] = float(compare_df["rank_shift_last_touch"].abs().mean()) if not compare_df.empty else np.nan
if "share_shift_last_touch" in compare_df.columns:
    summary_attr["mean_abs_share_shift_last_touch"] = float(compare_df["share_shift_last_touch"].abs().mean()) if not compare_df.empty else np.nan

display(pd.DataFrame(list(summary_attr.items()), columns=["metric", "value"]))
if "save_named_json" in globals():
    save_named_json(summary_attr, "summary_campaign_rank_shift")

CAMPAIGN_FIG_DIR = PAPER_OUT if "PAPER_OUT" in globals() else PAPER_OUTPUT_DIR
CAMPAIGN_FIG_DIR.mkdir(parents=True, exist_ok=True)

def _scatter_compare(x, y, xlabel, ylabel, title, filename):
    frame = pd.DataFrame({"x": x, "y": y}).dropna()
    if frame.empty:
        return None
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(frame["x"], frame["y"], alpha=0.7)
    lim = float(max(frame["x"].max(), frame["y"].max())) if len(frame) else 0.0
    if lim > 0:
        ax.plot([0, lim], [0, lim], linestyle="--", color="black", linewidth=1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    out_path = CAMPAIGN_FIG_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)
    return out_path

def _plot_rank_shift(frame, filename):
    if frame.empty:
        return None
    plot_df = frame.copy()
    plot_df["abs_rank_shift_exposure"] = plot_df["rank_shift_exposure"].abs()
    plot_df = plot_df.sort_values("abs_rank_shift_exposure", ascending=False).head(15)
    if plot_df.empty:
        return None
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df["campaign"].astype(str), plot_df["rank_shift_exposure"])
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("Exposure rank - diagnostic rank")
    ax.set_title("Largest campaign rank shifts (exposure vs diagnostic)")
    fig.tight_layout()
    out_path = CAMPAIGN_FIG_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)
    return out_path

_scatter_compare(compare_df.get("exposure_share"), compare_df.get("diagnostic_share_proxy"), "Exposure share", "Diagnostic share proxy", "Exposure-based vs Diagnostic campaign importance", "figure_campaign_exposure_vs_diagnostic_share.png")
_scatter_compare(compare_df.get("click_share"), compare_df.get("diagnostic_share_proxy"), "Click share", "Diagnostic share proxy", "Click-based vs Diagnostic campaign importance", "figure_campaign_click_vs_diagnostic_share.png")
_plot_rank_shift(compare_df, "figure_campaign_rank_shift_exposure_vs_diagnostic.png")

## 8. RAMP Layer 3: Campaign-Level Aggregation

This section constructs a selection-adjusted campaign signal from the residualized Step D objects. The resulting ranking is conditional on observed support and intended for managerial prioritization and screening, not as a universal diagnostic league table across campaigns that reach different users.


In [ ]:
# ====== Step H2: Support-aware selection-adjusted campaign ranking ======
import numpy as np
import pandas as pd

assert "dml_analysis_df" in globals(), "dml_analysis_df not found. Run Step D4 first."
assert "save_table" in globals(), "save_table not found. Run Step F0 first."

CAMPAIGN_MIN_OBS = 500
CAMPAIGN_MIN_USERS = 100
CAMPAIGN_MIN_CLICKS = 20


def _rank_desc(series):
    ser = pd.to_numeric(pd.Series(series), errors="coerce")
    rank = ser.rank(ascending=False, method="min")
    return rank.astype("Int64")


def _safe_corr(a, b):
    frame = pd.DataFrame({"a": a, "b": b}).dropna()
    if frame.shape[0] < 2:
        return np.nan
    return float(frame["a"].corr(frame["b"], method="spearman"))


def _top_overlap(frame, rank_a, rank_b, k):
    if frame.empty or rank_a not in frame.columns or rank_b not in frame.columns:
        return None
    use = frame[["campaign", rank_a, rank_b]].dropna()
    if use.empty:
        return None
    effective_k = int(min(k, len(use)))
    top_a = set(use.loc[use[rank_a] <= effective_k, "campaign"].astype(str))
    top_b = set(use.loc[use[rank_b] <= effective_k, "campaign"].astype(str))
    return {
        "k": effective_k,
        "overlap_count": int(len(top_a & top_b)),
        "top_a_only": sorted(list(top_a - top_b))[:10],
        "top_b_only": sorted(list(top_b - top_a))[:10],
    }


def _build_last_touch_baseline(candidate_campaigns):
    empty = pd.DataFrame(columns=["campaign", "last_touch_credit", "last_touch_share", "last_touch_rank"])
    if "df" not in globals():
        print("Step H2 last-touch baseline unavailable: raw df is not in notebook memory.")
        return empty

    raw_for_last_touch = df.copy()
    if "ts_sec" not in raw_for_last_touch.columns and "build_conversion_event_columns" in globals():
        try:
            raw_for_last_touch, _ = build_conversion_event_columns(raw_for_last_touch)
        except Exception as exc:
            print("Step H2 last-touch baseline skipped while constructing ts_sec:", exc)
            return empty

    time_col = "ts_sec" if "ts_sec" in raw_for_last_touch.columns else ("timestamp" if "timestamp" in raw_for_last_touch.columns else None)
    if time_col is None:
        print("Step H2 last-touch baseline unavailable: no usable time column.")
        return empty

    raw_for_last_touch[time_col] = pd.to_numeric(raw_for_last_touch[time_col], errors="coerce")
    raw_for_last_touch["conversion"] = (
        pd.to_numeric(raw_for_last_touch["conversion"], errors="coerce").fillna(0).astype(int) > 0
    ).astype(int)
    raw_for_last_touch = raw_for_last_touch.loc[
        raw_for_last_touch["uid"].notna() & raw_for_last_touch[time_col].notna() & raw_for_last_touch["campaign"].notna()
    ].copy()
    if raw_for_last_touch.empty:
        return empty

    raw_for_last_touch["campaign"] = raw_for_last_touch["campaign"].astype(str)
    raw_for_last_touch = raw_for_last_touch.sort_values(["uid", time_col]).reset_index(drop=True)

    last_touch_rows = []
    for _, group in raw_for_last_touch.groupby("uid", sort=False):
        conv_times = group.loc[group["conversion"] == 1, time_col].dropna().to_numpy()
        if conv_times.size == 0:
            continue
        touches = group.loc[group[time_col].notna(), [time_col, "campaign"]]
        for conv_time in conv_times:
            prior = touches.loc[touches[time_col] < conv_time]
            if prior.empty:
                continue
            picked_campaign = str(prior.iloc[-1]["campaign"]).strip()
            if picked_campaign:
                last_touch_rows.append(picked_campaign)

    if not last_touch_rows:
        print("Step H2 last-touch baseline unavailable: no attributable last touches found.")
        return empty

    heur_last_touch = (
        pd.Series(last_touch_rows, name="campaign")
        .value_counts()
        .rename_axis("campaign")
        .reset_index(name="last_touch_credit")
    )
    heur_last_touch = heur_last_touch.loc[heur_last_touch["campaign"].isin(set(candidate_campaigns))].copy()
    if heur_last_touch.empty:
        return empty
    total_credit = float(heur_last_touch["last_touch_credit"].sum())
    heur_last_touch["last_touch_share"] = heur_last_touch["last_touch_credit"] / total_credit if total_credit > 0 else np.nan
    heur_last_touch["last_touch_rank"] = _rank_desc(heur_last_touch["last_touch_credit"])
    return heur_last_touch.sort_values(["last_touch_rank", "campaign"]).reset_index(drop=True)


campaign_signal_df = dml_analysis_df.copy().reset_index(drop=True)
campaign_signal_df = campaign_signal_df.loc[campaign_signal_df["campaign"].notna()].copy()
campaign_signal_df["campaign"] = campaign_signal_df["campaign"].astype(str).str.strip()
campaign_signal_df = campaign_signal_df.loc[
    (campaign_signal_df["campaign"] != "")
    & (campaign_signal_df["campaign"].str.lower() != "nan")
].copy()
campaign_signal_df["score_proxy"] = pd.to_numeric(campaign_signal_df["D_tilde"], errors="coerce") * pd.to_numeric(campaign_signal_df["Y_tilde"], errors="coerce")
campaign_signal_df["selection_adjusted_campaign_signal"] = campaign_signal_df["score_proxy"]
campaign_signal_df["restriction_name"] = dml_analysis_metadata.get("restriction_name") if "dml_analysis_metadata" in globals() else None
campaign_signal_df["outcome_hours"] = dml_analysis_metadata.get("outcome_hours") if "dml_analysis_metadata" in globals() else None
campaign_signal_df["trim_eps"] = dml_analysis_metadata.get("trim_eps") if "dml_analysis_metadata" in globals() else None
campaign_signal_df["outcome_mode"] = dml_analysis_metadata.get("outcome_mode") if "dml_analysis_metadata" in globals() else None
campaign_signal_df["stepd_payload_key"] = dml_analysis_metadata.get("stepd_payload_key") if "dml_analysis_metadata" in globals() else None

selection_adjusted_campaign_signal_df = campaign_signal_df.copy()
if campaign_signal_df.empty:
    raise ValueError("Step H2 could not build campaign_signal_df because all kept rows have missing campaigns.")

campaign_support_diagnostics_df = (
    campaign_signal_df.groupby("campaign", as_index=False)
    .agg(
        n_obs=("campaign", "size"),
        n_users=("uid", "nunique"),
        n_clicks=("D", "sum"),
        score_sum=("score_proxy", "sum"),
        score_mean=("score_proxy", "mean"),
        mean_propensity=("mhat", "mean"),
        propensity_dispersion=("mhat", lambda x: float(pd.to_numeric(pd.Series(x), errors="coerce").std(ddof=0))),
        y_mean=("Y", "mean"),
        d_mean=("D", "mean"),
    )
)
campaign_support_diagnostics_df["support_share"] = campaign_support_diagnostics_df["n_obs"] / float(len(campaign_signal_df))
campaign_support_diagnostics_df["enough_obs"] = campaign_support_diagnostics_df["n_obs"] >= CAMPAIGN_MIN_OBS
campaign_support_diagnostics_df["enough_users"] = campaign_support_diagnostics_df["n_users"] >= CAMPAIGN_MIN_USERS
campaign_support_diagnostics_df["enough_clicks"] = campaign_support_diagnostics_df["n_clicks"] >= CAMPAIGN_MIN_CLICKS
campaign_support_diagnostics_df["support_sufficient"] = (
    campaign_support_diagnostics_df["enough_obs"]
    & campaign_support_diagnostics_df["enough_users"]
    & campaign_support_diagnostics_df["enough_clicks"]
)
campaign_support_diagnostics_df["stepd_payload_key"] = dml_analysis_metadata.get("stepd_payload_key") if "dml_analysis_metadata" in globals() else None
campaign_support_diagnostics_df["restriction_name"] = dml_analysis_metadata.get("restriction_name") if "dml_analysis_metadata" in globals() else None
campaign_support_diagnostics_df["outcome_hours"] = dml_analysis_metadata.get("outcome_hours") if "dml_analysis_metadata" in globals() else None
campaign_support_diagnostics_df["trim_eps"] = dml_analysis_metadata.get("trim_eps") if "dml_analysis_metadata" in globals() else None
campaign_support_diagnostics_df = campaign_support_diagnostics_df.sort_values(
    ["support_sufficient", "score_sum", "n_obs"],
    ascending=[False, False, False],
).reset_index(drop=True)
save_table(campaign_support_diagnostics_df, "table_campaign_support_diagnostics")

campaign_dml_informed_ranking_df = campaign_support_diagnostics_df.copy()
campaign_dml_informed_ranking_df["incremental_proxy"] = campaign_dml_informed_ranking_df["score_sum"]
campaign_dml_informed_ranking_df["incremental_proxy_pos"] = campaign_dml_informed_ranking_df["incremental_proxy"].clip(lower=0.0)
total_incremental_proxy_pos = float(campaign_dml_informed_ranking_df["incremental_proxy_pos"].sum())
campaign_dml_informed_ranking_df["diagnostic_share_proxy"] = (
    campaign_dml_informed_ranking_df["incremental_proxy_pos"] / total_incremental_proxy_pos
    if total_incremental_proxy_pos > 0
    else np.nan
)
campaign_dml_informed_ranking_df["diagnostic_rank_proxy"] = _rank_desc(campaign_dml_informed_ranking_df["incremental_proxy"])
campaign_dml_informed_ranking_df["dml_informed_ranking"] = campaign_dml_informed_ranking_df["diagnostic_rank_proxy"]
campaign_dml_informed_ranking_df["selection_adjusted_campaign_signal"] = campaign_dml_informed_ranking_df["score_sum"]
campaign_dml_informed_ranking_df["conditional_managerial_ranking"] = campaign_dml_informed_ranking_df["diagnostic_rank_proxy"]
campaign_dml_informed_ranking_df["ranking_scope"] = "within_observed_support"
campaign_dml_informed_ranking_df["ranking_interpretation"] = "managerial_prioritization_signal"
campaign_dml_informed_ranking_df = campaign_dml_informed_ranking_df.sort_values(
    ["diagnostic_rank_proxy", "campaign"],
    ascending=[True, True],
).reset_index(drop=True)
save_table(campaign_dml_informed_ranking_df, "table_campaign_dml_informed_ranking")

heur_exposure = (
    campaign_signal_df.groupby("campaign", as_index=False)
    .agg(exposure_count=("campaign", "size"))
)
heur_exposure["exposure_share"] = heur_exposure["exposure_count"] / float(heur_exposure["exposure_count"].sum())
heur_exposure["exposure_rank"] = _rank_desc(heur_exposure["exposure_count"])

heur_click = (
    campaign_signal_df.groupby("campaign", as_index=False)
    .agg(click_count=("D", "sum"))
)
total_click_count = float(heur_click["click_count"].sum())
heur_click["click_share"] = heur_click["click_count"] / total_click_count if total_click_count > 0 else np.nan
heur_click["click_rank"] = _rank_desc(heur_click["click_count"])

heur_last_touch = _build_last_touch_baseline(campaign_dml_informed_ranking_df["campaign"].astype(str).tolist())
if heur_last_touch.empty:
    heur_last_touch = pd.DataFrame(columns=["campaign", "last_touch_credit", "last_touch_share", "last_touch_rank"])

campaign_rank_shift_dml_informed_df = campaign_dml_informed_ranking_df.copy()
campaign_rank_shift_dml_informed_df = campaign_rank_shift_dml_informed_df.merge(
    heur_exposure[["campaign", "exposure_count", "exposure_share", "exposure_rank"]],
    on="campaign",
    how="left",
)
campaign_rank_shift_dml_informed_df = campaign_rank_shift_dml_informed_df.merge(
    heur_click[["campaign", "click_count", "click_share", "click_rank"]],
    on="campaign",
    how="left",
)
if not heur_last_touch.empty:
    campaign_rank_shift_dml_informed_df = campaign_rank_shift_dml_informed_df.merge(
        heur_last_touch[["campaign", "last_touch_credit", "last_touch_share", "last_touch_rank"]],
        on="campaign",
        how="left",
    )
else:
    campaign_rank_shift_dml_informed_df["last_touch_credit"] = np.nan
    campaign_rank_shift_dml_informed_df["last_touch_share"] = np.nan
    campaign_rank_shift_dml_informed_df["last_touch_rank"] = pd.Series([pd.NA] * len(campaign_rank_shift_dml_informed_df), dtype="Int64")

campaign_rank_shift_dml_informed_df["rank_shift_exposure"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["exposure_rank"], errors="coerce")
    - pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce")
)
campaign_rank_shift_dml_informed_df["rank_shift_click"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["click_rank"], errors="coerce")
    - pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce")
)
campaign_rank_shift_dml_informed_df["rank_shift_last_touch"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["last_touch_rank"], errors="coerce")
    - pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce")
)
campaign_rank_shift_dml_informed_df["share_shift_exposure"] = campaign_rank_shift_dml_informed_df["diagnostic_share_proxy"] - campaign_rank_shift_dml_informed_df["exposure_share"]
campaign_rank_shift_dml_informed_df["share_shift_click"] = campaign_rank_shift_dml_informed_df["diagnostic_share_proxy"] - campaign_rank_shift_dml_informed_df["click_share"]
campaign_rank_shift_dml_informed_df["share_shift_last_touch"] = campaign_rank_shift_dml_informed_df["diagnostic_share_proxy"] - campaign_rank_shift_dml_informed_df["last_touch_share"]

campaign_rank_shift_dml_informed_df["heuristic_winner_diagnostic_loser_exposure"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["exposure_rank"], errors="coerce") <= 10
) & (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce") > 20
)
campaign_rank_shift_dml_informed_df["heuristic_loser_diagnostic_winner_exposure"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["exposure_rank"], errors="coerce") > 20
) & (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce") <= 10
)
campaign_rank_shift_dml_informed_df["heuristic_winner_diagnostic_loser_click"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["click_rank"], errors="coerce") <= 10
) & (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce") > 20
)
campaign_rank_shift_dml_informed_df["heuristic_loser_diagnostic_winner_click"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["click_rank"], errors="coerce") > 20
) & (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce") <= 10
)
campaign_rank_shift_dml_informed_df["heuristic_winner_diagnostic_loser_last_touch"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["last_touch_rank"], errors="coerce") <= 10
) & (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce") > 20
)
campaign_rank_shift_dml_informed_df["heuristic_loser_diagnostic_winner_last_touch"] = (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["last_touch_rank"], errors="coerce") > 20
) & (
    pd.to_numeric(campaign_rank_shift_dml_informed_df["diagnostic_rank_proxy"], errors="coerce") <= 10
)
campaign_rank_shift_dml_informed_df["heuristic_winner_diagnostic_loser"] = campaign_rank_shift_dml_informed_df["heuristic_winner_diagnostic_loser_exposure"]
campaign_rank_shift_dml_informed_df["heuristic_loser_diagnostic_winner"] = campaign_rank_shift_dml_informed_df["heuristic_loser_diagnostic_winner_exposure"]
campaign_rank_shift_dml_informed_df = campaign_rank_shift_dml_informed_df.sort_values(
    ["diagnostic_rank_proxy", "campaign"],
    ascending=[True, True],
).reset_index(drop=True)
save_table(campaign_rank_shift_dml_informed_df, "table_campaign_rank_shift_dml_informed")

campaign_rank_shift_support_filtered_df = campaign_rank_shift_dml_informed_df.loc[
    campaign_rank_shift_dml_informed_df["support_sufficient"]
].copy().reset_index(drop=True)
if not campaign_rank_shift_support_filtered_df.empty:
    support_total_pos = float(campaign_rank_shift_support_filtered_df["incremental_proxy_pos"].sum())
    campaign_rank_shift_support_filtered_df["diagnostic_share_proxy"] = (
        campaign_rank_shift_support_filtered_df["incremental_proxy_pos"] / support_total_pos
        if support_total_pos > 0
        else np.nan
    )
    exposure_total_support = float(campaign_rank_shift_support_filtered_df["exposure_count"].sum())
    click_total_support = float(campaign_rank_shift_support_filtered_df["click_count"].sum())
    last_touch_total_support = pd.to_numeric(
        campaign_rank_shift_support_filtered_df["last_touch_credit"],
        errors="coerce",
    ).sum(min_count=1)
    campaign_rank_shift_support_filtered_df["exposure_share"] = (
        campaign_rank_shift_support_filtered_df["exposure_count"] / exposure_total_support
        if exposure_total_support > 0
        else np.nan
    )
    campaign_rank_shift_support_filtered_df["click_share"] = (
        campaign_rank_shift_support_filtered_df["click_count"] / click_total_support
        if click_total_support > 0
        else np.nan
    )
    campaign_rank_shift_support_filtered_df["last_touch_share"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["last_touch_credit"], errors="coerce") / float(last_touch_total_support)
        if pd.notna(last_touch_total_support) and float(last_touch_total_support) > 0
        else np.nan
    )
    campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"] = _rank_desc(campaign_rank_shift_support_filtered_df["incremental_proxy"])
    campaign_rank_shift_support_filtered_df["dml_informed_ranking"] = campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"]
    campaign_rank_shift_support_filtered_df["conditional_managerial_ranking"] = campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"]
    campaign_rank_shift_support_filtered_df["exposure_rank"] = _rank_desc(campaign_rank_shift_support_filtered_df["exposure_share"])
    campaign_rank_shift_support_filtered_df["click_rank"] = _rank_desc(campaign_rank_shift_support_filtered_df["click_share"])
    campaign_rank_shift_support_filtered_df["last_touch_rank"] = _rank_desc(campaign_rank_shift_support_filtered_df["last_touch_share"])
    campaign_rank_shift_support_filtered_df["rank_shift_exposure"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["exposure_rank"], errors="coerce")
        - pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce")
    )
    campaign_rank_shift_support_filtered_df["rank_shift_click"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["click_rank"], errors="coerce")
        - pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce")
    )
    campaign_rank_shift_support_filtered_df["rank_shift_last_touch"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["last_touch_rank"], errors="coerce")
        - pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce")
    )
    campaign_rank_shift_support_filtered_df["share_shift_exposure"] = (
        campaign_rank_shift_support_filtered_df["diagnostic_share_proxy"] - campaign_rank_shift_support_filtered_df["exposure_share"]
    )
    campaign_rank_shift_support_filtered_df["share_shift_click"] = (
        campaign_rank_shift_support_filtered_df["diagnostic_share_proxy"] - campaign_rank_shift_support_filtered_df["click_share"]
    )
    campaign_rank_shift_support_filtered_df["share_shift_last_touch"] = (
        campaign_rank_shift_support_filtered_df["diagnostic_share_proxy"] - campaign_rank_shift_support_filtered_df["last_touch_share"]
    )
    campaign_rank_shift_support_filtered_df["heuristic_winner_diagnostic_loser_exposure"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["exposure_rank"], errors="coerce") <= 10
    ) & (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce") > 20
    )
    campaign_rank_shift_support_filtered_df["heuristic_loser_diagnostic_winner_exposure"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["exposure_rank"], errors="coerce") > 20
    ) & (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce") <= 10
    )
    campaign_rank_shift_support_filtered_df["heuristic_winner_diagnostic_loser_click"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["click_rank"], errors="coerce") <= 10
    ) & (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce") > 20
    )
    campaign_rank_shift_support_filtered_df["heuristic_loser_diagnostic_winner_click"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["click_rank"], errors="coerce") > 20
    ) & (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce") <= 10
    )
    campaign_rank_shift_support_filtered_df["heuristic_winner_diagnostic_loser_last_touch"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["last_touch_rank"], errors="coerce") <= 10
    ) & (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce") > 20
    )
    campaign_rank_shift_support_filtered_df["heuristic_loser_diagnostic_winner_last_touch"] = (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["last_touch_rank"], errors="coerce") > 20
    ) & (
        pd.to_numeric(campaign_rank_shift_support_filtered_df["diagnostic_rank_proxy"], errors="coerce") <= 10
    )
    campaign_rank_shift_support_filtered_df = campaign_rank_shift_support_filtered_df.sort_values(
        ["diagnostic_rank_proxy", "campaign"],
        ascending=[True, True],
    ).reset_index(drop=True)

save_table(campaign_rank_shift_support_filtered_df, "table_campaign_rank_shift_support_filtered")

summary_campaign_rank_shift_dml_informed = {
    "title": "Support-aware DML-informed campaign ranking",
    "stepd_payload_key": dml_analysis_metadata.get("stepd_payload_key") if "dml_analysis_metadata" in globals() else None,
    "restriction_name": dml_analysis_metadata.get("restriction_name") if "dml_analysis_metadata" in globals() else None,
    "outcome_hours": dml_analysis_metadata.get("outcome_hours") if "dml_analysis_metadata" in globals() else None,
    "trim_eps": dml_analysis_metadata.get("trim_eps") if "dml_analysis_metadata" in globals() else None,
    "ranking_is_conditional_on_observed_support": True,
    "ranking_is_selection_adjusted": True,
    "ranking_is_managerial_prioritization_signal": True,
    "ranking_is_not_universal_diagnostic_league_table": True,
    "screening_interpretation_text": "This ranking is conditional on observed support and is intended for screening/prioritization, not universal structural comparison.",
    "support_thresholds": {
        "min_obs": int(CAMPAIGN_MIN_OBS),
        "min_users": int(CAMPAIGN_MIN_USERS),
        "min_clicks": int(CAMPAIGN_MIN_CLICKS),
    },
    "n_campaigns_all": int(len(campaign_rank_shift_dml_informed_df)),
    "n_campaigns_support_sufficient": int(len(campaign_rank_shift_support_filtered_df)),
    "top10_overlap_exposure": _top_overlap(campaign_rank_shift_support_filtered_df, "diagnostic_rank_proxy", "exposure_rank", 10),
    "top10_overlap_click": _top_overlap(campaign_rank_shift_support_filtered_df, "diagnostic_rank_proxy", "click_rank", 10),
    "top10_overlap_last_touch": _top_overlap(campaign_rank_shift_support_filtered_df, "diagnostic_rank_proxy", "last_touch_rank", 10),
    "top20_overlap_exposure": _top_overlap(campaign_rank_shift_support_filtered_df, "diagnostic_rank_proxy", "exposure_rank", 20),
    "top20_overlap_click": _top_overlap(campaign_rank_shift_support_filtered_df, "diagnostic_rank_proxy", "click_rank", 20),
    "top20_overlap_last_touch": _top_overlap(campaign_rank_shift_support_filtered_df, "diagnostic_rank_proxy", "last_touch_rank", 20),
    "mean_abs_rank_shift_exposure": float(campaign_rank_shift_support_filtered_df["rank_shift_exposure"].abs().mean()) if not campaign_rank_shift_support_filtered_df.empty else None,
    "mean_abs_rank_shift_click": float(campaign_rank_shift_support_filtered_df["rank_shift_click"].abs().mean()) if not campaign_rank_shift_support_filtered_df.empty else None,
    "mean_abs_rank_shift_last_touch": float(campaign_rank_shift_support_filtered_df["rank_shift_last_touch"].abs().mean()) if not campaign_rank_shift_support_filtered_df.empty else None,
    "n_heuristic_winners_but_diagnostic_losers_exposure": int(campaign_rank_shift_support_filtered_df["heuristic_winner_diagnostic_loser_exposure"].sum()) if "heuristic_winner_diagnostic_loser_exposure" in campaign_rank_shift_support_filtered_df.columns else 0,
    "n_heuristic_losers_but_diagnostic_winners_exposure": int(campaign_rank_shift_support_filtered_df["heuristic_loser_diagnostic_winner_exposure"].sum()) if "heuristic_loser_diagnostic_winner_exposure" in campaign_rank_shift_support_filtered_df.columns else 0,
    "n_heuristic_winners_but_diagnostic_losers_click": int(campaign_rank_shift_support_filtered_df["heuristic_winner_diagnostic_loser_click"].sum()) if "heuristic_winner_diagnostic_loser_click" in campaign_rank_shift_support_filtered_df.columns else 0,
    "n_heuristic_losers_but_diagnostic_winners_click": int(campaign_rank_shift_support_filtered_df["heuristic_loser_diagnostic_winner_click"].sum()) if "heuristic_loser_diagnostic_winner_click" in campaign_rank_shift_support_filtered_df.columns else 0,
    "n_heuristic_winners_but_diagnostic_losers_last_touch": int(campaign_rank_shift_support_filtered_df["heuristic_winner_diagnostic_loser_last_touch"].sum()) if "heuristic_winner_diagnostic_loser_last_touch" in campaign_rank_shift_support_filtered_df.columns else 0,
    "n_heuristic_losers_but_diagnostic_winners_last_touch": int(campaign_rank_shift_support_filtered_df["heuristic_loser_diagnostic_winner_last_touch"].sum()) if "heuristic_loser_diagnostic_winner_last_touch" in campaign_rank_shift_support_filtered_df.columns else 0,
    "corr_exposure_vs_diagnostic_rank": _safe_corr(campaign_rank_shift_support_filtered_df.get("exposure_rank"), campaign_rank_shift_support_filtered_df.get("diagnostic_rank_proxy")),
    "corr_click_vs_diagnostic_rank": _safe_corr(campaign_rank_shift_support_filtered_df.get("click_rank"), campaign_rank_shift_support_filtered_df.get("diagnostic_rank_proxy")),
    "corr_last_touch_vs_diagnostic_rank": _safe_corr(campaign_rank_shift_support_filtered_df.get("last_touch_rank"), campaign_rank_shift_support_filtered_df.get("diagnostic_rank_proxy")),
    "top10_diagnostic_campaigns_support_filtered": campaign_rank_shift_support_filtered_df.sort_values("diagnostic_rank_proxy").head(10)["campaign"].astype(str).tolist(),
    "top20_diagnostic_campaigns_support_filtered": campaign_rank_shift_support_filtered_df.sort_values("diagnostic_rank_proxy").head(20)["campaign"].astype(str).tolist(),
    "writer_note": "Campaign prioritization is constructed from DML-informed residualized signal rather than raw click prominence, and rank shifts relative to heuristic attribution are substantial even after restricting attention to support-sufficient campaigns.",
}
save_named_json(dml_to_builtin(summary_campaign_rank_shift_dml_informed), "summary_campaign_rank_shift_dml_informed")

summary_campaign_ranking_interpretation = {
    "title": "Interpretation of support-aware DML-informed campaign ranking",
    "ranking_label": "dml_informed_ranking",
    "signal_label": "selection_adjusted_campaign_signal",
    "managerial_label": "conditional_managerial_ranking",
    "stepd_payload_key": dml_analysis_metadata.get("stepd_payload_key") if "dml_analysis_metadata" in globals() else None,
    "restriction_name": dml_analysis_metadata.get("restriction_name") if "dml_analysis_metadata" in globals() else None,
    "outcome_hours": dml_analysis_metadata.get("outcome_hours") if "dml_analysis_metadata" in globals() else None,
    "trim_eps": dml_analysis_metadata.get("trim_eps") if "dml_analysis_metadata" in globals() else None,
    "ranking_is_conditional_on_observed_support": True,
    "ranking_is_selection_adjusted": True,
    "ranking_is_universal_diagnostic_league_table": False,
    "key_principles": [
        "Different campaigns reach different observed users, so the ranking is not a universal diagnostic league table.",
        "Campaigns are compared within observed support using residualized Step D objects mhat, ghat, D_tilde, and Y_tilde.",
        "The output is intended for screening and prioritization among support-sufficient campaigns, not for universal structural comparison.",
        "ranking is conditional on observed support",
        "ranking is intended for screening/prioritization, not universal structural comparison",
    ],
    "recommended_use": "Use the ranking as a managerial prioritization signal when deciding which campaigns deserve deeper review, experimentation, or budget attention within observed support.",
    "not_recommended_use": "Do not interpret the ranking as universal structural truth across campaigns that serve different user populations.",
}
save_named_json(dml_to_builtin(summary_campaign_ranking_interpretation), "summary_campaign_ranking_interpretation")

# TODO: Appendix robustness extension can refit campaign-specific nuisance models for the top support-sufficient campaigns and compare local residualization against the global Step D residualization used here.

display(campaign_support_diagnostics_df.head(20))
display(campaign_dml_informed_ranking_df.head(20))
display(campaign_rank_shift_dml_informed_df.head(20))
display(campaign_rank_shift_support_filtered_df.head(20))


## 10. Campaign Reprioritization


In [ ]:
# ====== Step M0: Load DSS Inputs From Existing Exports ======
from pathlib import Path
import json
import zipfile
import shutil
import numpy as np
import pandas as pd

# This cell lets the DSS appendix run from previously exported Step D4/H2 artifacts.
# It never refits DML. If row-level residualized data are unavailable, later cells
# fall back to campaign-level tables and explicitly mark row-level diagnostics as unavailable.

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path.cwd()
if "PAPER_OUTPUT_DIR" not in globals():
    PAPER_OUTPUT_DIR = PROJECT_DIR / "paper_outputs"
if "PAPER_OUT" not in globals():
    PAPER_OUT = PAPER_TABLE_DIR
PAPER_OUT.mkdir(parents=True, exist_ok=True)
PAPER_TABLE_DIR = PAPER_OUT / "paper_tables"
PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)


def _dss_m0_read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def _dss_m0_find_file(search_roots, names):
    roots = [Path(r) for r in search_roots if r is not None and Path(r).exists()]
    for root in roots:
        for name in names:
            direct_candidates = [root / name, root / "paper_outputs" / name, root / "paper_tables" / name, root / "paper_outputs" / "paper_tables" / name]
            for candidate in direct_candidates:
                if candidate.exists():
                    return candidate
        for name in names:
            matches = list(root.rglob(name))
            if matches:
                matches = sorted(matches, key=lambda p: (len(p.parts), str(p)))
                return matches[0]
    return None


def _dss_m0_load_csv(search_roots, names, required=False):
    path = _dss_m0_find_file(search_roots, names)
    if path is None:
        if required:
            raise FileNotFoundError(f"Could not find any of: {names}")
        return None, None
    df = pd.read_csv(path)
    if "campaign" in df.columns:
        df["campaign"] = df["campaign"].astype(str)
    return df, path


def _dss_m0_load_json(search_roots, names):
    path = _dss_m0_find_file(search_roots, names)
    if path is None:
        return None, None
    return _dss_m0_read_json(path), path


def _dss_m0_load_row_level(search_roots):
    parquet_names = ["dml_analysis_df.parquet", "dss_dml_analysis_df.parquet", "step_d4_dml_analysis_df.parquet", "dml_analysis_frame.parquet"]
    csv_names = ["dml_analysis_df.csv", "dss_dml_analysis_df.csv", "step_d4_dml_analysis_df.csv", "dml_analysis_frame.csv", "table_dml_analysis_frame.csv"]
    path = _dss_m0_find_file(search_roots, parquet_names)
    if path is not None:
        try:
            return pd.read_parquet(path), path, "parquet"
        except Exception as exc:
            print("M0 found a row-level parquet file but could not read it:", path, exc)
    path = _dss_m0_find_file(search_roots, csv_names)
    if path is not None:
        return pd.read_csv(path), path, "csv"
    return None, None, None


def _dss_m0_zip_candidates():
    candidates = []
    if "DSS_INPUT_ZIP" in globals():
        candidates.append(Path(DSS_INPUT_ZIP))
    candidates.extend([
                PROJECT_DIR / "paper_outputs.zip",
        PROJECT_DIR / "paper_outputs-20260529T132954Z-3-001.zip",
    ])
    return [p for p in candidates if p.exists()]


def _dss_m0_dir_candidates():
    candidates = []
    if "DSS_INPUT_DIR" in globals():
        candidates.append(Path(DSS_INPUT_DIR))
    candidates.extend([
        PAPER_OUT,
        PAPER_TABLE_DIR,
        PROJECT_DIR / "paper_outputs",
                    ])
    return [p for p in candidates if p.exists()]

DSS_M0_EXTRACT_DIR = None
DSS_M0_INPUT_SOURCE = None
search_roots = []
zip_candidates = _dss_m0_zip_candidates()
if zip_candidates:
    zip_path = zip_candidates[0]
    DSS_M0_INPUT_SOURCE = str(zip_path)
    DSS_M0_EXTRACT_DIR = PROJECT_DIR / "dss_recovered_inputs" / zip_path.stem
    DSS_M0_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    marker = DSS_M0_EXTRACT_DIR / ".extracted_from_zip"
    if not marker.exists() or marker.read_text().strip() != str(zip_path):
        if DSS_M0_EXTRACT_DIR.exists():
            shutil.rmtree(DSS_M0_EXTRACT_DIR)
        DSS_M0_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DSS_M0_EXTRACT_DIR)
        marker.write_text(str(zip_path))
    search_roots.append(DSS_M0_EXTRACT_DIR)

search_roots.extend(_dss_m0_dir_candidates())
search_roots = [p for i, p in enumerate(search_roots) if p.exists() and p not in search_roots[:i]]

campaign_rank_shift_support_filtered_df, _m0_rank_support_path = _dss_m0_load_csv(search_roots, ["table_campaign_rank_shift_support_filtered.csv"])
campaign_dml_informed_ranking_df, _m0_rank_path = _dss_m0_load_csv(search_roots, ["table_campaign_dml_informed_ranking.csv"])
campaign_support_diagnostics_df, _m0_support_path = _dss_m0_load_csv(search_roots, ["table_campaign_support_diagnostics.csv"])
campaign_rank_shift_dml_informed_df, _m0_rank_shift_path = _dss_m0_load_csv(search_roots, ["table_campaign_rank_shift_dml_informed.csv"])

summary_dml_analysis_frame, _m0_d4_summary_path = _dss_m0_load_json(search_roots, ["summary_dml_analysis_frame.json"])
summary_campaign_rank_shift_dml_informed, _m0_rank_summary_path = _dss_m0_load_json(search_roots, ["summary_campaign_rank_shift_dml_informed.json"])
summary_campaign_ranking_interpretation, _m0_interp_path = _dss_m0_load_json(search_roots, ["summary_campaign_ranking_interpretation.json"])
stepD_final_spec, _m0_final_spec_path = _dss_m0_load_json(search_roots, ["stepD_final_spec.json"])

_dss_row_df, _m0_row_path, _m0_row_format = _dss_m0_load_row_level(search_roots)
DSS_ROW_LEVEL_AVAILABLE = False
if isinstance(_dss_row_df, pd.DataFrame) and not _dss_row_df.empty:
    required_row_cols = {"campaign", "uid", "D", "Y", "mhat", "ghat", "D_tilde", "Y_tilde"}
    missing_row_cols = sorted(required_row_cols - set(_dss_row_df.columns))
    if missing_row_cols:
        print("M0 found row-level data but it is missing required columns:", missing_row_cols)
        dss_analysis_df_recovered = _dss_row_df
    else:
        dml_analysis_df = _dss_row_df.copy()
        dml_analysis_df["campaign"] = dml_analysis_df["campaign"].astype(str)
        if "score_proxy" not in dml_analysis_df.columns:
            dml_analysis_df["score_proxy"] = pd.to_numeric(dml_analysis_df["D_tilde"], errors="coerce") * pd.to_numeric(dml_analysis_df["Y_tilde"], errors="coerce")
        DSS_ROW_LEVEL_AVAILABLE = True
        print("M0 loaded row-level dml_analysis_df:", _m0_row_path)

campaign_sources = {
    "campaign_rank_shift_support_filtered_df": str(_m0_rank_support_path) if _m0_rank_support_path else None,
    "campaign_dml_informed_ranking_df": str(_m0_rank_path) if _m0_rank_path else None,
    "campaign_support_diagnostics_df": str(_m0_support_path) if _m0_support_path else None,
    "campaign_rank_shift_dml_informed_df": str(_m0_rank_shift_path) if _m0_rank_shift_path else None,
}
loaded_campaign_tables = {k: v for k, v in campaign_sources.items() if v is not None}
DSS_CAMPAIGN_LEVEL_AVAILABLE = bool(loaded_campaign_tables)
DSS_CAMPAIGN_LEVEL_ONLY = DSS_CAMPAIGN_LEVEL_AVAILABLE and not DSS_ROW_LEVEL_AVAILABLE

DSS_M0_STATUS = {
    "input_source": DSS_M0_INPUT_SOURCE,
    "extract_dir": str(DSS_M0_EXTRACT_DIR) if DSS_M0_EXTRACT_DIR else None,
    "search_roots": [str(p) for p in search_roots],
    "row_level_available": bool(DSS_ROW_LEVEL_AVAILABLE),
    "row_level_path": str(_m0_row_path) if _m0_row_path else None,
    "row_level_format": _m0_row_format,
    "campaign_level_available": bool(DSS_CAMPAIGN_LEVEL_AVAILABLE),
    "campaign_level_only": bool(DSS_CAMPAIGN_LEVEL_ONLY),
    "loaded_campaign_tables": loaded_campaign_tables,
    "summary_dml_analysis_frame_path": str(_m0_d4_summary_path) if _m0_d4_summary_path else None,
    "note": "No DML is refit in M0. Row-level DSS diagnostics require a saved dml_analysis_df file; otherwise later cells use campaign-level exports and mark row-level simulations unavailable.",
}

print("Step M0 DSS input recovery status:")
print(json.dumps(DSS_M0_STATUS, indent=2))
if DSS_CAMPAIGN_LEVEL_ONLY:
    print("M0 warning: this export bundle does not contain row-level dml_analysis_df. M1/M3 can use recovered campaign tables; M2/M4/M5 will use available campaign-level diagnostics or skip row-level-only analyses with status flags.")


## 10. Campaign Reprioritization


In [ ]:
# ====== Step M1: DSS Reprioritization Metrics ======
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAPER_TABLE_DIR = (PAPER_TABLE_DIR if "PAPER_TABLE_DIR" in globals() else PAPER_OUTPUT_DIR / "tables")
PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
DSS_FIG_DIR = (PAPER_FIGURE_DIR if "PAPER_FIGURE_DIR" in globals() else PAPER_OUTPUT_DIR / "figures")
DSS_FIG_DIR.mkdir(parents=True, exist_ok=True)


def dss_to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): dss_to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [dss_to_builtin(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.tolist()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def dss_export_csv(df, filename):
    path = PAPER_TABLE_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


def dss_export_json(obj, filename):
    path = PAPER_TABLE_DIR / filename
    with open(path, "w") as f:
        json.dump(dss_to_builtin(obj), f, indent=2)
    print("Saved:", path)
    return path


def dss_save_figure(fig, stem):
    png_path = DSS_FIG_DIR / f"{stem}.png"
    pdf_path = DSS_FIG_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Saved:", png_path)
    print("Saved:", pdf_path)
    return {"png": str(png_path), "pdf": str(pdf_path)}


def dss_placeholder_figure(stem, message):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", wrap=True)
    ax.set_title(stem)
    dss_save_figure(fig, stem)
    plt.close(fig)


def dss_safe_corr(a, b, method="spearman"):
    frame = pd.DataFrame({"a": pd.to_numeric(a, errors="coerce"), "b": pd.to_numeric(b, errors="coerce")}).dropna()
    if len(frame) < 2:
        return np.nan
    try:
        return float(frame["a"].corr(frame["b"], method=method))
    except Exception:
        return np.nan


def dss_rank_desc(values):
    return pd.to_numeric(values, errors="coerce").rank(ascending=False, method="min")


def dss_semicolon(values, limit=None):
    vals = [str(v) for v in values if pd.notna(v)]
    if limit is not None:
        vals = vals[:limit]
    return ";".join(vals)


def dss_prepare_rank_frame():
    if "campaign_rank_shift_support_filtered_df" in globals() and isinstance(campaign_rank_shift_support_filtered_df, pd.DataFrame):
        source = campaign_rank_shift_support_filtered_df.copy()
        source_name = "campaign_rank_shift_support_filtered_df"
    elif "campaign_dml_informed_ranking_df" in globals() and isinstance(campaign_dml_informed_ranking_df, pd.DataFrame):
        source = campaign_dml_informed_ranking_df.copy()
        source_name = "campaign_dml_informed_ranking_df"
    else:
        source = pd.DataFrame()
        source_name = "not_available"

    if source.empty or "campaign" not in source.columns:
        return pd.DataFrame(columns=["campaign", "diagnostic_rank", "diagnostic_score", "status"]), source_name

    out = source.copy()
    out["campaign"] = out["campaign"].astype(str)
    rank_candidates = ["diagnostic_rank_proxy", "dml_informed_ranking", "conditional_managerial_ranking", "diagnostic_rank"]
    score_candidates = ["selection_adjusted_campaign_signal", "incremental_proxy", "diagnostic_score", "score_sum", "diagnostic_score_sum"]
    rank_col = next((c for c in rank_candidates if c in out.columns), None)
    score_col = next((c for c in score_candidates if c in out.columns), None)
    out["diagnostic_rank"] = pd.to_numeric(out[rank_col], errors="coerce") if rank_col else np.nan
    out["diagnostic_score"] = pd.to_numeric(out[score_col], errors="coerce") if score_col else np.nan
    if out["diagnostic_rank"].isna().all() and out["diagnostic_score"].notna().any():
        out["diagnostic_rank"] = dss_rank_desc(out["diagnostic_score"])
    out["status"] = "ok" if out["diagnostic_rank"].notna().any() else "missing_diagnostic_rank"
    return out, source_name


def dss_topk_sets(frame, diagnostic_rank_col, heuristic_rank_col, k):
    use = frame[["campaign", diagnostic_rank_col, heuristic_rank_col]].copy()
    use[diagnostic_rank_col] = pd.to_numeric(use[diagnostic_rank_col], errors="coerce")
    use[heuristic_rank_col] = pd.to_numeric(use[heuristic_rank_col], errors="coerce")
    use = use.dropna()
    if use.empty:
        return set(), set(), 0
    effective_k = int(min(k, len(use)))
    diag = set(use.loc[use[diagnostic_rank_col] <= effective_k, "campaign"].astype(str))
    heur = set(use.loc[use[heuristic_rank_col] <= effective_k, "campaign"].astype(str))
    return diag, heur, effective_k


dss_rank_frame, dss_rank_source = dss_prepare_rank_frame()
DSS_RANK_COMPARISONS = [
    ("Diagnostic vs exposure", "exposure_rank", "fig_dss_rank_scatter_diagnostic_vs_exposure"),
    ("Diagnostic vs click", "click_rank", "fig_dss_rank_scatter_diagnostic_vs_click"),
    ("Diagnostic vs last-touch", "last_touch_rank", "fig_dss_rank_scatter_diagnostic_vs_last_touch"),
]

metric_rows = []
detail_rows = []
for comparison, heuristic_rank_col, fig_stem in DSS_RANK_COMPARISONS:
    if dss_rank_frame.empty or heuristic_rank_col not in dss_rank_frame.columns or "diagnostic_rank" not in dss_rank_frame.columns:
        metric_rows.append({
            "comparison": comparison,
            "n_campaigns": 0,
            "spearman_rank_corr": np.nan,
            "kendall_rank_corr": np.nan,
            "mean_abs_rank_shift": np.nan,
            "median_abs_rank_shift": np.nan,
            "p90_abs_rank_shift": np.nan,
            "top5_overlap_count": np.nan,
            "top5_overlap_share": np.nan,
            "top10_overlap_count": np.nan,
            "top10_overlap_share": np.nan,
            "top20_overlap_count": np.nan,
            "top20_overlap_share": np.nan,
            "top50_overlap_count": np.nan,
            "top50_overlap_share": np.nan,
            "status": f"missing_{heuristic_rank_col}",
            "source": dss_rank_source,
        })
        for k in [5, 10, 20, 50]:
            detail_rows.append({
                "comparison": comparison,
                "k": k,
                "overlap_count": np.nan,
                "overlap_share": np.nan,
                "diagnostic_only_campaigns": "",
                "heuristic_only_campaigns": "",
                "status": f"missing_{heuristic_rank_col}",
            })
        dss_placeholder_figure(fig_stem, f"{comparison}: {heuristic_rank_col} is not available.")
        continue

    use = dss_rank_frame[["campaign", "diagnostic_rank", heuristic_rank_col]].copy()
    use["diagnostic_rank"] = pd.to_numeric(use["diagnostic_rank"], errors="coerce")
    use[heuristic_rank_col] = pd.to_numeric(use[heuristic_rank_col], errors="coerce")
    use = use.dropna().reset_index(drop=True)
    status = "ok" if len(use) >= 2 else "insufficient_valid_ranks"
    abs_shift = (use[heuristic_rank_col] - use["diagnostic_rank"]).abs() if not use.empty else pd.Series(dtype=float)

    row = {
        "comparison": comparison,
        "n_campaigns": int(len(use)),
        "spearman_rank_corr": dss_safe_corr(use["diagnostic_rank"], use[heuristic_rank_col], "spearman"),
        "kendall_rank_corr": dss_safe_corr(use["diagnostic_rank"], use[heuristic_rank_col], "kendall"),
        "mean_abs_rank_shift": float(abs_shift.mean()) if len(abs_shift) else np.nan,
        "median_abs_rank_shift": float(abs_shift.median()) if len(abs_shift) else np.nan,
        "p90_abs_rank_shift": float(abs_shift.quantile(0.90)) if len(abs_shift) else np.nan,
        "status": status,
        "source": dss_rank_source,
    }
    for k in [5, 10, 20, 50]:
        diag_set, heur_set, effective_k = dss_topk_sets(use, "diagnostic_rank", heuristic_rank_col, k)
        denom = effective_k if effective_k > 0 else np.nan
        overlap = diag_set & heur_set
        row[f"top{k}_overlap_count"] = int(len(overlap)) if effective_k else np.nan
        row[f"top{k}_overlap_share"] = float(len(overlap) / denom) if effective_k else np.nan
        detail_rows.append({
            "comparison": comparison,
            "k": int(k),
            "overlap_count": int(len(overlap)) if effective_k else np.nan,
            "overlap_share": float(len(overlap) / denom) if effective_k else np.nan,
            "diagnostic_only_campaigns": dss_semicolon(sorted(diag_set - heur_set)),
            "heuristic_only_campaigns": dss_semicolon(sorted(heur_set - diag_set)),
            "status": status,
        })
    metric_rows.append(row)

    if use.empty:
        dss_placeholder_figure(fig_stem, f"{comparison}: no valid rank pairs.")
    else:
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        ax.scatter(use[heuristic_rank_col], use["diagnostic_rank"], alpha=0.65, s=28)
        max_rank = float(np.nanmax([use[heuristic_rank_col].max(), use["diagnostic_rank"].max()]))
        ax.plot([1, max_rank], [1, max_rank], linestyle="--", color="black", linewidth=1)
        ax.set_xlabel(heuristic_rank_col.replace("_", " "))
        ax.set_ylabel("diagnostic rank")
        ax.set_title(comparison)
        ax.invert_xaxis()
        ax.invert_yaxis()
        label_df = use.assign(abs_shift=(use[heuristic_rank_col] - use["diagnostic_rank"]).abs()).nlargest(5, "abs_shift")
        for _, r in label_df.iterrows():
            ax.annotate(str(r["campaign"]), (r[heuristic_rank_col], r["diagnostic_rank"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
        dss_save_figure(fig, fig_stem)
        plt.close(fig)

table_dss_reprioritization_metrics = pd.DataFrame(metric_rows)
table_dss_topk_overlap_details = pd.DataFrame(detail_rows)
dss_export_csv(table_dss_reprioritization_metrics, "table_dss_reprioritization_metrics.csv")
dss_export_csv(table_dss_topk_overlap_details, "table_dss_topk_overlap_details.csv")
display(table_dss_reprioritization_metrics)


## 9. RAMP Layer 4: Diagnostic Gate


In [ ]:
# ====== Step M2: Diagnostic Gate Funnel ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DSS_SUPPORT_MIN_OBS = 500
DSS_SUPPORT_MIN_USERS = 100
DSS_SUPPORT_MIN_CLICKS = 20
DSS_ROW_LEVEL_AVAILABLE = "dml_analysis_df" in globals() and isinstance(dml_analysis_df, pd.DataFrame) and not dml_analysis_df.empty
DSS_CAMPAIGN_LEVEL_AVAILABLE = any(
    name in globals() and isinstance(globals()[name], pd.DataFrame) and not globals()[name].empty
    for name in ["campaign_rank_shift_support_filtered_df", "campaign_dml_informed_ranking_df", "campaign_support_diagnostics_df"]
)


def dss_row_frame_with_score():
    if not DSS_ROW_LEVEL_AVAILABLE:
        raise ValueError("dml_analysis_df is not available in memory.")
    row = dml_analysis_df.copy().reset_index(drop=True)
    if "campaign" not in row.columns:
        raise KeyError("dml_analysis_df must contain campaign for DSS diagnostics.")
    row["campaign"] = row["campaign"].astype(str)
    if "uid" not in row.columns:
        row["uid"] = np.arange(len(row)).astype(str)
    for col in ["D", "Y", "mhat", "ghat", "D_tilde", "Y_tilde"]:
        if col in row.columns:
            row[col] = pd.to_numeric(row[col], errors="coerce")
    if "score_proxy" not in row.columns:
        if {"D_tilde", "Y_tilde"}.issubset(row.columns):
            row["score_proxy"] = row["D_tilde"] * row["Y_tilde"]
        else:
            row["score_proxy"] = np.nan
    row["D_num"] = pd.to_numeric(row.get("D", np.nan), errors="coerce")
    row["Y_num"] = pd.to_numeric(row.get("Y", np.nan), errors="coerce")
    row["mhat_num"] = pd.to_numeric(row.get("mhat", np.nan), errors="coerce")
    row["score_proxy"] = pd.to_numeric(row["score_proxy"], errors="coerce")
    return row


def dss_group_placebo_scores(row):
    base = row[["campaign", "D_num"]].copy()
    status = "not_available"
    score = pd.Series(np.nan, index=row.index, dtype=float)
    y_placebo = None
    if "placebo_score_proxy" in row.columns:
        score = pd.to_numeric(row["placebo_score_proxy"], errors="coerce")
        status = "existing_placebo_score_proxy"
    elif "Y_placebo_tilde" in row.columns and "D_tilde" in row.columns:
        score = pd.to_numeric(row["D_tilde"], errors="coerce") * pd.to_numeric(row["Y_placebo_tilde"], errors="coerce")
        status = "existing_placebo_residual_score"
    elif "Y_placebo" in row.columns:
        y_placebo = pd.to_numeric(row["Y_placebo"], errors="coerce")
        status = "existing_placebo_group_contrast"
    elif "Y_placebo_same_sample" in row.columns:
        y_placebo = pd.to_numeric(row["Y_placebo_same_sample"], errors="coerce")
        status = "existing_placebo_same_sample_group_contrast"
    elif "build_placebo_outcome" in globals() and "ts_sec" in row.columns:
        try:
            horizon_hours = float(globals().get("dml_analysis_metadata", {}).get("outcome_hours", 24.0)) if "dml_analysis_metadata" in globals() else 24.0
            y_placebo = pd.Series(build_placebo_outcome(row, horizon_hours=horizon_hours), index=row.index).astype(float)
            status = "constructed_placebo_group_contrast"
        except Exception as exc:
            status = f"placebo_construction_failed: {exc}"
    if y_placebo is not None:
        tmp = base.assign(Y_placebo=y_placebo)
        clicked = tmp.loc[tmp["D_num"] == 1].groupby("campaign")["Y_placebo"].mean()
        unclicked = tmp.loc[tmp["D_num"] == 0].groupby("campaign")["Y_placebo"].mean()
        out = (clicked - unclicked).rename("placebo_score_mean").reset_index()
        out["placebo_status"] = status
        return out, status
    tmp = base.assign(placebo_score=score)
    out = tmp.groupby("campaign", as_index=False).agg(placebo_score_mean=("placebo_score", "mean"))
    out["placebo_status"] = status if out["placebo_score_mean"].notna().any() else "not_available"
    return out, status


def dss_build_gate_funnel(gate_df):
    total = int(len(gate_df))
    if total == 0:
        return pd.DataFrame(columns=["gate_step", "n_campaigns_remaining", "share_of_all", "incremental_removed", "criteria"])
    support = gate_df["gate_support"].fillna(False).astype(bool) if "gate_support" in gate_df.columns else pd.Series(False, index=gate_df.index)
    overlap_available = "gate_overlap" in gate_df.columns and gate_df["gate_overlap"].notna().any()
    overlap = gate_df["gate_overlap"].fillna(False).astype(bool) if overlap_available else pd.Series(False, index=gate_df.index)
    placebo_available = "gate_placebo" in gate_df.columns and gate_df["gate_placebo"].notna().any()
    stability_available = "gate_stability" in gate_df.columns and gate_df["gate_stability"].notna().any()
    placebo = gate_df["gate_placebo"].fillna(False).astype(bool) if placebo_available else pd.Series(False, index=gate_df.index)
    stability = gate_df["gate_stability"].fillna(False).astype(bool) if stability_available else pd.Series(False, index=gate_df.index)
    masks = [("All campaigns", pd.Series(True, index=gate_df.index), "campaign appears in DSS input tables")]
    masks.append(("Support pass", support, f"n_obs >= {DSS_SUPPORT_MIN_OBS}; n_users >= {DSS_SUPPORT_MIN_USERS}; n_clicks >= {DSS_SUPPORT_MIN_CLICKS}"))
    masks.append(("Overlap pass", support & overlap if overlap_available else None, "p01 >= 0.01; p99 <= 0.99; extreme propensity share <= 0.05" if overlap_available else "overlap quantiles require row-level mhat and are not available from this export bundle"))
    masks.append(("Placebo pass", support & overlap & placebo if placebo_available and overlap_available else None, "placebo diagnostic available" if placebo_available else "placebo diagnostic not available"))
    masks.append(("Stability pass", support & overlap & (placebo if placebo_available else True) & stability if stability_available and overlap_available else None, "ranking stability available" if stability_available else "ranking stability computed in Step M4 when row-level data are available"))
    available_mask = support.copy()
    available_criteria = ["support"]
    if overlap_available:
        available_mask = available_mask & overlap
        available_criteria.append("overlap")
    if placebo_available:
        available_mask = available_mask & placebo
        available_criteria.append("placebo")
    if stability_available:
        available_mask = available_mask & stability
        available_criteria.append("stability")
    full_mask = support & overlap & placebo & stability
    masks.append(("All available gates pass", available_mask, " and ".join(available_criteria)))
    masks.append(("All full gates pass", full_mask if overlap_available else pd.Series(False, index=gate_df.index), "support, overlap, placebo, and stability gates; missing gates treated as fail"))
    rows = []
    prev_count = None
    for step, mask, criteria in masks:
        if mask is None:
            count = np.nan; removed = np.nan; share = np.nan
        else:
            count = int(mask.sum())
            share = float(count / total) if total else np.nan
            removed = int(prev_count - count) if prev_count is not None and pd.notna(prev_count) else 0
            prev_count = count
        rows.append({"gate_step": step, "n_campaigns_remaining": count, "share_of_all": share, "incremental_removed": removed, "criteria": criteria})
    return pd.DataFrame(rows)


def dss_campaign_level_gate_table():
    if "campaign_rank_shift_support_filtered_df" in globals() and isinstance(campaign_rank_shift_support_filtered_df, pd.DataFrame) and not campaign_rank_shift_support_filtered_df.empty:
        src = campaign_rank_shift_support_filtered_df.copy(); status = "campaign_rank_shift_support_filtered_df"
    elif "campaign_dml_informed_ranking_df" in globals() and isinstance(campaign_dml_informed_ranking_df, pd.DataFrame) and not campaign_dml_informed_ranking_df.empty:
        src = campaign_dml_informed_ranking_df.copy(); status = "campaign_dml_informed_ranking_df"
    elif "campaign_support_diagnostics_df" in globals() and isinstance(campaign_support_diagnostics_df, pd.DataFrame) and not campaign_support_diagnostics_df.empty:
        src = campaign_support_diagnostics_df.copy(); status = "campaign_support_diagnostics_df"
    else:
        return pd.DataFrame([{"status": "no_campaign_level_inputs_available"}])
    src["campaign"] = src["campaign"].astype(str)
    out = pd.DataFrame({"campaign": src["campaign"]})
    rename_map = {"n_obs": "n_obs", "n_users": "n_users", "n_clicks": "n_clicks", "score_sum": "diagnostic_score_sum", "score_mean": "diagnostic_score_mean", "incremental_proxy": "diagnostic_score_sum", "selection_adjusted_campaign_signal": "diagnostic_score_sum", "mean_propensity": "mean_propensity", "y_mean": "outcome_mean", "d_mean": "click_rate"}
    for old, new in rename_map.items():
        if old in src.columns and new not in out.columns:
            out[new] = pd.to_numeric(src[old], errors="coerce")
    for col in ["n_obs", "n_users", "n_clicks", "diagnostic_score_sum", "diagnostic_score_mean", "mean_propensity", "outcome_mean", "click_rate"]:
        if col not in out.columns:
            out[col] = np.nan
    if out["diagnostic_score_mean"].isna().all() and out["diagnostic_score_sum"].notna().any() and out["n_obs"].notna().any():
        out["diagnostic_score_mean"] = out["diagnostic_score_sum"] / out["n_obs"].replace(0, np.nan)
    for col in ["p01_propensity", "p05_propensity", "p95_propensity", "p99_propensity", "extreme_propensity_share"]:
        out[col] = np.nan
    out["gate_support"] = (out["n_obs"] >= DSS_SUPPORT_MIN_OBS) & (out["n_users"] >= DSS_SUPPORT_MIN_USERS) & (out["n_clicks"] >= DSS_SUPPORT_MIN_CLICKS)
    if "support_sufficient" in src.columns:
        out["gate_support"] = src["support_sufficient"].fillna(out["gate_support"]).astype(bool)
    out["gate_overlap"] = np.nan
    out["overlap_status"] = "not_available_campaign_level_export"
    out["placebo_score_mean"] = np.nan
    out["placebo_status"] = "not_available_campaign_level_export"
    out["placebo_abs_score"] = np.nan
    out["placebo_gate_threshold"] = np.nan
    out["gate_placebo"] = np.nan
    out["gate_stability"] = np.nan
    out["stability_status"] = "pending_step_m4_requires_row_level"
    out["gate_all_available_only"] = out["gate_support"].fillna(False).astype(bool)
    out["gate_all_full"] = False
    out["status"] = f"campaign_level_recovered_from_{status}"
    return out

if DSS_ROW_LEVEL_AVAILABLE:
    dss_dml_row_df = dss_row_frame_with_score()
    table_dss_diagnostic_gate_campaigns = dss_dml_row_df.groupby("campaign", as_index=False).agg(
        n_obs=("campaign", "size"), n_users=("uid", "nunique"), n_clicks=("D_num", "sum"),
        diagnostic_score_sum=("score_proxy", "sum"), diagnostic_score_mean=("score_proxy", "mean"), mean_propensity=("mhat_num", "mean"),
        p01_propensity=("mhat_num", lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.01))),
        p05_propensity=("mhat_num", lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.05))),
        p95_propensity=("mhat_num", lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.95))),
        p99_propensity=("mhat_num", lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.99))),
        extreme_propensity_share=("mhat_num", lambda s: float(((pd.to_numeric(s, errors="coerce") < 0.01) | (pd.to_numeric(s, errors="coerce") > 0.99)).mean())),
        outcome_mean=("Y_num", "mean"), click_rate=("D_num", "mean"),
    )
    table_dss_diagnostic_gate_campaigns["n_clicks"] = pd.to_numeric(table_dss_diagnostic_gate_campaigns["n_clicks"], errors="coerce").fillna(0).astype(int)
    table_dss_diagnostic_gate_campaigns["gate_support"] = (table_dss_diagnostic_gate_campaigns["n_obs"] >= DSS_SUPPORT_MIN_OBS) & (table_dss_diagnostic_gate_campaigns["n_users"] >= DSS_SUPPORT_MIN_USERS) & (table_dss_diagnostic_gate_campaigns["n_clicks"] >= DSS_SUPPORT_MIN_CLICKS)
    table_dss_diagnostic_gate_campaigns["gate_overlap"] = (table_dss_diagnostic_gate_campaigns["p01_propensity"] >= 0.01) & (table_dss_diagnostic_gate_campaigns["p99_propensity"] <= 0.99) & (table_dss_diagnostic_gate_campaigns["extreme_propensity_share"] <= 0.05)
    table_dss_diagnostic_gate_campaigns["overlap_status"] = "row_level_mhat_quantiles"
    placebo_campaign_df, dss_placebo_status = dss_group_placebo_scores(dss_dml_row_df)
    table_dss_diagnostic_gate_campaigns = table_dss_diagnostic_gate_campaigns.merge(placebo_campaign_df, on="campaign", how="left")
    if "placebo_score_mean" not in table_dss_diagnostic_gate_campaigns.columns:
        table_dss_diagnostic_gate_campaigns["placebo_score_mean"] = np.nan
    if "placebo_status" not in table_dss_diagnostic_gate_campaigns.columns:
        table_dss_diagnostic_gate_campaigns["placebo_status"] = "not_available"
    table_dss_diagnostic_gate_campaigns["placebo_abs_score"] = pd.to_numeric(table_dss_diagnostic_gate_campaigns["placebo_score_mean"], errors="coerce").abs()
    threshold_source = table_dss_diagnostic_gate_campaigns.loc[table_dss_diagnostic_gate_campaigns["gate_support"], "placebo_abs_score"].dropna()
    placebo_gate_threshold = float(threshold_source.quantile(0.75)) if not threshold_source.empty else np.nan
    table_dss_diagnostic_gate_campaigns["placebo_gate_threshold"] = placebo_gate_threshold
    table_dss_diagnostic_gate_campaigns["gate_placebo"] = ((table_dss_diagnostic_gate_campaigns["placebo_abs_score"] <= placebo_gate_threshold) | (pd.to_numeric(table_dss_diagnostic_gate_campaigns["placebo_score_mean"], errors="coerce") <= 0)) if np.isfinite(placebo_gate_threshold) else np.nan
    table_dss_diagnostic_gate_campaigns["gate_stability"] = np.nan
    table_dss_diagnostic_gate_campaigns["stability_status"] = "pending_step_m4"
    table_dss_diagnostic_gate_campaigns["gate_all_available_only"] = table_dss_diagnostic_gate_campaigns["gate_support"] & table_dss_diagnostic_gate_campaigns["gate_overlap"]
    table_dss_diagnostic_gate_campaigns["gate_all_full"] = table_dss_diagnostic_gate_campaigns["gate_support"].fillna(False).astype(bool) & table_dss_diagnostic_gate_campaigns["gate_overlap"].fillna(False).astype(bool) & table_dss_diagnostic_gate_campaigns["gate_placebo"].fillna(False).astype(bool) & table_dss_diagnostic_gate_campaigns["gate_stability"].fillna(False).astype(bool)
    table_dss_diagnostic_gate_campaigns["status"] = "row_level"
else:
    table_dss_diagnostic_gate_campaigns = dss_campaign_level_gate_table()

table_dss_gate_funnel = dss_build_gate_funnel(table_dss_diagnostic_gate_campaigns)
dss_export_csv(table_dss_diagnostic_gate_campaigns, "table_dss_diagnostic_gate_campaigns.csv")
dss_export_csv(table_dss_gate_funnel, "table_dss_gate_funnel.csv")

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = table_dss_gate_funnel.dropna(subset=["n_campaigns_remaining"]).copy()
ax.bar(plot_df["gate_step"], plot_df["n_campaigns_remaining"], color="#4C78A8")
ax.set_ylabel("campaigns remaining")
ax.set_title("DSS diagnostic gate funnel")
ax.tick_params(axis="x", rotation=30)
for i, v in enumerate(plot_df["n_campaigns_remaining"]):
    ax.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=9)
dss_save_figure(fig, "fig_dss_gate_funnel")
plt.close(fig)

display(table_dss_gate_funnel)


## 9. RAMP Layer 4: Diagnostic Gate


In [ ]:
# ====== Step M3: Decision-Risk Classification ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "table_dss_diagnostic_gate_campaigns" not in globals():
    table_dss_decision_risk_classification = pd.DataFrame([{"status": "missing_table_dss_diagnostic_gate_campaigns"}])
    table_dss_decision_risk_summary = pd.DataFrame([{"status": "missing_table_dss_diagnostic_gate_campaigns"}])
else:
    gate_df = table_dss_diagnostic_gate_campaigns.copy()
    rank_df = dss_rank_frame.copy() if "dss_rank_frame" in globals() else pd.DataFrame()
    if not rank_df.empty:
        keep_cols = [c for c in [
            "campaign", "diagnostic_rank", "diagnostic_score", "exposure_share", "exposure_count", "exposure_rank",
            "click_share", "click_count", "click_rank", "last_touch_share", "last_touch_credit", "last_touch_rank",
            "selection_adjusted_campaign_signal", "incremental_proxy"
        ] if c in rank_df.columns]
        work = gate_df.merge(rank_df[keep_cols].drop_duplicates("campaign"), on="campaign", how="left")
    else:
        work = gate_df.copy()

    raw_options = [
        ("last_touch_share", "last_touch_rank", "last_touch"),
        ("last_touch_credit", "last_touch_rank", "last_touch"),
        ("click_share", "click_rank", "click"),
        ("click_count", "click_rank", "click"),
        ("exposure_share", "exposure_rank", "exposure"),
        ("exposure_count", "exposure_rank", "exposure"),
    ]
    raw_score_col = None
    raw_rank_col = None
    raw_source = "not_available"
    for score_col, rank_col, source_name in raw_options:
        if score_col in work.columns and pd.to_numeric(work[score_col], errors="coerce").notna().any():
            raw_score_col = score_col
            raw_rank_col = rank_col if rank_col in work.columns else None
            raw_source = source_name
            break

    work["raw_score"] = pd.to_numeric(work[raw_score_col], errors="coerce") if raw_score_col else np.nan
    if raw_rank_col and pd.to_numeric(work[raw_rank_col], errors="coerce").notna().any():
        work["raw_rank"] = pd.to_numeric(work[raw_rank_col], errors="coerce")
    else:
        work["raw_rank"] = dss_rank_desc(work["raw_score"])

    if "diagnostic_score" not in work.columns or pd.to_numeric(work["diagnostic_score"], errors="coerce").isna().all():
        work["diagnostic_score"] = pd.to_numeric(work.get("diagnostic_score_sum", np.nan), errors="coerce")
    if "diagnostic_rank" not in work.columns or pd.to_numeric(work["diagnostic_rank"], errors="coerce").isna().all():
        work["diagnostic_rank"] = dss_rank_desc(work["diagnostic_score"])

    support = work.loc[work.get("gate_support", False).fillna(False).astype(bool)].copy()
    n_support = int(len(support))
    if n_support == 0:
        table_dss_decision_risk_classification = work.assign(
            raw_source=raw_source,
            raw_percentile=np.nan,
            diagnostic_percentile=np.nan,
            decision_class="not_classified_insufficient_support",
            status="no_support_sufficient_campaigns",
        )
    else:
        denom = max(n_support - 1, 1)
        support["raw_percentile"] = 1.0 - (pd.to_numeric(support["raw_rank"], errors="coerce") - 1.0) / denom
        support["diagnostic_percentile"] = 1.0 - (pd.to_numeric(support["diagnostic_rank"], errors="coerce") - 1.0) / denom
        support["raw_percentile"] = support["raw_percentile"].clip(0, 1)
        support["diagnostic_percentile"] = support["diagnostic_percentile"].clip(0, 1)
        raw_high = support["raw_percentile"] >= 0.75
        diagnostic_high = support["diagnostic_percentile"] >= 0.75
        support["decision_class"] = np.select(
            [raw_high & diagnostic_high, raw_high & ~diagnostic_high, ~raw_high & diagnostic_high],
            ["credible_priority", "likely_over_attributed", "hidden_opportunity"],
            default="deprioritize_or_monitor",
        )
        support["raw_source"] = raw_source
        support["status"] = "ok"
        nonsupport = work.loc[~work.index.isin(support.index)].copy()
        if not nonsupport.empty:
            nonsupport["raw_percentile"] = np.nan
            nonsupport["diagnostic_percentile"] = np.nan
            nonsupport["decision_class"] = "not_classified_support_gate_fail"
            nonsupport["raw_source"] = raw_source
            nonsupport["status"] = "support_gate_fail"
        table_dss_decision_risk_classification = pd.concat([support, nonsupport], ignore_index=True, sort=False)

    wanted_cols = [c for c in [
        "campaign", "raw_score", "raw_rank", "diagnostic_score", "diagnostic_rank", "raw_percentile",
        "diagnostic_percentile", "decision_class", "raw_source", "n_obs", "n_users", "n_clicks",
        "gate_support", "gate_overlap", "p01_propensity", "p99_propensity", "extreme_propensity_share",
        "placebo_score_mean", "placebo_abs_score", "placebo_gate_threshold", "gate_placebo", "placebo_status",
        "gate_stability", "stability_status", "status"
    ] if c in table_dss_decision_risk_classification.columns]
    table_dss_decision_risk_classification = table_dss_decision_risk_classification[wanted_cols].sort_values(
        ["decision_class", "diagnostic_rank", "campaign"], na_position="last"
    ).reset_index(drop=True)

    support_classified = table_dss_decision_risk_classification.loc[table_dss_decision_risk_classification["status"].eq("ok")].copy()
    if support_classified.empty:
        table_dss_decision_risk_summary = pd.DataFrame([{"status": "no_classified_campaigns"}])
    else:
        summary_agg = {
            "campaign": "count",
            "n_obs": "mean",
            "n_users": "mean",
            "n_clicks": "mean",
            "raw_score": "mean",
            "diagnostic_score": "mean",
        }
        if "placebo_score_mean" in support_classified.columns:
            summary_agg["placebo_score_mean"] = "mean"
        table_dss_decision_risk_summary = support_classified.groupby("decision_class", as_index=False).agg(summary_agg)
        table_dss_decision_risk_summary = table_dss_decision_risk_summary.rename(columns={
            "campaign": "n_campaigns",
            "n_obs": "mean_n_obs",
            "n_users": "mean_n_users",
            "n_clicks": "mean_n_clicks",
            "raw_score": "mean_raw_score",
            "diagnostic_score": "mean_diagnostic_score",
            "placebo_score_mean": "mean_placebo_score",
        })
        table_dss_decision_risk_summary["share_campaigns"] = table_dss_decision_risk_summary["n_campaigns"] / float(len(support_classified))
        ordered_cols = ["decision_class", "n_campaigns", "share_campaigns", "mean_n_obs", "mean_n_users", "mean_n_clicks", "mean_raw_score", "mean_diagnostic_score"]
        if "mean_placebo_score" in table_dss_decision_risk_summary.columns:
            ordered_cols.append("mean_placebo_score")
        table_dss_decision_risk_summary = table_dss_decision_risk_summary[ordered_cols]

dss_export_csv(table_dss_decision_risk_classification, "table_dss_decision_risk_classification.csv")
dss_export_csv(table_dss_decision_risk_summary, "table_dss_decision_risk_summary.csv")

plot_df = table_dss_decision_risk_classification.loc[table_dss_decision_risk_classification.get("status", "").eq("ok")].copy() if "status" in table_dss_decision_risk_classification.columns else pd.DataFrame()
if plot_df.empty:
    dss_placeholder_figure("fig_dss_decision_risk_matrix", "Decision-risk matrix unavailable: no support-sufficient classified campaigns.")
else:
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    classes = sorted(plot_df["decision_class"].dropna().unique())
    colors = dict(zip(classes, plt.cm.tab10(np.linspace(0, 1, max(len(classes), 1)))))
    for cls, g in plot_df.groupby("decision_class"):
        ax.scatter(g["raw_percentile"], g["diagnostic_percentile"], label=cls, alpha=0.75, s=32, color=colors.get(cls))
    ax.axvline(0.75, color="black", linestyle="--", linewidth=1)
    ax.axhline(0.75, color="black", linestyle="--", linewidth=1)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("raw heuristic percentile")
    ax.set_ylabel("diagnostic percentile")
    ax.set_title("DSS decision-risk matrix")
    ax.text(0.86, 0.92, "credible priority", fontsize=9, ha="center")
    ax.text(0.86, 0.35, "likely over-attributed", fontsize=9, ha="center")
    ax.text(0.35, 0.92, "hidden opportunity", fontsize=9, ha="center")
    ax.text(0.35, 0.35, "monitor", fontsize=9, ha="center")
    ax.legend(loc="best", fontsize=8)
    dss_save_figure(fig, "fig_dss_decision_risk_matrix")
    plt.close(fig)

display(table_dss_decision_risk_summary)


## 10. Campaign Reprioritization


In [ ]:
# ====== Step M4: Ranking Stability ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DSS_ROW_LEVEL_AVAILABLE = "dml_analysis_df" in globals() and isinstance(dml_analysis_df, pd.DataFrame) and not dml_analysis_df.empty
if not DSS_ROW_LEVEL_AVAILABLE:
    table_dss_ranking_stability_summary = pd.DataFrame([
        {"comparison": "split-half", "n_campaigns": np.nan, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "skipped_missing_row_level_dml_analysis_df"},
        {"comparison": "bootstrap", "n_campaigns": np.nan, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "skipped_missing_row_level_dml_analysis_df"},
        {"comparison": "horizon 1h vs 24h", "n_campaigns": np.nan, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "skipped_missing_stepd_payloads_or_row_level"},
        {"comparison": "horizon 6h vs 24h", "n_campaigns": np.nan, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "skipped_missing_stepd_payloads_or_row_level"},
        {"comparison": "horizon 168h vs 24h", "n_campaigns": np.nan, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "skipped_missing_stepd_payloads_or_row_level"},
    ])
    base = dss_rank_frame.copy() if "dss_rank_frame" in globals() and isinstance(dss_rank_frame, pd.DataFrame) else pd.DataFrame()
    if not base.empty and "campaign" in base.columns:
        table_dss_campaign_rank_uncertainty = base[[c for c in ["campaign", "diagnostic_rank"] if c in base.columns]].copy()
    else:
        table_dss_campaign_rank_uncertainty = pd.DataFrame(columns=["campaign", "diagnostic_rank"])
    for col in ["bootstrap_rank_mean", "bootstrap_rank_sd", "top10_frequency", "top20_frequency", "stability_pass"]:
        table_dss_campaign_rank_uncertainty[col] = np.nan
    table_dss_campaign_rank_uncertainty["status"] = "skipped_missing_row_level_dml_analysis_df"
    if "table_dss_diagnostic_gate_campaigns" in globals() and "campaign" in table_dss_diagnostic_gate_campaigns.columns:
        table_dss_diagnostic_gate_campaigns = table_dss_diagnostic_gate_campaigns.drop(columns=[c for c in ["gate_stability", "stability_status"] if c in table_dss_diagnostic_gate_campaigns.columns])
        table_dss_diagnostic_gate_campaigns = table_dss_diagnostic_gate_campaigns.merge(table_dss_campaign_rank_uncertainty[["campaign", "stability_pass", "status"]].rename(columns={"stability_pass": "gate_stability", "status": "stability_status"}), on="campaign", how="left")
        table_dss_diagnostic_gate_campaigns["gate_all_full"] = False
        table_dss_gate_funnel = dss_build_gate_funnel(table_dss_diagnostic_gate_campaigns)
        dss_export_csv(table_dss_diagnostic_gate_campaigns, "table_dss_diagnostic_gate_campaigns.csv")
        dss_export_csv(table_dss_gate_funnel, "table_dss_gate_funnel.csv")
    dss_export_csv(table_dss_ranking_stability_summary, "table_dss_ranking_stability_summary.csv")
    dss_export_csv(table_dss_campaign_rank_uncertainty, "table_dss_campaign_rank_uncertainty.csv")
    dss_placeholder_figure("fig_dss_ranking_stability_top20", "Ranking stability requires row-level residualized dml_analysis_df; this export bundle only provides campaign-level tables.")
    display(table_dss_ranking_stability_summary)
else:
    
    
    
    def dss_numeric_time(row):
        if "ts_sec" in row.columns:
            return pd.to_numeric(row["ts_sec"], errors="coerce")
        if "timestamp" in row.columns:
            ser = row["timestamp"]
            if pd.api.types.is_datetime64_any_dtype(ser):
                return pd.to_datetime(ser, utc=True, errors="coerce").astype("int64") / 1e9
            return pd.to_numeric(ser, errors="coerce")
        return pd.Series(np.nan, index=row.index, dtype=float)
    
    
    def dss_campaign_scores(frame, campaign_universe=None, score_col="score_proxy"):
        if frame.empty or "campaign" not in frame.columns:
            return pd.DataFrame(columns=["campaign", "diagnostic_score", "diagnostic_rank"])
        tmp = frame.copy()
        tmp["campaign"] = tmp["campaign"].astype(str)
        if score_col not in tmp.columns:
            if {"D_tilde", "Y_tilde"}.issubset(tmp.columns):
                tmp[score_col] = pd.to_numeric(tmp["D_tilde"], errors="coerce") * pd.to_numeric(tmp["Y_tilde"], errors="coerce")
            else:
                tmp[score_col] = np.nan
        out = tmp.groupby("campaign", as_index=False).agg(diagnostic_score=(score_col, "sum"))
        if campaign_universe is not None:
            out = pd.DataFrame({"campaign": list(campaign_universe)}).merge(out, on="campaign", how="left")
            out["diagnostic_score"] = out["diagnostic_score"].fillna(0.0)
        out["diagnostic_rank"] = dss_rank_desc(out["diagnostic_score"])
        return out
    
    
    def dss_compare_rank_frames(left, right, left_rank="rank_left", right_rank="rank_right"):
        use = left.merge(right, on="campaign", how="inner")
        if use.empty or left_rank not in use.columns or right_rank not in use.columns:
            return {"n_campaigns": 0, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan}
        use[left_rank] = pd.to_numeric(use[left_rank], errors="coerce")
        use[right_rank] = pd.to_numeric(use[right_rank], errors="coerce")
        use = use.dropna(subset=[left_rank, right_rank])
        if use.empty:
            return {"n_campaigns": 0, "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan}
        out = {
            "n_campaigns": int(len(use)),
            "spearman_corr": dss_safe_corr(use[left_rank], use[right_rank], "spearman"),
            "kendall_corr": dss_safe_corr(use[left_rank], use[right_rank], "kendall"),
            "mean_abs_rank_shift": float((use[left_rank] - use[right_rank]).abs().mean()),
        }
        for k in [10, 20]:
            eff = int(min(k, len(use)))
            if eff <= 0:
                out[f"top{k}_overlap_share"] = np.nan
            else:
                a = set(use.loc[use[left_rank] <= eff, "campaign"].astype(str))
                b = set(use.loc[use[right_rank] <= eff, "campaign"].astype(str))
                out[f"top{k}_overlap_share"] = float(len(a & b) / eff)
        return out
    
    
    row = dss_row_frame_with_score() if "dss_row_frame_with_score" in globals() else dml_analysis_df.copy().reset_index(drop=True)
    row["campaign"] = row["campaign"].astype(str)
    support_campaigns = []
    if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_support" in table_dss_diagnostic_gate_campaigns.columns:
        support_campaigns = table_dss_diagnostic_gate_campaigns.loc[table_dss_diagnostic_gate_campaigns["gate_support"].fillna(False).astype(bool), "campaign"].astype(str).tolist()
    if not support_campaigns:
        support_campaigns = sorted(row["campaign"].dropna().astype(str).unique().tolist())
    
    base_rank_df = dss_campaign_scores(row.loc[row["campaign"].isin(support_campaigns)], support_campaigns).rename(columns={"diagnostic_rank": "diagnostic_rank_base"})
    stability_rows = []
    
    # M4A split-half stability.
    time_values = dss_numeric_time(row)
    if time_values.notna().sum() < 2 or len(support_campaigns) < 2:
        stability_rows.append({"comparison": "split-half", "n_campaigns": len(support_campaigns), "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "missing_time_or_campaign_support"})
    else:
        median_time = float(time_values.dropna().median())
        tmp = row.assign(_dss_time=time_values)
        split1 = dss_campaign_scores(tmp.loc[(tmp["_dss_time"] <= median_time) & tmp["campaign"].isin(support_campaigns)], support_campaigns).rename(columns={"diagnostic_rank": "rank_left"})
        split2 = dss_campaign_scores(tmp.loc[(tmp["_dss_time"] > median_time) & tmp["campaign"].isin(support_campaigns)], support_campaigns).rename(columns={"diagnostic_rank": "rank_right"})
        comp = dss_compare_rank_frames(split1[["campaign", "rank_left"]], split2[["campaign", "rank_right"]])
        stability_rows.append({"comparison": "split-half", **comp, "status": "ok"})
    
    # M4B user-level bootstrap stability.
    B = int(globals().get("DSS_BOOTSTRAP_B", 100))
    if len(row) > 1_000_000 or row["uid"].nunique() > 100_000:
        B = min(B, 30)
    rng = np.random.default_rng(20260529)
    if "uid" not in row.columns or len(support_campaigns) < 2:
        table_dss_campaign_rank_uncertainty = base_rank_df.rename(columns={"diagnostic_rank_base": "diagnostic_rank"})
        table_dss_campaign_rank_uncertainty["bootstrap_rank_mean"] = np.nan
        table_dss_campaign_rank_uncertainty["bootstrap_rank_sd"] = np.nan
        table_dss_campaign_rank_uncertainty["top10_frequency"] = np.nan
        table_dss_campaign_rank_uncertainty["top20_frequency"] = np.nan
        table_dss_campaign_rank_uncertainty["stability_pass"] = np.nan
        table_dss_campaign_rank_uncertainty["status"] = "bootstrap_unavailable"
        stability_rows.append({"comparison": "bootstrap", "n_campaigns": len(support_campaigns), "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "missing_uid_or_campaign_support", "bootstrap_B": 0})
    else:
        boot_source = row.loc[row["campaign"].isin(support_campaigns), ["uid", "campaign", "score_proxy"]].copy()
        boot_source["uid"] = boot_source["uid"].astype(str)
        boot_source["score_proxy"] = pd.to_numeric(boot_source["score_proxy"], errors="coerce").fillna(0.0)
        user_campaign = boot_source.groupby(["uid", "campaign"], as_index=False).agg(score=("score_proxy", "sum"))
        users = user_campaign["uid"].drop_duplicates().to_numpy()
        rank_records = []
        for b in range(B):
            sampled = rng.choice(users, size=len(users), replace=True)
            weights = pd.Series(sampled).value_counts().rename_axis("uid").reset_index(name="weight")
            weighted = user_campaign.merge(weights, on="uid", how="inner")
            scores = weighted.assign(weighted_score=weighted["score"] * weighted["weight"]).groupby("campaign")["weighted_score"].sum()
            scores = scores.reindex(support_campaigns).fillna(0.0)
            ranks = dss_rank_desc(scores).rename("rank").reset_index().rename(columns={"index": "campaign"})
            ranks["bootstrap_id"] = b
            rank_records.append(ranks)
        boot_ranks = pd.concat(rank_records, ignore_index=True) if rank_records else pd.DataFrame(columns=["campaign", "rank", "bootstrap_id"])
        table_dss_campaign_rank_uncertainty = boot_ranks.groupby("campaign", as_index=False).agg(
            bootstrap_rank_mean=("rank", "mean"),
            bootstrap_rank_sd=("rank", "std"),
            top10_frequency=("rank", lambda s: float((pd.to_numeric(s, errors="coerce") <= 10).mean())),
            top20_frequency=("rank", lambda s: float((pd.to_numeric(s, errors="coerce") <= 20).mean())),
        )
        table_dss_campaign_rank_uncertainty = base_rank_df[["campaign", "diagnostic_rank_base"]].rename(columns={"diagnostic_rank_base": "diagnostic_rank"}).merge(table_dss_campaign_rank_uncertainty, on="campaign", how="left")
        p75_rank_sd = table_dss_campaign_rank_uncertainty["bootstrap_rank_sd"].dropna().quantile(0.75) if table_dss_campaign_rank_uncertainty["bootstrap_rank_sd"].notna().any() else np.nan
        top20_base = pd.to_numeric(table_dss_campaign_rank_uncertainty["diagnostic_rank"], errors="coerce") <= 20
        table_dss_campaign_rank_uncertainty["stability_pass"] = np.where(
            top20_base,
            table_dss_campaign_rank_uncertainty["top20_frequency"] >= 0.5,
            table_dss_campaign_rank_uncertainty["bootstrap_rank_sd"] <= p75_rank_sd if np.isfinite(p75_rank_sd) else np.nan,
        )
        table_dss_campaign_rank_uncertainty["status"] = "ok"
        base_vs_boot = table_dss_campaign_rank_uncertainty.rename(columns={"diagnostic_rank": "rank_left", "bootstrap_rank_mean": "rank_right"})
        comp = dss_compare_rank_frames(base_vs_boot[["campaign", "rank_left"]], base_vs_boot[["campaign", "rank_right"]])
        stability_rows.append({"comparison": "bootstrap", **comp, "status": "ok", "bootstrap_B": B})
    
    # M4C cached horizon stability.
    def dss_payload_campaign_rank(payload, campaign_universe):
        try:
            dfp = payload["df"].copy().reset_index(drop=True)
            keep = np.asarray(payload.get("keep", np.ones(len(dfp), dtype=bool))).astype(bool)
            dfp = dfp.loc[keep].copy().reset_index(drop=True)
            D = np.asarray(payload["D"])[keep].astype(float)
            Y = np.asarray(payload["Y"])[keep].astype(float)
            mhat = np.asarray(payload["mhat"])[keep].astype(float)
            ghat = np.asarray(payload["ghat"])[keep].astype(float)
            dfp["campaign"] = dfp["campaign"].astype(str)
            dfp["score_proxy"] = (D - mhat) * (Y - ghat)
            return dss_campaign_scores(dfp.loc[dfp["campaign"].isin(campaign_universe)], campaign_universe)
        except Exception:
            return pd.DataFrame(columns=["campaign", "diagnostic_score", "diagnostic_rank"])
    
    horizon_ranks = {}
    if "STEPD_PAYLOADS" in globals() and isinstance(STEPD_PAYLOADS, dict):
        for key, payload in STEPD_PAYLOADS.items():
            try:
                horizon = float((payload.get("summary") or {}).get("horizon_hours", np.nan))
            except Exception:
                horizon = np.nan
            if np.isfinite(horizon) and horizon in [1.0, 6.0, 24.0, 168.0] and horizon not in horizon_ranks:
                rank = dss_payload_campaign_rank(payload, support_campaigns)
                if not rank.empty:
                    horizon_ranks[horizon] = rank.rename(columns={"diagnostic_rank": f"rank_{int(horizon)}h"})
    if 24.0 not in horizon_ranks and not base_rank_df.empty:
        horizon_ranks[24.0] = base_rank_df.rename(columns={"diagnostic_rank_base": "rank_24h", "diagnostic_score": "diagnostic_score"})
    for h in [1.0, 6.0, 168.0]:
        label = f"horizon {int(h)}h vs 24h"
        if h in horizon_ranks and 24.0 in horizon_ranks:
            left = horizon_ranks[h][["campaign", f"rank_{int(h)}h"]].rename(columns={f"rank_{int(h)}h": "rank_left"})
            right = horizon_ranks[24.0][["campaign", "rank_24h"]].rename(columns={"rank_24h": "rank_right"})
            comp = dss_compare_rank_frames(left, right)
            stability_rows.append({"comparison": label, **comp, "status": "ok"})
        else:
            stability_rows.append({"comparison": label, "n_campaigns": len(support_campaigns), "spearman_corr": np.nan, "kendall_corr": np.nan, "top10_overlap_share": np.nan, "top20_overlap_share": np.nan, "mean_abs_rank_shift": np.nan, "status": "cached_horizon_payload_not_available"})
    
    table_dss_ranking_stability_summary = pd.DataFrame(stability_rows)
    dss_export_csv(table_dss_ranking_stability_summary, "table_dss_ranking_stability_summary.csv")
    dss_export_csv(table_dss_campaign_rank_uncertainty, "table_dss_campaign_rank_uncertainty.csv")
    
    # Update M2 gate table with stability gate.
    if "table_dss_diagnostic_gate_campaigns" in globals():
        update_cols = ["campaign", "stability_pass", "bootstrap_rank_sd", "top20_frequency", "status"]
        stability_update = table_dss_campaign_rank_uncertainty[[c for c in update_cols if c in table_dss_campaign_rank_uncertainty.columns]].copy()
        stability_update = stability_update.rename(columns={"stability_pass": "gate_stability", "status": "stability_status"})
        drop_cols = [c for c in ["gate_stability", "bootstrap_rank_sd", "top20_frequency", "stability_status"] if c in table_dss_diagnostic_gate_campaigns.columns]
        table_dss_diagnostic_gate_campaigns = table_dss_diagnostic_gate_campaigns.drop(columns=drop_cols).merge(stability_update, on="campaign", how="left")
        table_dss_diagnostic_gate_campaigns["gate_all_full"] = (
            table_dss_diagnostic_gate_campaigns["gate_support"].fillna(False).astype(bool)
            & table_dss_diagnostic_gate_campaigns["gate_overlap"].fillna(False).astype(bool)
            & table_dss_diagnostic_gate_campaigns["gate_placebo"].fillna(False).astype(bool)
            & table_dss_diagnostic_gate_campaigns["gate_stability"].fillna(False).astype(bool)
        )
        available_mask = table_dss_diagnostic_gate_campaigns["gate_support"].fillna(False).astype(bool) & table_dss_diagnostic_gate_campaigns["gate_overlap"].fillna(False).astype(bool)
        if table_dss_diagnostic_gate_campaigns["gate_placebo"].notna().any():
            available_mask = available_mask & table_dss_diagnostic_gate_campaigns["gate_placebo"].fillna(False).astype(bool)
        if table_dss_diagnostic_gate_campaigns["gate_stability"].notna().any():
            available_mask = available_mask & table_dss_diagnostic_gate_campaigns["gate_stability"].fillna(False).astype(bool)
        table_dss_diagnostic_gate_campaigns["gate_all_available_only"] = available_mask
        table_dss_gate_funnel = dss_build_gate_funnel(table_dss_diagnostic_gate_campaigns)
        dss_export_csv(table_dss_diagnostic_gate_campaigns, "table_dss_diagnostic_gate_campaigns.csv")
        dss_export_csv(table_dss_gate_funnel, "table_dss_gate_funnel.csv")
    
    plot_df = table_dss_campaign_rank_uncertainty.sort_values("diagnostic_rank").head(20).copy()
    if plot_df.empty or "top20_frequency" not in plot_df.columns:
        dss_placeholder_figure("fig_dss_ranking_stability_top20", "Ranking stability figure unavailable: bootstrap diagnostics were not computed.")
    else:
        fig, ax = plt.subplots(figsize=(9, 4.8))
        ax.bar(plot_df["campaign"].astype(str), plot_df["top20_frequency"], color="#59A14F")
        ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("top-20 bootstrap frequency")
        ax.set_title("DSS ranking stability for diagnostic top 20")
        ax.tick_params(axis="x", rotation=75)
        dss_save_figure(fig, "fig_dss_ranking_stability_top20")
        plt.close(fig)
    
    display(table_dss_ranking_stability_summary)


## 11. Time-Split Managerial Decision Simulation


In [ ]:
# ====== Step M5: Time-Split Decision Simulation ======
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DSS_ROW_LEVEL_AVAILABLE = "dml_analysis_df" in globals() and isinstance(dml_analysis_df, pd.DataFrame) and not dml_analysis_df.empty
if not DSS_ROW_LEVEL_AVAILABLE:
    table_dss_time_split_decision_simulation = pd.DataFrame([{
        "ranking_rule": "not_available", "k": np.nan, "selected_campaigns": "", "n_eval_obs": 0,
        "future_conversion_rate": np.nan, "future_click_rate": np.nan, "future_attributed_conversion_rate": np.nan,
        "future_diagnostic_score_mean": np.nan, "future_placebo_risk": np.nan,
        "status": "skipped_missing_row_level_dml_analysis_df",
        "note": "Time-split decision simulation requires row-level timestamps and residualized scores; recovered campaign-level exports are insufficient."
    }])
    dss_export_csv(table_dss_time_split_decision_simulation, "table_dss_time_split_decision_simulation.csv")
    dss_placeholder_figure("fig_dss_time_split_decision_simulation", "Time-split decision simulation requires row-level dml_analysis_df with timestamps and residualized scores.")
    display(table_dss_time_split_decision_simulation)
else:
    
    
    row = dss_row_frame_with_score() if "dss_row_frame_with_score" in globals() else dml_analysis_df.copy().reset_index(drop=True)
    row["campaign"] = row["campaign"].astype(str)
    row["_dss_time"] = dss_numeric_time(row) if "dss_numeric_time" in globals() else pd.to_numeric(row.get("ts_sec", np.nan), errors="coerce")
    
    support_campaigns = []
    if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_support" in table_dss_diagnostic_gate_campaigns.columns:
        support_campaigns = table_dss_diagnostic_gate_campaigns.loc[table_dss_diagnostic_gate_campaigns["gate_support"].fillna(False).astype(bool), "campaign"].astype(str).tolist()
    if not support_campaigns:
        support_campaigns = sorted(row["campaign"].dropna().astype(str).unique().tolist())
    
    
    def dss_last_touch_counts(frame):
        needed = {"uid", "campaign", "_dss_time", "conversion"}
        if not needed.issubset(frame.columns):
            return pd.DataFrame(columns=["campaign", "score"]), "not_available"
        tmp = frame.loc[frame["_dss_time"].notna()].copy()
        if tmp.empty:
            return pd.DataFrame(columns=["campaign", "score"]), "missing_time"
        tmp["conversion"] = (pd.to_numeric(tmp["conversion"], errors="coerce").fillna(0) > 0).astype(int)
        tmp = tmp.sort_values(["uid", "_dss_time"])
        picked = []
        for _, g in tmp.groupby("uid", sort=False):
            touches = g[["_dss_time", "campaign"]]
            for conv_time in g.loc[g["conversion"] == 1, "_dss_time"]:
                prior = touches.loc[touches["_dss_time"] < conv_time]
                if not prior.empty:
                    picked.append(str(prior.iloc[-1]["campaign"]))
        if not picked:
            return pd.DataFrame(columns=["campaign", "score"]), "no_attributable_last_touch"
        out = pd.Series(picked).value_counts().rename_axis("campaign").reset_index(name="score")
        return out, "ok"
    
    
    def dss_training_rankings(train):
        train = train.loc[train["campaign"].isin(support_campaigns)].copy()
        rankings = []
        exposure = train.groupby("campaign", as_index=False).size().rename(columns={"size": "score"})
        exposure["ranking_rule"] = "raw_exposure"
        rankings.append(exposure)
        click = train.groupby("campaign", as_index=False).agg(score=("D_num", "sum")) if "D_num" in train.columns else pd.DataFrame(columns=["campaign", "score"])
        click["ranking_rule"] = "raw_click"
        rankings.append(click)
        diag = train.groupby("campaign", as_index=False).agg(score=("score_proxy", "sum")) if "score_proxy" in train.columns else pd.DataFrame(columns=["campaign", "score"])
        diag["ranking_rule"] = "diagnostic"
        rankings.append(diag)
        last_touch, last_touch_status = dss_last_touch_counts(train)
        if not last_touch.empty:
            last_touch["ranking_rule"] = "last_touch"
            rankings.append(last_touch)
        out = pd.concat(rankings, ignore_index=True, sort=False) if rankings else pd.DataFrame(columns=["campaign", "score", "ranking_rule"])
        out["score"] = pd.to_numeric(out["score"], errors="coerce")
        out["rank"] = out.groupby("ranking_rule")["score"].rank(ascending=False, method="min")
        return out, last_touch_status
    
    
    def dss_eval_selected(eval_df, selected):
        selected = [str(c) for c in selected]
        sub = eval_df.loc[eval_df["campaign"].isin(selected)].copy()
        if sub.empty:
            return {
                "n_eval_obs": 0,
                "future_conversion_rate": np.nan,
                "future_click_rate": np.nan,
                "future_attributed_conversion_rate": np.nan,
                "future_diagnostic_score_mean": np.nan,
                "future_placebo_risk": np.nan,
            }
        placebo_risk = np.nan
        if "placebo_score_proxy" in sub.columns:
            placebo_risk = float(pd.to_numeric(sub["placebo_score_proxy"], errors="coerce").mean())
        elif "Y_placebo" in sub.columns:
            placebo_risk = float(pd.to_numeric(sub["Y_placebo"], errors="coerce").mean())
        elif "build_placebo_outcome" in globals() and "ts_sec" in sub.columns:
            try:
                horizon_hours = float(globals().get("dml_analysis_metadata", {}).get("outcome_hours", 24.0)) if "dml_analysis_metadata" in globals() else 24.0
                placebo_risk = float(np.mean(build_placebo_outcome(sub, horizon_hours=horizon_hours)))
            except Exception:
                placebo_risk = np.nan
        return {
            "n_eval_obs": int(len(sub)),
            "future_conversion_rate": float(pd.to_numeric(sub.get("Y", np.nan), errors="coerce").mean()),
            "future_click_rate": float(pd.to_numeric(sub.get("D", np.nan), errors="coerce").mean()),
            "future_attributed_conversion_rate": float(pd.to_numeric(sub["conversion"], errors="coerce").mean()) if "conversion" in sub.columns else np.nan,
            "future_diagnostic_score_mean": float(pd.to_numeric(sub.get("score_proxy", np.nan), errors="coerce").mean()),
            "future_placebo_risk": placebo_risk,
        }
    
    if row["_dss_time"].notna().sum() < 2:
        table_dss_time_split_decision_simulation = pd.DataFrame([{
            "ranking_rule": "not_available", "k": np.nan, "selected_campaigns": "", "n_eval_obs": 0,
            "future_conversion_rate": np.nan, "future_click_rate": np.nan, "future_attributed_conversion_rate": np.nan,
            "future_diagnostic_score_mean": np.nan, "future_placebo_risk": np.nan, "status": "missing_time_column",
        }])
    else:
        cutoff = float(row["_dss_time"].dropna().quantile(0.70))
        train = row.loc[row["_dss_time"] <= cutoff].copy()
        eval_df = row.loc[row["_dss_time"] > cutoff].copy()
        rankings, last_touch_status = dss_training_rankings(train)
        sim_rows = []
        for rule, g in rankings.groupby("ranking_rule"):
            ranked = g.dropna(subset=["score"]).sort_values(["rank", "campaign"])
            for k in [5, 10, 20]:
                effective_k = int(min(k, len(ranked)))
                selected = ranked.head(effective_k)["campaign"].astype(str).tolist()
                metrics = dss_eval_selected(eval_df, selected)
                sim_rows.append({
                    "ranking_rule": rule,
                    "k": int(k),
                    "selected_campaigns": dss_semicolon(selected),
                    **metrics,
                    "status": "ok" if effective_k > 0 else "no_selected_campaigns",
                    "time_split_cutoff_quantile": 0.70,
                    "last_touch_status": last_touch_status if rule == "last_touch" else "not_applicable",
                })
        if "last_touch" not in set(rankings.get("ranking_rule", [])):
            for k in [5, 10, 20]:
                sim_rows.append({
                    "ranking_rule": "last_touch", "k": int(k), "selected_campaigns": "", "n_eval_obs": 0,
                    "future_conversion_rate": np.nan, "future_click_rate": np.nan, "future_attributed_conversion_rate": np.nan,
                    "future_diagnostic_score_mean": np.nan, "future_placebo_risk": np.nan, "status": f"skipped_{last_touch_status}",
                    "time_split_cutoff_quantile": 0.70, "last_touch_status": last_touch_status,
                })
        table_dss_time_split_decision_simulation = pd.DataFrame(sim_rows)
    
    dss_export_csv(table_dss_time_split_decision_simulation, "table_dss_time_split_decision_simulation.csv")
    
    plot_df = table_dss_time_split_decision_simulation.loc[table_dss_time_split_decision_simulation.get("status", "").eq("ok")].copy() if "status" in table_dss_time_split_decision_simulation.columns else pd.DataFrame()
    if plot_df.empty:
        dss_placeholder_figure("fig_dss_time_split_decision_simulation", "Time-split decision simulation unavailable: no valid time split or rankings.")
    else:
        metric_col = "future_conversion_rate" if plot_df["future_conversion_rate"].notna().any() else "future_diagnostic_score_mean"
        pivot = plot_df.pivot_table(index="k", columns="ranking_rule", values=metric_col, aggfunc="mean")
        fig, ax = plt.subplots(figsize=(8, 4.8))
        pivot.plot(kind="bar", ax=ax)
        ax.set_ylabel(metric_col.replace("_", " "))
        ax.set_title("Time-split decision simulation")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(title="ranking rule", fontsize=8)
        dss_save_figure(fig, "fig_dss_time_split_decision_simulation")
        plt.close(fig)
    
    display(table_dss_time_split_decision_simulation)


## 13. Reproduce Paper Figures


In [ ]:
# ====== Step M6: Export DSS Experiment Bundle ======
import numpy as np
import pandas as pd

DSS_TABLE_EXPORTS = {
    "table_dss_reprioritization_metrics": "table_dss_reprioritization_metrics.csv",
    "table_dss_topk_overlap_details": "table_dss_topk_overlap_details.csv",
    "table_dss_diagnostic_gate_campaigns": "table_dss_diagnostic_gate_campaigns.csv",
    "table_dss_gate_funnel": "table_dss_gate_funnel.csv",
    "table_dss_decision_risk_classification": "table_dss_decision_risk_classification.csv",
    "table_dss_decision_risk_summary": "table_dss_decision_risk_summary.csv",
    "table_dss_ranking_stability_summary": "table_dss_ranking_stability_summary.csv",
    "table_dss_campaign_rank_uncertainty": "table_dss_campaign_rank_uncertainty.csv",
    "table_dss_time_split_decision_simulation": "table_dss_time_split_decision_simulation.csv",
}

for obj_name, filename in DSS_TABLE_EXPORTS.items():
    obj = globals().get(obj_name)
    if isinstance(obj, pd.DataFrame):
        dss_export_csv(obj, filename)
    else:
        dss_export_csv(pd.DataFrame([{"status": f"{obj_name}_not_available"}]), filename)

summary_dss_experiments = {
    "title": "DSS campaign-level decision-support experiments",
    "number_of_campaigns": int(len(table_dss_diagnostic_gate_campaigns)) if "table_dss_diagnostic_gate_campaigns" in globals() else None,
    "number_support_pass": int(table_dss_diagnostic_gate_campaigns["gate_support"].fillna(False).sum()) if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_support" in table_dss_diagnostic_gate_campaigns.columns else None,
    "number_overlap_pass": int(table_dss_diagnostic_gate_campaigns["gate_overlap"].fillna(False).sum()) if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_overlap" in table_dss_diagnostic_gate_campaigns.columns else None,
    "number_placebo_pass": int(table_dss_diagnostic_gate_campaigns["gate_placebo"].fillna(False).sum()) if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_placebo" in table_dss_diagnostic_gate_campaigns.columns and table_dss_diagnostic_gate_campaigns["gate_placebo"].notna().any() else None,
    "number_stability_pass": int(table_dss_diagnostic_gate_campaigns["gate_stability"].fillna(False).sum()) if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_stability" in table_dss_diagnostic_gate_campaigns.columns and table_dss_diagnostic_gate_campaigns["gate_stability"].notna().any() else None,
    "number_all_gates_pass": int(table_dss_diagnostic_gate_campaigns["gate_all_full"].fillna(False).sum()) if "table_dss_diagnostic_gate_campaigns" in globals() and "gate_all_full" in table_dss_diagnostic_gate_campaigns.columns else None,
    "top_k_overlap_results": table_dss_reprioritization_metrics.to_dict(orient="records") if "table_dss_reprioritization_metrics" in globals() else [],
    "decision_risk_class_counts": table_dss_decision_risk_classification["decision_class"].value_counts(dropna=False).to_dict() if "table_dss_decision_risk_classification" in globals() and "decision_class" in table_dss_decision_risk_classification.columns else {},
    "stability_summary": table_dss_ranking_stability_summary.to_dict(orient="records") if "table_dss_ranking_stability_summary" in globals() else [],
    "time_split_simulation_highlights": (
        table_dss_time_split_decision_simulation.loc[table_dss_time_split_decision_simulation.get("status", "").eq("ok")]
        .sort_values(["k", "future_conversion_rate"], ascending=[True, False])
        .groupby("k", as_index=False)
        .head(1)
        .to_dict(orient="records")
        if "table_dss_time_split_decision_simulation" in globals() and "future_conversion_rate" in table_dss_time_split_decision_simulation.columns else []
    ),
    "figure_directory": str(DSS_FIG_DIR) if "DSS_FIG_DIR" in globals() else None,
    "paper_table_directory": str(PAPER_TABLE_DIR),
    "interpretation_note": "Time-split outputs are predictive decision metrics and illustrative decision-value evidence, not treatment-effect or ROI estimates.",
}
dss_export_json(summary_dss_experiments, "summary_dss_experiments.json")

print("\nDSS top-K overlap summary")
if "table_dss_reprioritization_metrics" in globals():
    cols = [c for c in ["comparison", "n_campaigns", "spearman_rank_corr", "top5_overlap_share", "top10_overlap_share", "top20_overlap_share"] if c in table_dss_reprioritization_metrics.columns]
    display(table_dss_reprioritization_metrics[cols])

print("\nDSS diagnostic gate funnel")
if "table_dss_gate_funnel" in globals():
    display(table_dss_gate_funnel[["gate_step", "n_campaigns_remaining", "share_of_all"]])

print("\nDSS decision-risk class counts")
if "table_dss_decision_risk_classification" in globals() and "decision_class" in table_dss_decision_risk_classification.columns:
    display(table_dss_decision_risk_classification["decision_class"].value_counts(dropna=False).rename_axis("decision_class").reset_index(name="n_campaigns"))

print("\nDSS time-split simulation comparison")
if "table_dss_time_split_decision_simulation" in globals():
    cols = [c for c in ["ranking_rule", "k", "n_eval_obs", "future_conversion_rate", "future_click_rate", "future_diagnostic_score_mean", "status"] if c in table_dss_time_split_decision_simulation.columns]
    display(table_dss_time_split_decision_simulation[cols])


## 12. Reproduce Paper Tables

This section gathers the main tables from Steps F, G, and H into a single paper-ready directory and writes a compact summary bundle for drafting results sections.


In [ ]:
# ====== Step I: Export paper table bundle ======
from pathlib import Path
import json
import pandas as pd
import shutil

PAPER_TABLE_DIR = (PAPER_TABLE_DIR if "PAPER_TABLE_DIR" in globals() else PAPER_OUTPUT_DIR / "tables")
PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)

def export_csv(df, filename):
    path = PAPER_TABLE_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def _to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, tuple):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    return obj

def export_json(obj, filename):
    path = PAPER_TABLE_DIR / filename
    with open(path, "w") as f:
        json.dump(_to_builtin(obj), f, indent=2)
    print("Saved:", path)
    return path

if "raw_outcome_overall_df" in globals():
    export_csv(raw_outcome_overall_df, "table_raw_outcome_overall.csv")
if "raw_outcome_by_campaign_decile_df" in globals():
    export_csv(raw_outcome_by_campaign_decile_df, "table_raw_outcome_by_campaign_decile.csv")
if "raw_outcome_by_user_activity_decile_df" in globals():
    export_csv(raw_outcome_by_user_activity_decile_df, "table_raw_outcome_by_user_activity_decile.csv")
if "raw_outcome_by_hour_df" in globals():
    export_csv(raw_outcome_by_hour_df, "table_raw_outcome_by_hour.csv")
if "raw_outcome_by_day_index_df" in globals():
    export_csv(raw_outcome_by_day_index_df, "table_raw_outcome_by_day_index.csv")
if "raw_outcome_by_recency_bucket_df" in globals():
    export_csv(raw_outcome_by_recency_bucket_df, "table_raw_outcome_by_recency_bucket.csv")
if "raw_outcome_top_vs_tail_campaigns_df" in globals():
    export_csv(raw_outcome_top_vs_tail_campaigns_df, "table_raw_outcome_top_vs_tail_campaigns.csv")

if "restriction_prior_activity_df" in globals():
    export_csv(restriction_prior_activity_df, "table_restriction_prior_activity.csv")
if "restriction_overlap_trim_df" in globals():
    export_csv(restriction_overlap_trim_df, "table_restriction_overlap_trim_grid.csv")
if "restriction_campaign_relevance_df" in globals():
    export_csv(restriction_campaign_relevance_df, "table_restriction_campaign_relevance.csv")
if "restriction_near_conversion_exclusion_df" in globals():
    export_csv(restriction_near_conversion_exclusion_df, "table_restriction_near_conversion_exclusion.csv")
if "stepD_restriction_grid_df" in globals():
    export_csv(stepD_restriction_grid_df, "table_stepD_restriction_grid.csv")
if "stepD_horizon_grid_baseline_df" in globals():
    export_csv(stepD_horizon_grid_baseline_df, "table_stepD_horizon_grid_baseline.csv")
if "stepD_horizon_grid_preferred_df" in globals():
    export_csv(stepD_horizon_grid_preferred_df, "table_stepD_horizon_grid_preferred.csv")

if "sample_summary_df" in globals():
    export_csv(sample_summary_df, "table1_sample_construction.csv")
if "main_ate_df" in globals():
    export_csv(main_ate_df, "table2_main_ate_24h.csv")
if "overlap_df" in globals():
    export_csv(overlap_df, "table3_overlap_diagnostics.csv")
if "nuisance_df" in globals():
    export_csv(nuisance_df, "table4_nuisance_diagnostics.csv")
if "horizon_df" in globals():
    export_csv(horizon_df, "table5_horizon_robustness.csv")
if "trim_df" in globals():
    export_csv(trim_df, "table6_trim_robustness.csv")

if "hetero_propensity_df" in globals():
    export_csv(hetero_propensity_df, "table7a_heterogeneity_propensity_quintile.csv")
if "hetero_recency_df" in globals():
    export_csv(hetero_recency_df, "table7b_heterogeneity_recency.csv")
if "hetero_intensity_df" in globals():
    export_csv(hetero_intensity_df, "table7c_heterogeneity_impression_intensity.csv")
if "hetero_all_df" in globals():
    export_csv(hetero_all_df, "table7_all_heterogeneity_segments.csv")

if "campaign_credit" in globals():
    export_csv(campaign_credit, "table8a_campaign_diagnostic_score_proxy.csv")
if "compare_df" in globals():
    export_csv(compare_df, "table8b_campaign_rank_shift.csv")

    top_diagnostic = compare_df.sort_values("diagnostic_rank_proxy").head(20).copy()
    export_csv(top_diagnostic, "table8c_top20_diagnostic_campaigns.csv")

    if "rank_shift_exposure" in compare_df.columns:
        top_shift = compare_df.assign(
            abs_rank_shift_exposure=compare_df["rank_shift_exposure"].abs()
        ).sort_values("abs_rank_shift_exposure", ascending=False).head(20).copy()
        export_csv(top_shift, "table8d_top20_rank_shifts_exposure_vs_diagnostic.csv")

    if "heuristic_winner_diagnostic_loser" in compare_df.columns:
        export_csv(compare_df.loc[compare_df["heuristic_winner_diagnostic_loser"]].copy(), "table8e_heuristic_winners_diagnostic_losers.csv")
    if "heuristic_loser_diagnostic_winner" in compare_df.columns:
        export_csv(compare_df.loc[compare_df["heuristic_loser_diagnostic_winner"]].copy(), "table8f_heuristic_losers_diagnostic_winners.csv")

if "campaign_results" in globals() and isinstance(campaign_results, pd.DataFrame):
    export_csv(campaign_results, "table9_campaign_level_results_with_gates.csv")
    if "classification" in campaign_results.columns:
        robust_campaigns = campaign_results.loc[
            campaign_results["classification"].isin(["robust_positive", "robust_negative", "imprecise_but_clean"])
        ].copy()
        export_csv(robust_campaigns, "table9b_robust_campaigns.csv")

if "summary_raw_outcome_diagnostics" in globals():
    export_json(summary_raw_outcome_diagnostics, "summary_raw_outcome_diagnostics.json")
if "raw_outcome_overall_df" in globals():
    export_json(raw_outcome_overall_df.iloc[0].to_dict(), "summary_raw_outcome_overall.json")
if "summary_stepD_restriction_grid" in globals():
    export_json(summary_stepD_restriction_grid, "summary_stepD_restriction_grid.json")
if "summary_stepD_horizon_selection" in globals():
    export_json(summary_stepD_horizon_selection, "summary_stepD_horizon_selection.json")
if "stepD_final_spec" in globals():
    export_json(stepD_final_spec, "stepD_final_spec.json")

summary_bundle = {}
if "summary_numbers" in globals():
    summary_bundle["main_ate_summary"] = summary_numbers
if "analysis_validation" in globals():
    summary_bundle["step_de_validation"] = analysis_validation
if "summary_attr" in globals():
    summary_bundle["campaign_rank_summary"] = summary_attr
if "summary_raw_outcome_diagnostics" in globals():
    summary_bundle["raw_outcome_diagnostics"] = summary_raw_outcome_diagnostics
if "summary_stepD_restriction_grid" in globals():
    summary_bundle["stepd_restriction_summary"] = summary_stepD_restriction_grid
if "summary_stepD_horizon_selection" in globals():
    summary_bundle["stepd_horizon_selection"] = summary_stepD_horizon_selection
if "stepD_final_spec" in globals():
    summary_bundle["stepd_final_spec"] = stepD_final_spec
if "main_ate_df" in globals() and len(main_ate_df) > 0:
    summary_bundle["main_24h_result"] = main_ate_df.iloc[0].to_dict()
if "horizon_df" in globals():
    summary_bundle["horizon_results"] = horizon_df.to_dict(orient="records")
if "stepD_horizon_grid_preferred_df" in globals():
    summary_bundle["preferred_horizon_grid"] = stepD_horizon_grid_preferred_df.to_dict(orient="records")

export_json(summary_bundle, "summary_bundle_for_writing.json")

table_index = [
    {"paper_table": "D0", "file": "table_raw_outcome_overall.csv", "content": "Overall 24h raw outcome sparsity diagnostics"},
    {"paper_table": "D0", "file": "table_raw_outcome_by_campaign_decile.csv", "content": "Raw outcome diagnostics by campaign decile"},
    {"paper_table": "D0", "file": "table_raw_outcome_by_user_activity_decile.csv", "content": "Raw outcome diagnostics by user activity decile"},
    {"paper_table": "D0", "file": "table_raw_outcome_by_hour.csv", "content": "Raw outcome diagnostics by hour of day"},
    {"paper_table": "D0", "file": "table_raw_outcome_by_day_index.csv", "content": "Raw outcome diagnostics by day index"},
    {"paper_table": "D0", "file": "table_raw_outcome_by_recency_bucket.csv", "content": "Raw outcome diagnostics by recency bucket"},
    {"paper_table": "D0", "file": "table_raw_outcome_top_vs_tail_campaigns.csv", "content": "Raw outcome diagnostics for top vs tail campaigns"},
    {"paper_table": "D1", "file": "table_stepD_restriction_grid.csv", "content": "Master restriction comparison grid"},
    {"paper_table": "D2", "file": "table_stepD_horizon_grid_baseline.csv", "content": "Horizon comparison on the baseline sample"},
    {"paper_table": "D2", "file": "table_stepD_horizon_grid_preferred.csv", "content": "Horizon comparison on the preferred restricted sample"},
    {"paper_table": "D3", "file": "summary_raw_outcome_diagnostics.json", "content": "Summary of raw outcome diagnostics"},
    {"paper_table": "D3", "file": "summary_raw_outcome_overall.json", "content": "Overall 24h raw outcome diagnostic summary"},
    {"paper_table": "D3", "file": "summary_stepD_restriction_grid.json", "content": "Summary of restriction comparison"},
    {"paper_table": "D3", "file": "summary_stepD_horizon_selection.json", "content": "Summary of horizon selection"},
    {"paper_table": "D3", "file": "stepD_final_spec.json", "content": "Final selected Step D specification"},
    {"paper_table": "Table 1", "file": "table1_sample_construction.csv", "content": "Sample construction and outcome summary"},
    {"paper_table": "Table 2", "file": "table2_main_ate_24h.csv", "content": "Main 24-hour DML estimate"},
    {"paper_table": "Table 3", "file": "table3_overlap_diagnostics.csv", "content": "Overlap diagnostics"},
    {"paper_table": "Table 4", "file": "table4_nuisance_diagnostics.csv", "content": "Nuisance-model diagnostics"},
    {"paper_table": "Table 5", "file": "table5_horizon_robustness.csv", "content": "Horizon robustness"},
    {"paper_table": "Table 6", "file": "table6_trim_robustness.csv", "content": "Trim robustness"},
    {"paper_table": "Table 7A", "file": "table7a_heterogeneity_propensity_quintile.csv", "content": "Heterogeneity by propensity quintile"},
    {"paper_table": "Table 7B", "file": "table7b_heterogeneity_recency.csv", "content": "Heterogeneity by prior-click recency"},
    {"paper_table": "Table 7C", "file": "table7c_heterogeneity_impression_intensity.csv", "content": "Heterogeneity by impression intensity"},
    {"paper_table": "Table 8A", "file": "table8a_campaign_diagnostic_score_proxy.csv", "content": "Campaign-level diagnostic credit proxy"},
    {"paper_table": "Table 8B", "file": "table8b_campaign_rank_shift.csv", "content": "Campaign rank shift: diagnostic vs heuristic"},
    {"paper_table": "Table 8C", "file": "table8c_top20_diagnostic_campaigns.csv", "content": "Top 20 campaigns under diagnostic ranking"},
    {"paper_table": "Table 8D", "file": "table8d_top20_rank_shifts_exposure_vs_diagnostic.csv", "content": "Largest rank shifts"},
    {"paper_table": "Table 8E", "file": "table8e_heuristic_winners_diagnostic_losers.csv", "content": "Heuristic winners but diagnostic losers"},
    {"paper_table": "Table 8F", "file": "table8f_heuristic_losers_diagnostic_winners.csv", "content": "Heuristic losers but diagnostic winners"},
    {"paper_table": "Summary bundle", "file": "summary_bundle_for_writing.json", "content": "Key numbers for abstract/results writing"},
]
table_index_df = pd.DataFrame(table_index)
display(table_index_df)
export_csv(table_index_df, "paper_table_index.csv")

zip_path = shutil.make_archive(str(PAPER_TABLE_DIR), "zip", PAPER_TABLE_DIR)
print("Created zip:", zip_path)

## 12. Reproduce Paper Tables

This section appends the new selection-adjusted heterogeneity and support-aware campaign ranking outputs to the paper export bundle and refreshes `paper_table_index.csv` plus `summary_bundle_for_writing.json`.


In [ ]:
# ====== Step I2: Export DML heterogeneity + ranking bundle ======
from pathlib import Path
import pandas as pd

PAPER_TABLE_DIR = (PAPER_TABLE_DIR if "PAPER_TABLE_DIR" in globals() else PAPER_OUTPUT_DIR / "tables")
PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)

if "export_csv" not in globals():
    def export_csv(df, filename):
        path = PAPER_TABLE_DIR / filename
        df.to_csv(path, index=False)
        print("Saved:", path)
        return path

if "export_json" not in globals():
    import json

    def export_json(obj, filename):
        path = PAPER_TABLE_DIR / filename
        with open(path, "w") as f:
            json.dump(dml_to_builtin(obj), f, indent=2)
        print("Saved:", path)
        return path

if "dml_analysis_frame_summary_df" in globals():
    export_csv(dml_analysis_frame_summary_df, "table_dml_analysis_frame_summary.csv")
if "summary_dml_analysis_frame" in globals():
    export_json(summary_dml_analysis_frame, "summary_dml_analysis_frame.json")

if "dml_heterogeneity_propensity_df" in globals():
    export_csv(dml_heterogeneity_propensity_df, "table_heterogeneity_propensity.csv")
if "dml_heterogeneity_recency_df" in globals():
    export_csv(dml_heterogeneity_recency_df, "table_heterogeneity_recency.csv")
if "dml_heterogeneity_intensity_df" in globals():
    export_csv(dml_heterogeneity_intensity_df, "table_heterogeneity_intensity.csv")
if "dml_heterogeneity_all_df" in globals():
    export_csv(dml_heterogeneity_all_df, "table_heterogeneity_all.csv")

if "campaign_support_diagnostics_df" in globals():
    export_csv(campaign_support_diagnostics_df, "table_campaign_support_diagnostics.csv")
if "campaign_dml_informed_ranking_df" in globals():
    export_csv(campaign_dml_informed_ranking_df, "table_campaign_dml_informed_ranking.csv")
if "campaign_rank_shift_dml_informed_df" in globals():
    export_csv(campaign_rank_shift_dml_informed_df, "table_campaign_rank_shift_dml_informed.csv")
if "campaign_rank_shift_support_filtered_df" in globals():
    export_csv(campaign_rank_shift_support_filtered_df, "table_campaign_rank_shift_support_filtered.csv")
if "campaign_local_vs_global_ranking_df" in globals() and isinstance(campaign_local_vs_global_ranking_df, pd.DataFrame):
    export_csv(campaign_local_vs_global_ranking_df, "table_campaign_local_vs_global_ranking.csv")

if "summary_heterogeneity_managerial" in globals():
    export_json(summary_heterogeneity_managerial, "summary_heterogeneity_managerial.json")
if "summary_campaign_rank_shift_dml_informed" in globals():
    export_json(summary_campaign_rank_shift_dml_informed, "summary_campaign_rank_shift_dml_informed.json")
if "summary_campaign_ranking_interpretation" in globals():
    export_json(summary_campaign_ranking_interpretation, "summary_campaign_ranking_interpretation.json")

summary_bundle_i2 = dict(summary_bundle) if "summary_bundle" in globals() else {}
if "summary_dml_analysis_frame" in globals():
    summary_bundle_i2["dml_analysis_frame"] = summary_dml_analysis_frame
if "summary_heterogeneity_managerial" in globals():
    summary_bundle_i2["dml_adjusted_heterogeneity"] = summary_heterogeneity_managerial
if "summary_campaign_rank_shift_dml_informed" in globals():
    summary_bundle_i2["campaign_rank_shift_dml_informed"] = summary_campaign_rank_shift_dml_informed
if "summary_campaign_ranking_interpretation" in globals():
    summary_bundle_i2["campaign_ranking_interpretation"] = summary_campaign_ranking_interpretation
export_json(summary_bundle_i2, "summary_bundle_for_writing.json")
summary_bundle = summary_bundle_i2

existing_table_index = list(table_index) if "table_index" in globals() else []
existing_table_index.extend(
    [
        {"paper_table": "D4", "file": "table_dml_analysis_frame_summary.csv", "content": "Standardized Step D analysis frame summary"},
        {"paper_table": "D4", "file": "summary_dml_analysis_frame.json", "content": "Metadata for the standardized Step D analysis frame"},
        {"paper_table": "Main paper", "file": "table_heterogeneity_propensity.csv", "content": "DML-adjusted heterogeneity across propensity states"},
        {"paper_table": "Main paper", "file": "table_heterogeneity_recency.csv", "content": "DML-adjusted heterogeneity across recency states"},
        {"paper_table": "Main paper", "file": "table_heterogeneity_intensity.csv", "content": "DML-adjusted heterogeneity across intensity states"},
        {"paper_table": "Main paper", "file": "table_campaign_dml_informed_ranking.csv", "content": "Support-aware DML-informed campaign ranking"},
        {"paper_table": "Main paper", "file": "table_campaign_rank_shift_dml_informed.csv", "content": "Rank shifts versus heuristic campaign baselines"},
        {"paper_table": "Appendix", "file": "table_heterogeneity_all.csv", "content": "Combined DML-adjusted heterogeneity table"},
        {"paper_table": "Appendix", "file": "table_campaign_support_diagnostics.csv", "content": "Campaign support diagnostics for DML-informed ranking"},
        {"paper_table": "Appendix", "file": "table_campaign_rank_shift_support_filtered.csv", "content": "Rank shifts restricted to support-sufficient campaigns"},
        {"paper_table": "Appendix", "file": "summary_heterogeneity_managerial.json", "content": "Managerial summary of DML-adjusted heterogeneity"},
        {"paper_table": "Appendix", "file": "summary_campaign_rank_shift_dml_informed.json", "content": "Summary of support-filtered campaign rank shifts"},
        {"paper_table": "Appendix", "file": "summary_campaign_ranking_interpretation.json", "content": "Interpretation guardrails for campaign ranking"},
        {"paper_table": "Summary bundle", "file": "summary_bundle_for_writing.json", "content": "Updated summary bundle including DML heterogeneity and ranking"},
    ]
)
if "campaign_local_vs_global_ranking_df" in globals() and isinstance(campaign_local_vs_global_ranking_df, pd.DataFrame):
    existing_table_index.append(
        {"paper_table": "Appendix", "file": "table_campaign_local_vs_global_ranking.csv", "content": "Local versus global residualization comparison for campaign ranking"}
    )

table_index_df = (
    pd.DataFrame(existing_table_index)
    .drop_duplicates(subset=["file"], keep="last")
    .reset_index(drop=True)
)
table_index = table_index_df.to_dict(orient="records")
export_csv(table_index_df, "paper_table_index.csv")
display(table_index_df)
